# Customer EDA — Active CIFs

**Goal.** Describe who the customer is — the typical profile, how concentrated the book is, and how the main customer segments differ: **Individual vs. Business** (`IorB`), **Foreign vs. Domestic** (`ForD`), **Select vs. Non-Select**, and *every combination of the three*.

**Grain.** One row per **CIF** (customer). Balances are an overnight snapshot; account columns are counts at CIF level.

**Scope.** Only **active CIFs** (`CIF_Status == 'Active'`) in §3–§19. Inactive and prospect CIFs are removed in §3; they come back in §11.1 (survival by vintage) and in **§20–§22**, the attrition and history sections. §23–§26 add banking views of the active base.

**How to read this notebook**

| Section | Question it answers |
|---|---|
| 1–2 Setup & load | Brand palette, configuration, data dictionary, explicit column classification |
| 3 Scope | Which CIFs are in / out |
| 4 Data quality | What is missing, inconsistent or impossible — including the **DD / CD / LN account-totals reconciliation** |
| 5 Features | Derived fields used everywhere else (Select flag, Deposits, Loans, Total balance, age & tenure bands, region, product mix) |
| 6 Portfolio overview | The customer base on one page: KPIs and the 8-cell segment grid |
| 7 Segment deep-dive | **Every granularity** (IorB, ForD, Select and all combinations) × every feature |
| 8 Age | Age distribution of individuals and business age, by segment |
| 9 Concentration & value | Who holds the money: Lorenz among holders, value pyramid, top-100 profile, tail table |
| 10 Balances in depth | Shape of the distributions, deposit-tier mix per segment, loan-to-deposit relationship |
| 11 Time | Vintages, survival by vintage (from the full file), closure signals inside active CIFs |
| 12 Geography | Country and region, Venezuela vs. the rest, value per country, treemap |
| 13 Branches, officers & segment codes | Channel view, branch × segment, house codes vs. relationship officers |
| 14 Relationships & tests | Correlations, age / tenure / products vs. balances, Select uplift with confidence intervals, effect sizes |
| 15 What makes a valuable customer | Explanatory models: permutation importance + the simplest decision rules |
| 16 Segment profiles | The final "who is our customer" table and fingerprint heatmap |
| 17 Personas (optional) | k-means personas per IorB |
| 18 Key findings | Auto-generated numbers + narrative template |
| 19 Open questions & data to request | What to pull next |
| 20 **Attrition — non-active CIFs** | Who left, when, how long they stayed, how many accounts they had, how many had loans / CDs / deposits — by IorB, ForD and Venezuela / other foreign (no Select: it is only known today) |
| 21 **The customer over time** | Base bridge (new − lost per year), how the customer base and the new customers have changed, cohort retention |
| 22 **BCG matrix** | Which segments grow and which hold the money: stars, cash cows, question marks, dogs |
| 23 **Customer value matrix** | Value × relationship depth, with an action per cell |
| 24 **Funding profile** | Insured vs uninsured deposits, DDA vs CD, depositor and country concentration |
| 25 **Early warning** | Rule-based attrition flags on active customers, risk tiers, watch list |
| 26 **Synthesis** | Who the customer is and who it has been, one row per IorB × ForD; enriched export |

**Color legend (brand palette).** <span style="color:#003C71">■</span> Blue `#003C71` = Individual · Domestic · first level of a split &nbsp; <span style="color:#007953">■</span> Green `#007953` = Business · Foreign · Select &nbsp; <span style="color:#5B8BC5">■</span><span style="color:#4FB08E">■</span> lighter tints = second level of a 2-way split &nbsp; <span style="color:#E1261C">■</span> Red `#E1261C` = **reserved** for alerts, negatives and highlights.

**Conventions.** Medians describe the typical customer (balances are heavy-tailed, so averages mislead). Balance distributions and concentration are shown **among holders** (balance > 0) with the share of holders annotated. Median bars carry **95% bootstrap confidence intervals**; comparisons come with an effect size (Cliff's δ), not just a p-value. Every chart has a table next to it with the exact numbers.

## Data dictionary

| Column | Role | Notes |
|---|---|---|
| `CIF` | ID | Customer key. Must be unique after scoping. Kept as text. |
| `IorB` | Dimension (2) | Individual / Business — primary split. |
| `ForD` | Dimension (2) | Foreign / Domestic — secondary split. |
| `Banesco Select` | Dimension (flag) | `'S'` = Select; blank = Non-Select (**assumption**, stated in §5). |
| `Segment` | Dimension (code) | `[I\|D\|P][P\|B][tier]`, e.g. `IP2`, `DB5`, or `Unassigned`. Decoded in §4.5 and cross-checked against IorB / ForD. |
| `BranchGroupName`, `BranchName` | Dimension | Channel. `BranchCode` is excluded (redundant with the name). |
| `OfficerCode` | Dimension (high cardinality) | Kept as **text** (leading zeros). Analyzed as portfolio concentration, never level by level. `OfficerName` excluded (PII). |
| `Country` | Dimension (high cardinality) | Mapped to `Region` in §5. Cross-checked against `ForD` in §4.5. |
| `CIF_Status` | Scope filter | Active / Inactive / Prospect. Only Active is kept. |
| `AGE`, `BIRTHDATE_TEXT` | Measure / Date | For Individuals = age of the person. For Business = **years since incorporation**. Split into `Age` and `BusinessAge` in §5. |
| `effectiveCIFCreationDate`, `CIF_Tenure_Days`, `CIF_Tenure_Years` | Date / Measure | Tenure. `CIF_Tenure_Years` is used; `_Days` is redundant. |
| `FirstAccountCreationDate`, `LastAccountCloseDate` | Date | Blank `LastAccountCloseDate` = the CIF never closed an account (not a data problem). |
| `TotalAccountsEver`, `ActiveAccountsCount`, `ClosedAccountsCount` | Measure (count) | **All** account types. |
| `DD/CD/LN Accounts Active`, `… Closed` | Measure (count) | Only three product families are in the extract, so they do **not** add up to the totals above — reconciled in §4.3. |
| `DD/CD/LN Accounts Balance` | Measure ($) | Overnight balances. DD can be negative (overdraft). LN = outstanding loan balance. |
| `CustomerName`, `OfficerName`, `HouseholdName`, `HouseholdID`, `PRIMARY` | Excluded | PII, IDs, not analyzed. |

## Executive summary — who the customer is

*Fill in after running the notebook on the current snapshot (numbers come from §18, §20.9 and the observation cells).*

**Headline.** By headcount … ; by money … .

**The customer in four sentences.**

1. **The typical individual** … 
2. **The typical business** … 
3. **Where the money is** … 
4. **Select** … 

**The eight segments in one line each** are in §16; the personas in §17.

**Three risks the numbers show.** … 

**Two blind spots.** … 

# 1. Setup

Imports, the brand palette, and the plotting helpers used by every section. Nothing here depends on the data.

In [ ]:
import os, re, math, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from scipy.stats import chi2_contingency
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

# ---------------- Brand palette ----------------
BLUE, GREEN, RED = '#003C71', '#007953', '#E1261C'
BLUE_T, GREEN_T  = '#5B8BC5', '#4FB08E'            # tints = second level of a 2-way split
GRAY             = '#8B95A5'                       # neutral, used only for "Other / unclassified"
INK = dict(primary='#1F2937', secondary='#4B5563', muted='#8B95A5', grid='#E6E8EC', surface='#FFFFFF', card='#F5F7FA')
CMAP_BLUE  = LinearSegmentedColormap.from_list('brand_blue',  ['#F3F6FA', BLUE])
CMAP_GREEN = LinearSegmentedColormap.from_list('brand_green', ['#F1F8F5', GREEN])
CMAP_DIV   = LinearSegmentedColormap.from_list('brand_div',   [RED, '#F2F2F2', BLUE])
BLUES_7    = [CMAP_BLUE(x) for x in np.linspace(0.18, 1.0, 7)]

sns.set_theme(style='white')
mpl.rcParams.update({
    'figure.facecolor': INK['surface'], 'axes.facecolor': INK['surface'], 'savefig.facecolor': INK['surface'],
    'figure.dpi': 105, 'savefig.dpi': 150,
    'font.size': 10, 'axes.titlesize': 10.5, 'axes.titleweight': 'semibold', 'axes.titlelocation': 'left', 'axes.titlecolor': INK['primary'],
    'axes.labelsize': 9, 'axes.labelcolor': INK['secondary'],
    'axes.edgecolor': INK['grid'], 'axes.linewidth': 0.8, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'axes.grid.axis': 'y', 'grid.color': INK['grid'], 'grid.linewidth': 0.7,
    'xtick.color': INK['secondary'], 'ytick.color': INK['secondary'], 'xtick.labelsize': 8.5, 'ytick.labelsize': 8.5,
    'legend.frameon': False, 'legend.fontsize': 8.5,
    'axes.prop_cycle': mpl.cycler(color=[BLUE, GREEN, BLUE_T, GREEN_T]),
})

# ---------------- Segment dimensions: fixed order and fixed colors ----------------
ORDER = {'IorB': ['Individual', 'Business'], 'ForD': ['Domestic', 'Foreign'], 'Select': ['Select', 'Non-Select'],
         'ForeignSplit': ['Domestic', 'Venezuela', 'Other foreign'], 'ValueBand': ['Top 1%', 'Next 9%', 'Next 40%', 'Bottom 50%']}
COLOR = {'IorB': {'Individual': BLUE, 'Business': GREEN},
         'ForD': {'Domestic': BLUE, 'Foreign': GREEN},
         'Select': {'Select': GREEN, 'Non-Select': GREEN_T},
         'ForeignSplit': {'Domestic': BLUE, 'Venezuela': GREEN, 'Other foreign': GREEN_T}}
SEP = ' · '

def group_spec(dims):
    """Ordered labels + palette for 1 or 2 dims. Hue = dims[0]; full color / tint = levels of dims[1]."""
    if len(dims) == 1:
        return list(ORDER[dims[0]]), dict(COLOR[dims[0]])
    d1, d2 = dims[0], dims[1]
    labels, pal = [], {}
    for a in ORDER[d1]:
        base = COLOR[d1][a]
        full, tint = (BLUE, BLUE_T) if base == BLUE else (GREEN, GREEN_T)
        for j, b in enumerate(ORDER[d2]):
            lab = a + SEP + b
            labels.append(lab); pal[lab] = full if j == 0 else tint
    return labels, pal

def split_dims(dims):
    """Color encodes up to two dims; a third dim becomes side-by-side panels."""
    return (list(dims[:2]), dims[2]) if len(dims) == 3 else (list(dims), None)

def make_label(d, dims):
    lab = d[dims[0]].astype(str)
    for c in dims[1:]:
        lab = lab + SEP + d[c].astype(str)
    return lab

# ---------------- Formatters ----------------
def money(x, pos=None):
    if x is None or (isinstance(x, float) and np.isnan(x)): return ''
    s = '-' if x < 0 else ''; a = abs(x)
    if a >= 1e9: return f'{s}$' + f'{a/1e9:.1f}'.rstrip('0').rstrip('.') + 'B'
    if a >= 1e6: return f'{s}$' + f'{a/1e6:.1f}'.rstrip('0').rstrip('.') + 'M'
    if a >= 1e3: return f'{s}${a/1e3:.0f}K'
    return f'{s}${a:,.0f}'
MONEY = FuncFormatter(money)
PCT0  = PercentFormatter(100, decimals=0)
def fmt_int(x): return '' if pd.isna(x) else f'{x:,.0f}'
def fmt_pct(x): return '' if pd.isna(x) else f'{x:.1f}%'
def fmt_1(x):   return '' if pd.isna(x) else f'{x:,.1f}'
def is_(s, v):  return s.eq(v).fillna(False).astype(bool)

# ---------------- Figure helpers ----------------
def finish(fig, title=None, subtitle=None, name=None):
    """Left-aligned title block in ink colors, tight layout, optional PNG export, show."""
    h = fig.get_figheight()
    if title:
        block = 0.66 if subtitle else 0.42
        fig.tight_layout(rect=[0, 0, 1, 1 - block / h])
        fig.text(0.01, 1 - 0.14 / h, title, ha='left', va='top', fontsize=13, fontweight='bold', color=INK['primary'])
        if subtitle:
            fig.text(0.01, 1 - 0.40 / h, subtitle, ha='left', va='top', fontsize=9.3, color=INK['muted'])
    else:
        fig.tight_layout()
    if globals().get('SAVE_FIGS') and name:
        fig.savefig(os.path.join(FIG_DIR, f'{name}.png'), bbox_inches='tight')
    plt.show(); plt.close(fig)

def style(ax, title=None, xlabel='', ylabel='', money_x=False, money_y=False, pct_x=False, pct_y=False):
    if title is not None: ax.set_title(title)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    if money_x: ax.xaxis.set_major_formatter(MONEY)
    if money_y: ax.yaxis.set_major_formatter(MONEY)
    if pct_x:   ax.xaxis.set_major_formatter(PCT0)
    if pct_y:   ax.yaxis.set_major_formatter(PCT0)
    return ax

def hbar(ax, labels, values, colors, fmt=fmt_int, hatches=None, height=0.62):
    """Horizontal bars with direct value labels. Values must be >= 0."""
    y = np.arange(len(labels)); values = np.asarray(values, dtype=float)
    bars = ax.barh(y, values, color=colors, height=height, edgecolor=INK['surface'], linewidth=0.8)
    if hatches:
        for b, hch in zip(bars, hatches): b.set_hatch(hch)
    ax.set_yticks(y); ax.set_yticklabels(labels); ax.invert_yaxis()
    ax.grid(axis='y', visible=False); ax.grid(axis='x', color=INK['grid'])
    vmax = np.nanmax(values) if len(values) and np.nanmax(values) > 0 else 1
    for b, v in zip(bars, values):
        if not np.isnan(v):
            ax.text(b.get_width() + vmax * 0.012, b.get_y() + b.get_height() / 2, fmt(v), va='center', ha='left', fontsize=8.3, color=INK['secondary'])
    ax.set_xlim(0, vmax * 1.2)
    return bars

def grouped_bars(ax, ct, order, pal, fmt=None, lo=None, hi=None):
    """ct: index = x categories, columns = groups (subset of order). lo/hi: optional CI bounds (same shape as ct)."""
    cats = list(ct.index); k = max(len([g for g in order if g in ct.columns]), 1); w = 0.8 / k
    x = np.arange(len(cats)); j = 0
    for g in order:
        if g not in ct.columns: continue
        vals = ct[g].values.astype(float); xs = x + (j - (k - 1) / 2) * w
        ax.bar(xs, vals, width=w * 0.9, color=pal[g], label=g, edgecolor=INK['surface'], linewidth=0.5)
        top = vals.copy()
        if lo is not None and hi is not None and g in lo.columns:
            l = lo[g].reindex(ct.index).values.astype(float); h = hi[g].reindex(ct.index).values.astype(float)
            ok = ~np.isnan(l) & ~np.isnan(h) & ~np.isnan(vals)
            if ok.any():
                ax.errorbar(xs[ok], vals[ok], yerr=[np.clip(vals[ok] - l[ok], 0, None), np.clip(h[ok] - vals[ok], 0, None)],
                            fmt='none', ecolor=INK['secondary'], elinewidth=1, capsize=2.5, zorder=3)
                top[ok] = np.maximum(vals[ok], h[ok])
        if fmt:
            for xi, v, t in zip(xs, vals, top):
                if not np.isnan(v) and v > 0: ax.text(xi, t, fmt(v), ha='center', va='bottom', fontsize=7.2, color=INK['secondary'])
        j += 1
    ax.set_xticks(x); ax.set_xticklabels(cats)
    ax.legend(); return ax

def median_ci(s, n_boot=300, alpha=0.05, seed=1):
    """95% bootstrap percentile interval for the median."""
    a = np.asarray(pd.Series(s).dropna(), dtype=float)
    if len(a) < 10: return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    meds = np.median(a[rng.integers(0, len(a), size=(n_boot, len(a)))], axis=1)
    return (float(np.quantile(meds, alpha / 2)), float(np.quantile(meds, 1 - alpha / 2)))

def median_table(d, col, row, colgrp, row_order, col_order, holders=True):
    """Median of `col` for every row × column group (among holders if holders=True) with bootstrap CIs. Returns (median, lo, hi)."""
    dd = d[[col, row, colgrp]].dropna()
    if holders: dd = dd[dd[col] > 0]
    med = pd.DataFrame(np.nan, index=row_order, columns=col_order, dtype=float); lo = med.copy(); hi = med.copy()
    for r in row_order:
        for c in col_order:
            s = dd.loc[is_(dd[row], r) & is_(dd[colgrp], c), col]
            if len(s):
                med.loc[r, c] = s.median(); lo.loc[r, c], hi.loc[r, c] = median_ci(s)
    return med, lo, hi

def lump(s, k=6, other='Other'):
    """Keep the k most frequent levels, fold the rest into `other`."""
    keep = s.value_counts().head(k).index
    return s.where(s.isin(keep) | s.isna(), other)

def text_on(color):
    r, g, b = mpl.colors.to_rgb(color)
    return 'white' if (0.2126 * r + 0.7152 * g + 0.0722 * b) < 0.5 else INK['primary']

# ---- Squarified treemap layout (Bruls, Huizing & van Wijk) ----
def _sq_layout(sizes, x, y, dx, dy):
    area = sum(sizes); rects = []
    if dx >= dy:
        w = area / dy
        for sz in sizes: rects.append((x, y, w, sz / w)); y += sz / w
    else:
        h = area / dx
        for sz in sizes: rects.append((x, y, sz / h, h)); x += sz / h
    return rects
def _sq_leftover(sizes, x, y, dx, dy):
    area = sum(sizes)
    return (x + area / dy, y, dx - area / dy, dy) if dx >= dy else (x, y + area / dx, dx, dy - area / dx)
def _sq_worst(sizes, x, y, dx, dy):
    return max(max(w / h, h / w) for _, _, w, h in _sq_layout(sizes, x, y, dx, dy))
def squarify(sizes, x=0.0, y=0.0, dx=100.0, dy=100.0):
    """sizes: positive numbers sorted descending. Returns (x, y, w, h) rectangles filling dx × dy."""
    sizes = [float(v) for v in sizes]; total = sum(sizes)
    sizes = [v * dx * dy / total for v in sizes]; rects = []
    while sizes:
        if dx <= 1e-9 or dy <= 1e-9:
            rects += [(x, y, 0.0, 0.0)] * len(sizes); break
        i = 1
        while i < len(sizes) and _sq_worst(sizes[:i], x, y, dx, dy) >= _sq_worst(sizes[:i + 1], x, y, dx, dy): i += 1
        rects += _sq_layout(sizes[:i], x, y, dx, dy)
        x, y, dx, dy = _sq_leftover(sizes[:i], x, y, dx, dy)
        sizes = sizes[i:]
    return rects

def kde_by(ax, d, col, group_col, order, pal, clip=None, bw=1.0):
    """Overlaid density curves per group with a dashed median line."""
    drawn = 0
    for g in order:
        s = d.loc[is_(d[group_col], g), col].dropna()
        if clip is not None: s = s.clip(*clip)
        if len(s) < 5 or s.nunique() < 2: continue
        sns.kdeplot(s, ax=ax, color=pal[g], fill=True, alpha=0.16, linewidth=2, cut=0, bw_adjust=bw, label=f'{g} (n={len(s):,}, median {s.median():,.0f})')
        ax.axvline(s.median(), color=pal[g], lw=1, ls='--', alpha=0.9); drawn += 1
    ax.set_yticks([]); ax.set_ylabel(''); ax.grid(False)
    if drawn:
        ax.set_ylim(0, ax.get_ylim()[1] * 1.45); ax.legend(loc='upper right')
    else: ax.text(0.5, 0.5, 'no data', ha='center', va='center', transform=ax.transAxes, color=INK['muted'])
    return ax

def box_by(ax, d, col, group_col, order, pal, holders_only=True, annotate=True):
    """Horizontal box plot per group on a log $ axis, among holders (balance > 0). Annotates % holders and n."""
    dd = d[[col, group_col]].dropna()
    pop = dd.groupby(group_col)[col].size()
    if holders_only: dd = dd[dd[col] > 0]
    present = [g for g in order if is_(dd[group_col], g).any()]
    if not present:
        ax.text(0.5, 0.5, 'no holders', ha='center', va='center', transform=ax.transAxes, color=INK['muted']); ax.set_yticks([]); return ax
    dd = dd[dd[group_col].isin(present)].copy(); dd[group_col] = dd[group_col].astype(str)
    sns.boxplot(data=dd, x=col, y=group_col, order=present, hue=group_col, palette=pal, legend=False,
                showfliers=False, width=0.55, linewidth=1.0, ax=ax, log_scale=True)
    lo, hi = ax.get_xlim(); ax.set_xlim(lo, hi * 60)
    for i, g in enumerate(present):
        s = dd.loc[dd[group_col] == g, col]
        ax.text(s.median(), i - 0.36, money(s.median()), ha='center', va='bottom', fontsize=7.8, color=INK['secondary'])
        if annotate:
            share = len(s) / pop.get(g, np.nan) * 100
            ax.text(0.995, i, f'{share:.0f}% hold · n={len(s):,}', transform=ax.get_yaxis_transform(), ha='right', va='center', fontsize=7.8, color=INK['muted'])
    ax.xaxis.set_major_formatter(MONEY); ax.set_ylabel(''); ax.set_xlabel('')
    ax.grid(axis='x', color=INK['grid']); ax.grid(axis='y', visible=False)
    return ax

def styled(tbl, pct_cols=(), money_cols=(), int_cols=(), one_dec=(), gradient_cols=(), cmap=None):
    fmt = {}
    fmt.update({c: fmt_pct for c in pct_cols if c in tbl.columns})
    fmt.update({c: money for c in money_cols if c in tbl.columns})
    fmt.update({c: fmt_int for c in int_cols if c in tbl.columns})
    fmt.update({c: fmt_1 for c in one_dec if c in tbl.columns})
    st = tbl.style.format(fmt, na_rep='')
    g = [c for c in gradient_cols if c in tbl.columns]
    if g: st = st.background_gradient(cmap=cmap or CMAP_BLUE, subset=g, vmin=0)
    return st

def cramers_v(a, b):
    try:
        t = pd.crosstab(a.astype(str), b.astype(str))
        t = t.loc[t.sum(1) > 0, t.sum(0) > 0]
        if t.shape[0] < 2 or t.shape[1] < 2: return np.nan
        chi2 = chi2_contingency(t, correction=False)[0]
        return float(np.sqrt(chi2 / (t.values.sum() * (min(t.shape) - 1))))
    except Exception:
        return np.nan

print('Setup OK · pandas', pd.__version__, '· seaborn', sns.__version__)

# 2. Load & configuration

The configuration cell is the only one you normally edit. In Colab, the file is uploaded automatically if it is not already in the working directory.

In [ ]:
# ============================ CONFIGURATION ============================
FILE_PATH     = 'Data EDA.xlsx'     # xlsx / xls / csv / parquet
SHEET         = 0                   # sheet name or index (Excel only)
HEADER        = 0                   # header row (0 = first row)

SNAPSHOT_DATE = None                # e.g. '2026-09-01'. None = latest date found in the file. Used for age / tenure cross-checks.

ID_COL        = 'CIF'
ACTIVE_STATUS = 'Active'            # value of CIF_Status that defines the population

EXCLUDE_COLS  = ['CustomerName', 'OfficerName', 'HouseholdName', 'HouseholdID', 'BranchCode', 'PRIMARY']   # never analyzed
CODE_COLS     = ['CIF', 'OfficerCode', 'BranchCode', 'HouseholdID', 'Segment']                              # read as text (leading zeros)

# Outputs
SAVE_FIGS     = False               # True → every figure is also saved as PNG in FIG_DIR (for the deck / Power BI)
FIG_DIR       = 'figures'
EXPORT_CLEAN  = True                # True → clean CIF-level table with derived features written to CLEAN_PATH (Power BI source)
CLEAN_PATH    = 'active_cifs_clean.csv'

# Analysis knobs
RUN_PERSONAS  = True                # §14 k-means personas (descriptive only)
K_PERSONAS    = 4
TOP_COUNTRIES = 15

if SAVE_FIGS: os.makedirs(FIG_DIR, exist_ok=True)

In [ ]:
# ---- Colab: upload the file if it is not already present ----
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB and not os.path.exists(FILE_PATH):
    from google.colab import files
    uploaded = files.upload()
    if uploaded: FILE_PATH = next(iter(uploaded))

def load(path, sheet=0, header=0):
    p = str(path).lower()
    if p.endswith('.csv'):
        head = pd.read_csv(path, header=header, nrows=0)
        return pd.read_csv(path, header=header, dtype={c: str for c in CODE_COLS if c in head.columns}, low_memory=False)
    if p.endswith('.parquet'):
        return pd.read_parquet(path)
    xls = pd.ExcelFile(path)
    print('Sheets:', xls.sheet_names)
    head = pd.read_excel(xls, sheet_name=sheet, header=header, nrows=0)
    return pd.read_excel(xls, sheet_name=sheet, header=header, dtype={c: str for c in CODE_COLS if c in head.columns})

raw = load(FILE_PATH, SHEET, HEADER)
raw.columns = [str(c).strip() for c in raw.columns]
raw = raw.drop(columns=[c for c in EXCLUDE_COLS if c in raw.columns])
empty = [c for c in raw.columns if raw[c].isna().all()]
if empty:
    print('Dropping fully empty columns:', empty); raw = raw.drop(columns=empty)
print(f'Loaded {len(raw):,} rows × {raw.shape[1]} columns')

## 2.1 Column classification

Explicit, not guessed: every column is assigned a role from the data dictionary. Any column in the file that is not in the dictionary is reported so it can be classified on purpose.

In [ ]:
DIMS      = ['IorB', 'ForD', 'Banesco Select', 'Segment', 'BranchGroupName', 'BranchName', 'Country', 'OfficerCode', 'CIF_Status']
DATES     = ['BIRTHDATE_TEXT', 'effectiveCIFCreationDate', 'FirstAccountCreationDate', 'LastAccountCloseDate']
BAL_COLS  = ['DD Accounts Balance', 'CD Accounts Balance', 'LN Accounts Balance']
CNT_COLS  = ['TotalAccountsEver', 'ActiveAccountsCount', 'ClosedAccountsCount',
             'DD Accounts Active', 'DD Accounts Closed', 'CD Accounts Active', 'CD Accounts Closed', 'LN Accounts Active', 'LN Accounts Closed']
OTHER_NUM = ['AGE', 'CIF_Tenure_Days', 'CIF_Tenure_Years']
FAM       = ['DD', 'CD', 'LN']

known   = [ID_COL] + DIMS + DATES + BAL_COLS + CNT_COLS + OTHER_NUM
unknown = [c for c in raw.columns if c not in known]
missing = [c for c in known if c not in raw.columns]
if unknown: print('⚠ Columns in the file that are NOT in the dictionary (classify them in DIMS / measures / EXCLUDE_COLS):', unknown)
if missing: print('⚠ Dictionary columns MISSING from the file:', missing)
DIMS      = [c for c in DIMS if c in raw.columns]
DATES     = [c for c in DATES if c in raw.columns]
OTHER_NUM = [c for c in OTHER_NUM if c in raw.columns]
MEASURES  = BAL_COLS + CNT_COLS + OTHER_NUM

# ---- Type enforcement ----
df = raw.copy()
PLACEHOLDERS = {'MISSING DATA', 'MISSING', 'N/A', 'NA', 'N.A.', 'NULL', 'NONE', 'UNKNOWN', 'TBD', '-', '--', '?', '.', ''}
def clean_text(v): return v.strip() if isinstance(v, str) else v
for c in DIMS + [ID_COL]:
    df[c] = df[c].map(clean_text).astype(object)
if 'Country' in df.columns:
    df['Country'] = df['Country'].map(lambda v: v.upper() if isinstance(v, str) else v)
for c in DIMS:
    ph = df[c].map(lambda v: isinstance(v, str) and v.strip().upper() in PLACEHOLDERS)
    if ph.any():
        print(f'{c}: {ph.sum():,} placeholder values ({sorted(df.loc[ph, c].unique())[:5]}) treated as missing')
        df.loc[ph, c] = np.nan
for c in DATES:
    s = pd.to_datetime(df[c], errors='coerce', utc=True)
    df[c] = s.dt.tz_localize(None)
for c in MEASURES:
    s = df[c]
    if not pd.api.types.is_numeric_dtype(s):
        s = s.astype(str).str.replace(',', '', regex=False).str.replace('$', '', regex=False).str.strip()
    df[c] = pd.to_numeric(s, errors='coerce').astype('float64').replace([np.inf, -np.inf], np.nan)

role = {ID_COL: 'id'}
role.update({c: 'dimension' for c in DIMS}); role.update({c: 'date' for c in DATES})
role.update({c: 'measure · balance ($)' for c in BAL_COLS}); role.update({c: 'measure · count' for c in CNT_COLS}); role.update({c: 'measure · other' for c in OTHER_NUM})
classification = pd.DataFrame({
    'role': pd.Series(role), 'dtype': df.dtypes.astype(str),
    'nulls %': (df.isna().mean() * 100).round(1), 'distinct': df.nunique(),
    'example': [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
}).loc[df.columns]
display(classification)

# 3. Scope — active CIFs only

**Rule.** Keep `CIF_Status == 'Active'`. Inactive CIFs and prospects (no products) are removed here; they come back only in §11.1 and in the attrition and history sections (§20–§22). The status split itself is a finding: it tells how large the dormant / lost base is relative to the active one.

In [ ]:
status = df['CIF_Status'].fillna('(missing)').value_counts().rename_axis('CIF_Status').to_frame('CIFs')
status['% of file'] = status['CIFs'] / status['CIFs'].sum() * 100
display(styled(status, pct_cols=['% of file'], int_cols=['CIFs']))

fig, ax = plt.subplots(figsize=(7, 0.6 + 0.5 * len(status)))
cols = [BLUE if s == ACTIVE_STATUS else GRAY for s in status.index]
hbar(ax, status.index.tolist(), status['CIFs'].values, cols, fmt=lambda v: f'{v:,.0f}  ({v / status["CIFs"].sum():.0%})')
style(ax, xlabel='CIFs')
finish(fig, 'CIF status in the file', f'Only "{ACTIVE_STATUS}" (blue) is analyzed from here on', name='03_status')

n0 = len(df)
df = df[is_(df['CIF_Status'], ACTIVE_STATUS)].copy()
print(f'Kept {len(df):,} active CIFs out of {n0:,} rows ({len(df) / n0:.1%}). Removed {n0 - len(df):,}.')
dup = int(df[ID_COL].duplicated().sum())
assert dup == 0, f'{dup} duplicated CIFs after scoping — the grain is not one row per CIF; fix before continuing'
DIMS = [c for c in DIMS if c != 'CIF_Status']       # constant now

# Sanity: the two primary splits must contain exactly the expected levels
for c in ['IorB', 'ForD']:
    extra = sorted(set(df[c].dropna().unique()) - set(ORDER[c]))
    if extra: print(f'⚠ Unexpected values in {c}: {extra} — add them to ORDER/COLOR in §1 or fix the source')
    if df[c].isna().any(): print(f'⚠ {c} has {df[c].isna().sum():,} missing values')

# 4. Data quality

Before interpreting anything: what is missing, whether the key is unique, whether the account totals reconcile, and which values are impossible. Nothing is deleted here; §5 applies the rules that come out of this section.

## 4.1 Nulls and structure

In [ ]:
quality = pd.DataFrame({
    'dtype': df.dtypes.astype(str), 'nulls': df.isna().sum(), 'pct_nulls': (df.isna().mean() * 100).round(1),
    'distinct': df.nunique(), 'example': [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
})
quality['flag'] = np.where(quality.pct_nulls > 20, 'HIGH', np.where(quality.pct_nulls > 5, 'medium', ''))
display(quality.sort_values('pct_nulls', ascending=False))

nulls = quality[quality.nulls > 0].sort_values('pct_nulls', ascending=False)
if len(nulls):
    fig, ax = plt.subplots(figsize=(8, 0.6 + 0.42 * len(nulls)))
    EXPECTED_BLANKS = {'Banesco Select', 'LastAccountCloseDate'}
    cols = [GRAY if c in EXPECTED_BLANKS else (RED if v > 20 else BLUE) for c, v in zip(nulls.index, nulls.pct_nulls)]
    hbar(ax, nulls.index.tolist(), nulls.pct_nulls.values, cols, fmt=fmt_pct)
    style(ax, xlabel='% of active CIFs'); ax.xaxis.set_major_formatter(PCT0)
    finish(fig, 'Missing values per column', 'Gray = blank by design (flag / never-closed). Red = more than 20% missing and unexplained.', name='04_nulls')
else:
    print('No missing values')

**Reading the nulls.** Two "HIGH" columns are not data problems: `LastAccountCloseDate` is blank when the CIF never closed an account, and `Banesco Select` is blank for Non-Select customers (it is a flag, not a missing value). `AGE` / `BIRTHDATE_TEXT` gaps are mostly businesses without an incorporation date — checked in §4.4.

## 4.2 Key integrity

In [ ]:
print(f'Fully duplicated rows: {df.duplicated().sum():,}')
print(f'{ID_COL}: {df[ID_COL].nunique():,} distinct in {len(df):,} rows · repeated: {df[ID_COL].duplicated(keep=False).sum():,} · null: {df[ID_COL].isna().sum():,}')
print('Grain confirmed: one row per CIF' if df[ID_COL].is_unique else '⚠ CIF is NOT unique')

## 4.3 Account totals reconciliation — DD / CD / LN

The extract carries three product families (**DD** demand deposits, **CD** certificates of deposit, **LN** loans) next to CIF-level totals (`ActiveAccountsCount`, `ClosedAccountsCount`, `TotalAccountsEver`). Three identities are tested:

1. `TotalAccountsEver = ActiveAccountsCount + ClosedAccountsCount`
2. `ActiveAccountsCount = DD + CD + LN active` — if the gap is **positive**, the CIF holds account types that are not in the extract (savings, money market, time deposits, cards, etc.). If it is **negative**, the row is wrong.
3. Same for closed accounts.

The gap is kept as `OtherAccountsActive` / `OtherAccountsClosed` so the missing families are visible in every chart instead of silently distorting the totals.

In [ ]:
act = [f'{f} Accounts Active' for f in FAM]; cls = [f'{f} Accounts Closed' for f in FAM]
df['Sum3_Active'] = df[act].sum(axis=1)
df['Sum3_Closed'] = df[cls].sum(axis=1)
df['OtherAccountsActive'] = df['ActiveAccountsCount'] - df['Sum3_Active']
df['OtherAccountsClosed'] = df['ClosedAccountsCount'] - df['Sum3_Closed']

tests = {
    'Total = Active + Closed':                          df['TotalAccountsEver'].eq(df['ActiveAccountsCount'] + df['ClosedAccountsCount']),
    'Active ≥ DD+CD+LN active (no negative gap)':       df['OtherAccountsActive'].ge(0),
    'Active = DD+CD+LN active (no other product types)':df['OtherAccountsActive'].eq(0),
    'Closed ≥ DD+CD+LN closed (no negative gap)':       df['OtherAccountsClosed'].ge(0),
    'Closed = DD+CD+LN closed (no other product types)':df['OtherAccountsClosed'].eq(0),
    'DD balance ≠ 0 only if DD active > 0':             ~(df['DD Accounts Balance'].ne(0) & df['DD Accounts Active'].eq(0)),
    'CD balance ≠ 0 only if CD active > 0':             ~(df['CD Accounts Balance'].ne(0) & df['CD Accounts Active'].eq(0)),
    'LN balance ≠ 0 only if LN active > 0':             ~(df['LN Accounts Balance'].ne(0) & df['LN Accounts Active'].eq(0)),
}
recon = pd.DataFrame({'CIFs passing': {k: int(v.fillna(False).sum()) for k, v in tests.items()}})
recon['CIFs failing'] = len(df) - recon['CIFs passing']
recon['% passing'] = recon['CIFs passing'] / len(df) * 100
for g in ORDER['IorB']:
    m = is_(df['IorB'], g)
    recon[f'% passing · {g}'] = [tests[k][m].fillna(False).mean() * 100 for k in recon.index]
display(styled(recon, pct_cols=[c for c in recon.columns if c.startswith('%')], int_cols=['CIFs passing', 'CIFs failing']))

# Where does the gap sit?
def gap_band(v):
    if pd.isna(v): return 'n/a'
    if v < 0: return 'negative (error)'
    if v == 0: return '0'
    if v <= 2: return f'{int(v)}'
    return '3+'
gap = pd.crosstab(df['OtherAccountsActive'].map(gap_band), df['IorB'].astype(str), margins=True, margins_name='All')
gap = gap.reindex([r for r in ['negative (error)', '0', '1', '2', '3+', 'n/a', 'All'] if r in gap.index])
print('\nActive accounts NOT in DD/CD/LN (gap per CIF) by IorB:'); display(gap)

acct_share = pd.DataFrame({
    'Active accounts': [df[c].sum() for c in act] + [df['OtherAccountsActive'].clip(lower=0).sum()],
    'Closed accounts': [df[c].sum() for c in cls] + [df['OtherAccountsClosed'].clip(lower=0).sum()],
}, index=FAM + ['Other (not in extract)'])
acct_share_pct = acct_share / acct_share.sum() * 100
print(f'\nShare of all accounts by family (active: {acct_share["Active accounts"].sum():,.0f} · closed: {acct_share["Closed accounts"].sum():,.0f}):')
display(styled(acct_share_pct, pct_cols=acct_share_pct.columns))

# Figure: accounts by family, per IorB — Other shown in gray so the blind spot is visible
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
fam_colors = {'DD': BLUE, 'CD': BLUE_T, 'LN': GREEN, 'Other (not in extract)': GRAY}
for ax, (kind, cols_) in zip(axes, [('Active', act + ['OtherAccountsActive']), ('Closed', cls + ['OtherAccountsClosed'])]):
    tot = df.groupby('IorB')[cols_].sum().clip(lower=0).reindex(ORDER['IorB'])
    tot.columns = FAM + ['Other (not in extract)']
    share = tot.div(tot.sum(axis=1), axis=0) * 100
    left = np.zeros(len(share))
    for fam in share.columns:
        vals = share[fam].values
        ax.barh(share.index, vals, left=left, color=fam_colors[fam], label=fam, height=0.55, edgecolor=INK['surface'], linewidth=1)
        for i, (l, v) in enumerate(zip(left, vals)):
            if v >= 6: ax.text(l + v / 2, i, f'{v:.0f}%', ha='center', va='center', fontsize=8, color='white' if fam != 'Other (not in extract)' else INK['primary'])
        left += vals
    ax.invert_yaxis(); ax.set_xlim(0, 100); ax.xaxis.set_major_formatter(PCT0); ax.grid(axis='y', visible=False); ax.grid(axis='x', color=INK['grid'])
    style(ax, title=f'{kind} accounts by family', xlabel='% of accounts' if kind == 'Active' else '')
axes[1].legend(loc='lower right', ncol=4, bbox_to_anchor=(1, -0.42))
finish(fig, 'Which account types the extract covers', 'Gray = accounts counted in the totals but not present as DD / CD / LN → request the missing product families', name='04_account_families')

**How to read §4.3.** Rows failing "no negative gap" are genuine errors in the source (more DD+CD+LN accounts than the total) and should go back to the data owner with the CIF list (§4.6). Rows with a *positive* gap are not errors: the customer holds product families that were not extracted. The gray share in the chart is the size of that blind spot — it is the single most useful thing to request next (see §19).

## 4.4 Suspicious values — ages, balances, dates

In [ ]:
date_cols_snap = [c for c in ['effectiveCIFCreationDate', 'FirstAccountCreationDate', 'LastAccountCloseDate'] if c in df.columns]
SNAPSHOT = pd.Timestamp(SNAPSHOT_DATE) if SNAPSHOT_DATE else max(df[c].max() for c in date_cols_snap)
print('Snapshot date used for cross-checks:', SNAPSHOT.date(), '(set SNAPSHOT_DATE in §2 to override)')

ind = is_(df['IorB'], 'Individual'); biz = is_(df['IorB'], 'Business')
age_calc    = (SNAPSHOT - df['BIRTHDATE_TEXT']).dt.days / 365.25
tenure_calc = (SNAPSHOT - df['effectiveCIFCreationDate']).dt.days / 365.25
dd_neg = df['DD Accounts Balance'] < 0

rows = []
def add(name, mask, note=''):
    mask = mask.fillna(False).astype(bool)
    rows.append({'check': name, 'CIFs': int(mask.sum()), '% of active': mask.mean() * 100, 'note': note})
add('Individuals · AGE missing',                       ind & df['AGE'].isna())
add('Individuals · AGE < 18 (minor / custodial)',      ind & (df['AGE'] < 18))
add('Individuals · AGE > 100 (implausible)',           ind & (df['AGE'] > 100), 'set to missing in §5')
add('Individuals · AGE = 0',                           ind & df['AGE'].eq(0))
add('Business · AGE (years since incorporation) missing', biz & df['AGE'].isna())
add('Business · AGE = 0 (incorporated < 1 year ago)',  biz & df['AGE'].eq(0), 'plausible — new companies')
add('AGE vs. age computed from BIRTHDATE differ > 1y', (df['AGE'] - age_calc).abs() > 1)
add('Tenure vs. tenure computed from CIF creation differ > 1y', (df['CIF_Tenure_Years'] - tenure_calc).abs() > 1)
add('DD balance negative (overdraft)',                  dd_neg, f'total {money(df.loc[dd_neg, "DD Accounts Balance"].sum())} — kept, real exposure')
add('CD balance negative',                              df['CD Accounts Balance'] < 0)
add('LN balance negative',                              df['LN Accounts Balance'] < 0)
add('Active CIF with 0 active accounts',                df['ActiveAccountsCount'].eq(0))
add('Active CIF with DD = CD = LN balance = 0',         df[BAL_COLS].abs().sum(axis=1).eq(0), 'may hold only non-extracted products')
add('First account opened BEFORE CIF creation date',    df['FirstAccountCreationDate'] < df['effectiveCIFCreationDate'], 'CIF re-created / merged?')
add('Last close BEFORE first account opened',           df['LastAccountCloseDate'] < df['FirstAccountCreationDate'])
add('Any date after the snapshot',                      (df[DATES] > SNAPSHOT).any(axis=1))
add('Individuals born after CIF creation',              ind & (df['BIRTHDATE_TEXT'] > df['effectiveCIFCreationDate']))
suspicious = pd.DataFrame(rows).set_index('check')
display(styled(suspicious, pct_cols=['% of active'], int_cols=['CIFs']))

# Numeric profile of every measure
prof = []
for c in MEASURES:
    s = df[c]
    prof.append({'column': c, 'negatives': int((s < 0).sum()), 'zeros': int((s == 0).sum()), 'min': s.min(), 'p1': s.quantile(.01),
                 'median': s.median(), 'p99': s.quantile(.99), 'max': s.max(), 'skew': float(s.skew())})
display(pd.DataFrame(prof).set_index('column'))
for c in DATES:
    s = df[c]; print(f'{c}: {s.min()} → {s.max()} · nulls {s.isna().sum():,} · after snapshot {(s > SNAPSHOT).sum():,}')

## 4.5 Consistency — segment code, country vs. ForD, text hygiene

`Segment` looks like `[geo][type][tier]`: geo **I** = International, **D** = Domestic, **P** = Puerto Rico; type **P** = personal (Individual), **B** = Business; tier = 1…n. If the decoding is right, the letters must agree with `ForD` and `IorB`. Country must agree with `ForD` too: Domestic ↔ United States / Puerto Rico.

In [ ]:
# ---- Segment code decoding ----
seg = df['Segment'].astype(object)
parts = seg.str.extract(r'^([A-Z])([PB])(\d+)$')
df['Segment_Geo']  = parts[0].map({'I': 'International', 'D': 'Domestic', 'P': 'Puerto Rico'})
df['Segment_Type'] = parts[1].map({'P': 'Individual', 'B': 'Business'})
df['Segment_Tier'] = pd.to_numeric(parts[2], errors='coerce')
decodable = df['Segment_Type'].notna()
print(f'Segment decodable: {decodable.mean():.1%} of CIFs · Unassigned: {is_(seg, "Unassigned").sum():,} · other undecodable: {(~decodable & ~is_(seg, "Unassigned")).sum():,}')
undec = seg[~decodable].value_counts().head(10)
if len(undec): print('Undecodable segment values:'); print(undec.to_string())

exp_ford = df['Segment_Geo'].map({'International': 'Foreign', 'Domestic': 'Domestic', 'Puerto Rico': 'Domestic'})
df['Segment_vs_IorB_ok'] = df['Segment_Type'].eq(df['IorB']).where(decodable)
df['Segment_vs_ForD_ok'] = exp_ford.eq(df['ForD']).where(decodable)
print(f'\nSegment type vs IorB agree: {df["Segment_vs_IorB_ok"].dropna().mean():.1%}   ·   Segment geo vs ForD agree: {df["Segment_vs_ForD_ok"].dropna().mean():.1%}  (decodable CIFs only)')
print('\nSegment type × IorB'); display(pd.crosstab(df['Segment_Type'].fillna('(undecodable)'), df['IorB'].astype(str), margins=True))
print('Segment geo × ForD');    display(pd.crosstab(df['Segment_Geo'].fillna('(undecodable)'), df['ForD'].astype(str), margins=True))

# ---- Country vs ForD ----
DOMESTIC_COUNTRIES = {'UNITED STATES', 'UNITED STATES OF AMERICA', 'USA', 'US', 'U.S.', 'U.S.A.', 'PUERTO RICO', 'PR',
                      'VIRGIN ISLANDS (U.S.)', 'U.S. VIRGIN ISLANDS', 'US VIRGIN ISLANDS', 'GUAM'}
ctry_kind = np.where(df['Country'].isna(), '(missing)', np.where(df['Country'].isin(DOMESTIC_COUNTRIES), 'US / PR', 'Other country'))
df['Country_vs_ForD_ok'] = np.where(df['Country'].isna(), np.nan,
                            np.where(is_(df['ForD'], 'Domestic'), df['Country'].isin(DOMESTIC_COUNTRIES), ~df['Country'].isin(DOMESTIC_COUNTRIES)))
print('\nCountry group × ForD'); display(pd.crosstab(pd.Series(ctry_kind, index=df.index, name='Country group'), df['ForD'].astype(str), margins=True))
bad = df[df['Country_vs_ForD_ok'].eq(0)]
print(f'CIFs whose country disagrees with ForD: {len(bad):,}')
if len(bad): display(bad.groupby(['ForD', 'Country']).size().sort_values(ascending=False).head(10).to_frame('CIFs'))

# ---- Text hygiene in dimensions ----
for c in DIMS:
    s = df[c].dropna().astype(str)
    if s.nunique() != s.str.strip().str.upper().nunique():
        print(f'⚠ {c}: {s.nunique()} raw values vs {s.str.strip().str.upper().nunique()} after upper/strip — inconsistent formatting')
placeholders = {'N/A', 'NA', 'NULL', 'NONE', 'UNKNOWN', 'TEST', '-', '--', '?', '.', '0', '999999', '9999'}
for c in DIMS:
    hits = df[c].dropna().astype(str).str.strip().str.upper().isin(placeholders).sum()
    if hits: print(f'⚠ {c}: {hits:,} placeholder values')

## 4.6 Outliers worth knowing by name

Extreme CIFs drive totals. They are listed so the reader can decide whether they are real (a corporate with hundreds of accounts, a large depositor) or errors.

In [ ]:
show_cols = [ID_COL, 'IorB', 'ForD', 'Segment', 'BranchGroupName', 'Country', 'ActiveAccountsCount', 'DD Accounts Balance', 'CD Accounts Balance', 'LN Accounts Balance']
show_cols = [c for c in show_cols if c in df.columns]
for c, k in [('ActiveAccountsCount', 5), ('DD Accounts Balance', 5), ('CD Accounts Balance', 5), ('LN Accounts Balance', 5)]:
    print(f'\nTop {k} CIFs by {c}:'); display(df.nlargest(k, c)[show_cols])
print('\nCIFs with a negative reconciliation gap (more DD+CD+LN accounts than the total):')
display(df[df['OtherAccountsActive'] < 0][show_cols + ['Sum3_Active', 'OtherAccountsActive']].head(10))

**Observations §4** *(fill in after the run)*

- **Missing data:** …
- **Account totals:** …
- **Impossible / odd values:** …
- **Segment code vs IorB / ForD:** …
- **Country vs ForD:** …

# 5. Feature engineering — rules applied

Every derived field used later is created here, with its rule stated once:

| Feature | Rule |
|---|---|
| `Select` | `'Select'` if `Banesco Select == 'S'`, else `'Non-Select'` (blank = Non-Select). |
| `Age` | `AGE` for Individuals only; values > 105 set to missing. |
| `BusinessAge` | `AGE` for Business only = years since incorporation. |
| `Deposits` | `DD balance + CD balance`. |
| `Loans` | `LN balance` (outstanding). |
| `TotalBalance` | `Deposits + Loans` — total relationship footings. |
| `Has_DD / Has_CD / Has_LN` | active accounts of that family > 0. |
| `ProductFamilies` | number of families held (0–3). |
| `ProductMix` | which families: `DD`, `DD+CD`, `DD+LN`, `DD+CD+LN`, `CD`, `CD+LN`, `LN`, `None`. |
| `AgeBand`, `BusinessAgeBand`, `TenureBand`, `DepositTier` | fixed bands (see code). |
| `Region` | country → region map (unmapped countries are printed — extend the map). |
| `OtherAccountsActive`, `Has_Other` | from §4.3, clipped at 0 for charts; flag = holds account types not in the extract. |
| `ForeignSplit` | `Domestic` / `Venezuela` / `Other foreign` — the foreign book is dominated by one country, so it gets its own level. |
| `RelationshipStart`, `RelationshipYears`, `CohortYear` | earlier of CIF creation and first account; cohort year = CIF creation year (the bank's definition). |
| `ValueBand`, `TopDecile` | rank of every CIF by `TotalBalance`: Top 1% / Next 9% / Next 40% / Bottom 50%. |
| `ClosureRatio`, `MonthsSinceLastClose` | closed ÷ total accounts ever; months since the last closure (blank = never closed). |

In [ ]:
df['Select'] = np.where(is_(df['Banesco Select'], 'S'), 'Select', 'Non-Select')
print('Banesco Select raw values:', df['Banesco Select'].fillna('(blank)').value_counts().to_dict(), '→ Select:', df['Select'].value_counts().to_dict())

df['Age'] = df['AGE'].where(ind)
df.loc[df['Age'] > 105, 'Age'] = np.nan
df['BusinessAge'] = df['AGE'].where(biz)

df['Deposits']     = df['DD Accounts Balance'].fillna(0) + df['CD Accounts Balance'].fillna(0)
df['Loans']        = df['LN Accounts Balance'].fillna(0)
df['TotalBalance'] = df['Deposits'] + df['Loans']
for f in FAM:
    df[f'Has_{f}'] = df[f'{f} Accounts Active'].gt(0)
df['ProductFamilies'] = df[[f'Has_{f}' for f in FAM]].sum(axis=1).astype(int)
df['ProductMix'] = df[[f'Has_{f}' for f in FAM]].apply(lambda r: '+'.join([f for f in FAM if r[f'Has_{f}']]) or 'None', axis=1)
MIX_ORDER = ['DD', 'DD+CD', 'DD+LN', 'DD+CD+LN', 'CD', 'CD+LN', 'LN', 'None']

AGE_BANDS   = ['<25', '25–34', '35–44', '45–54', '55–64', '65–74', '75+']
YEAR_BANDS  = ['<1', '1–2', '3–5', '6–10', '11–20', '20+']
TIER_BANDS  = ['≤ $0', '$0–10K', '$10–50K', '$50–250K', '$250K–1M', '$1M+']
df['AgeBand']         = pd.cut(df['Age'], [0, 25, 35, 45, 55, 65, 75, np.inf], right=False, labels=AGE_BANDS).astype(object)
df['BusinessAgeBand'] = pd.cut(df['BusinessAge'], [0, 1, 3, 6, 11, 21, np.inf], right=False, labels=YEAR_BANDS).astype(object)
df['TenureBand']      = pd.cut(df['CIF_Tenure_Years'], [0, 1, 3, 6, 11, 21, np.inf], right=False, labels=YEAR_BANDS).astype(object)
df['DepositTier']     = pd.cut(df['Deposits'], [-np.inf, 0, 10e3, 50e3, 250e3, 1e6, np.inf], labels=TIER_BANDS).astype(object)

# ---- Country → Region ----
REGION = {
 'North America': ['UNITED STATES', 'UNITED STATES OF AMERICA', 'USA', 'US', 'PUERTO RICO', 'CANADA', 'VIRGIN ISLANDS (U.S.)', 'U.S. VIRGIN ISLANDS'],
 'Mexico & Central America': ['MEXICO', 'GUATEMALA', 'EL SALVADOR', 'HONDURAS', 'NICARAGUA', 'COSTA RICA', 'PANAMA', 'BELIZE'],
 'South America': ['VENEZUELA', 'COLOMBIA', 'ECUADOR', 'PERU', 'BOLIVIA', 'CHILE', 'ARGENTINA', 'URUGUAY', 'PARAGUAY', 'BRAZIL', 'GUYANA', 'SURINAME',
                   'VENEZUELA, BOLIVARIAN REPUBLIC OF', 'BOLIVARIAN REPUBLIC OF VENEZUELA', 'BOLIVIA, PLURINATIONAL STATE OF'],
 'Caribbean': ['DOMINICAN REPUBLIC', 'CUBA', 'HAITI', 'JAMAICA', 'BAHAMAS', 'TRINIDAD AND TOBAGO', 'BARBADOS', 'CAYMAN ISLANDS', 'BRITISH VIRGIN ISLANDS',
               'VIRGIN ISLANDS, BRITISH', 'ARUBA', 'CURACAO', 'CURAÇAO', 'SINT MAARTEN', 'SAINT MARTIN', 'BERMUDA', 'TURKS AND CAICOS ISLANDS', 'ANTIGUA AND BARBUDA',
               'SAINT KITTS AND NEVIS', 'SAINT LUCIA', 'SAINT VINCENT AND THE GRENADINES', 'GRENADA', 'DOMINICA', 'ANGUILLA', 'MONTSERRAT', 'BONAIRE'],
 'Europe': ['SPAIN', 'ITALY', 'FRANCE', 'GERMANY', 'UNITED KINGDOM', 'PORTUGAL', 'SWITZERLAND', 'NETHERLANDS', 'BELGIUM', 'IRELAND', 'SWEDEN', 'NORWAY', 'DENMARK',
            'FINLAND', 'AUSTRIA', 'GREECE', 'POLAND', 'LUXEMBOURG', 'MONACO', 'ANDORRA', 'MALTA', 'CYPRUS', 'CZECH REPUBLIC', 'CZECHIA', 'HUNGARY', 'ROMANIA', 'UKRAINE',
            'RUSSIA', 'RUSSIAN FEDERATION', 'GIBRALTAR', 'ISLE OF MAN', 'JERSEY', 'GUERNSEY', 'LIECHTENSTEIN', 'CROATIA', 'SLOVAKIA', 'SLOVENIA', 'BULGARIA', 'SERBIA', 'ESTONIA', 'LATVIA', 'LITHUANIA', 'ICELAND'],
 'Middle East & Africa': ['ISRAEL', 'UNITED ARAB EMIRATES', 'SAUDI ARABIA', 'QATAR', 'KUWAIT', 'BAHRAIN', 'TURKEY', 'TÜRKIYE', 'LEBANON', 'JORDAN', 'EGYPT', 'MOROCCO',
                          'SOUTH AFRICA', 'NIGERIA', 'KENYA', 'GHANA', 'ANGOLA', 'TUNISIA', 'ALGERIA'],
 'Asia & Oceania': ['CHINA', 'HONG KONG', 'HONG KONG SAR', 'HONG KONG SAR CHINA', 'JAPAN', 'INDIA', 'SINGAPORE', 'SOUTH KOREA', 'KOREA, REPUBLIC OF', 'TAIWAN', 'PHILIPPINES',
                    'THAILAND', 'VIETNAM', 'MALAYSIA', 'INDONESIA', 'PAKISTAN', 'AUSTRALIA', 'NEW ZEALAND', 'MACAU', 'MACAO', 'KAZAKHSTAN', 'MARSHALL ISLANDS', 'VANUATU', 'SAMOA',
                    'COOK ISLANDS', 'FIJI', 'BANGLADESH', 'SRI LANKA', 'NEPAL'],
}
REGION['Caribbean'] += ['ST. KITTS AND NEVIS', 'ST KITTS AND NEVIS', 'ST. LUCIA', 'ST LUCIA', 'ST. VINCENT AND THE GRENADINES', 'TURKS & CAICOS', 'TURKS AND CAICOS',
                        'BAHAMAS, THE', 'THE BAHAMAS', 'NETHERLANDS ANTILLES', 'SAINT BARTHELEMY', 'ST. BARTHELEMY', 'MARTINIQUE', 'GUADELOUPE', 'ANTIGUA', 'VIRGIN ISLANDS, U.S.']
REGION['South America'] += ['FRENCH GUIANA', 'VENEZUELA (BOLIVARIAN REPUBLIC OF)']
REGION['Europe'] += ['HOLLAND', 'GREAT BRITAIN', 'ENGLAND', 'SCOTLAND', 'UK', 'U.K.', 'CZECH REP.', 'SAN MARINO', 'VATICAN CITY', 'ALBANIA', 'MONTENEGRO', 'BOSNIA AND HERZEGOVINA', 'NORTH MACEDONIA', 'MOLDOVA', 'BELARUS', 'GEORGIA', 'ARMENIA', 'AZERBAIJAN']
REGION['Middle East & Africa'] += ['IRAN', 'IRAQ', 'SYRIA', 'OMAN', 'YEMEN', 'ETHIOPIA', 'TANZANIA', 'UGANDA', 'SENEGAL', 'IVORY COAST', "COTE D'IVOIRE", 'CAMEROON', 'CONGO', 'ZIMBABWE', 'ZAMBIA', 'MOZAMBIQUE', 'BOTSWANA', 'NAMIBIA', 'MAURITIUS', 'SEYCHELLES', 'LIBYA', 'SUDAN', 'CABO VERDE', 'CAPE VERDE', 'EQUATORIAL GUINEA']
country_to_region = {c: r for r, lst in REGION.items() for c in lst}
def region_of(c):
    if not isinstance(c, str): return '(missing)'
    return country_to_region.get(c.strip().upper(), 'Other / unmapped')
df['Region'] = df['Country'].map(region_of)
REGION_ORDER = [r for r in list(REGION) + ['Other / unmapped', '(missing)'] if r in set(df['Region'])]
unmapped = df.loc[df['Region'].eq('Other / unmapped'), 'Country'].value_counts()
if len(unmapped): print(f'\n⚠ {len(unmapped)} countries not in the region map (add them to REGION): {unmapped.head(20).to_dict()}')

df['Group2'] = make_label(df, ['IorB', 'ForD'])
df['Group3'] = make_label(df, ['IorB', 'ForD', 'Select'])
GROUP2_ORDER = group_spec(['IorB', 'ForD'])[0]
GROUP3_ORDER = [a + SEP + b + SEP + c for a in ORDER['IorB'] for b in ORDER['ForD'] for c in ORDER['Select']]
df['OtherAccountsActive_c'] = df['OtherAccountsActive'].clip(lower=0)
df['Has_Other'] = df['OtherAccountsActive_c'].gt(0)

# ---- Venezuela vs the rest of the foreign book ----
df['ForeignSplit'] = np.where(is_(df['ForD'], 'Domestic'), 'Domestic', np.where(is_(df['Country'], 'VENEZUELA'), 'Venezuela', 'Other foreign'))

# ---- Relationship start = earlier of CIF creation and first account; cohort year = CIF creation year (the bank's definition) ----
df['RelationshipStart'] = df[['effectiveCIFCreationDate', 'FirstAccountCreationDate']].min(axis=1)
df['RelationshipYears'] = (SNAPSHOT - df['RelationshipStart']).dt.days / 365.25
df['CohortYear'] = df['effectiveCIFCreationDate'].dt.year
print(f'Tenure understated for {(df["RelationshipYears"] - df["CIF_Tenure_Years"] > 1).mean():.1%} of CIFs by more than a year (first account predates CIF creation) — RelationshipYears keeps the earlier date')

# ---- Value bands by total balance (rank over all active CIFs) ----
pr = df['TotalBalance'].rank(ascending=False, method='first') / len(df)
df['ValueBand'] = np.select([pr <= 0.01, pr <= 0.10, pr <= 0.50], ['Top 1%', 'Next 9%', 'Next 40%'], 'Bottom 50%')
df['TopDecile'] = pr <= 0.10
df['log_Deposits'] = np.log1p(df['Deposits'].clip(lower=0)); df['log_Loans'] = np.log1p(df['Loans'].clip(lower=0)); df['log_Total'] = np.log1p(df['TotalBalance'].clip(lower=0))

# ---- Closure signals ----
df['ClosureRatio'] = (df['ClosedAccountsCount'] / df['TotalAccountsEver'].replace(0, np.nan)).fillna(0)
df['MonthsSinceLastClose'] = (SNAPSHOT - df['LastAccountCloseDate']).dt.days / 30.44

# ---- Lists used by the analysis ----
HEADLINE_BAL = ['Deposits', 'Loans', 'TotalBalance']
PRODUCT_BAL  = BAL_COLS
LABEL = {'DD Accounts Balance': 'DDA balance', 'CD Accounts Balance': 'CD balance', 'LN Accounts Balance': 'Loan balance',
         'Deposits': 'Deposits (DD + CD)', 'Loans': 'Loans (LN)', 'TotalBalance': 'Total balance (deposits + loans)',
         'CIF_Tenure_Years': 'Tenure (years)', 'Age': 'Age (Individuals)', 'BusinessAge': 'Business age (years since incorporation)',
         'ActiveAccountsCount': 'Active accounts', 'ProductFamilies': 'Product families held'}
NUM_ANALYSIS = ['Age', 'BusinessAge', 'CIF_Tenure_Years', 'ActiveAccountsCount', 'ProductFamilies', 'Deposits', 'Loans', 'TotalBalance'] + BAL_COLS
print(f'\nFeatures ready · {len(df):,} active CIFs · {df.shape[1]} columns')

if EXPORT_CLEAN:
    df.to_csv(CLEAN_PATH, index=False)
    print(f'Clean CIF-level table exported → {CLEAN_PATH}')

## 5.1 Analysis helpers

The segment-profile table and the three standard figures (demographics, product depth, balances) are defined once and reused for **every** granularity in §7.

In [ ]:
def profile_table(d, dims):
    """One row per segment of `dims`: size, demographics, product depth, deposits and loans."""
    g = d.groupby(dims, dropna=False)
    t = pd.DataFrame({'CIFs': g.size()})
    t['% CIFs'] = t['CIFs'] / t['CIFs'].sum() * 100
    t['Median age (Ind.)'] = g['Age'].median()
    t['Median business age'] = g['BusinessAge'].median()
    t['Median tenure (y)'] = g['CIF_Tenure_Years'].median()
    t['Avg active accts'] = g['ActiveAccountsCount'].mean()
    t['Avg families'] = g['ProductFamilies'].mean()
    for f in FAM: t[f'% has {f}'] = g[f'Has_{f}'].mean() * 100
    if 'Select' not in dims: t['% Select'] = g['Select'].apply(lambda s: is_(s, 'Select').mean() * 100)
    if 'ForD' not in dims:   t['% Foreign'] = g['ForD'].apply(lambda s: is_(s, 'Foreign').mean() * 100)
    t['Deposits $'] = g['Deposits'].sum(); t['% Deposits'] = t['Deposits $'] / t['Deposits $'].sum() * 100
    t['Deposit holders'] = g['Deposits'].apply(lambda s: int((s > 0).sum()))
    t['Median deposits (holders)'] = g['Deposits'].apply(lambda s: s[s > 0].median())
    t['Loans $'] = g['Loans'].sum(); t['% Loans'] = t['Loans $'] / t['Loans $'].sum() * 100 if t['Loans $'].sum() else np.nan
    t['Loan holders'] = g['Loans'].apply(lambda s: int((s > 0).sum()))
    t['Median loans (holders)'] = g['Loans'].apply(lambda s: s[s > 0].median())
    if len(dims) == 1: idx = pd.Index(ORDER[dims[0]], name=dims[0])
    else: idx = pd.MultiIndex.from_product([ORDER[c] for c in dims], names=dims)
    return t.reindex(idx).dropna(subset=['CIFs'])

PCT_COLS   = ['% CIFs', '% Select', '% Foreign', '% Deposits', '% Loans', '% has DD', '% has CD', '% has LN']
MONEY_COLS = ['Deposits $', 'Median deposits (holders)', 'Loans $', 'Median loans (holders)']
ONE_DEC    = ['Median age (Ind.)', 'Median business age', 'Median tenure (y)', 'Avg active accts', 'Avg families']
def show_profile(t): display(styled(t, pct_cols=PCT_COLS, money_cols=MONEY_COLS, int_cols=['CIFs', 'Deposit holders', 'Loan holders'], one_dec=ONE_DEC, gradient_cols=['% CIFs', '% Deposits', '% Loans']))

def fig_demographics(d, dims, name=None):
    cdims, facet = split_dims(dims)
    order, pal = group_spec(cdims)
    d = d.assign(_grp=make_label(d, cdims))
    facets = ORDER[facet] if facet else [None]
    panels = [('Age', 'Age — Individuals', (0, 100), is_(d['IorB'], 'Individual')),
              ('BusinessAge', 'Business age (years since incorporation) — Business', (0, 60), is_(d['IorB'], 'Business')),
              ('CIF_Tenure_Years', 'CIF tenure (years) — all', (0, 45), pd.Series(True, index=d.index))]
    if facet: nrows, ncols = len(panels), len(facets)
    else:     nrows, ncols = 1, len(panels)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.3 * ncols, 3.3 * nrows), squeeze=False)
    for i, (col, ttl, clip, mask) in enumerate(panels):
        for j, fv in enumerate(facets):
            ax = axes[i, j] if facet else axes[0, i]
            m = mask & (is_(d[facet], fv) if fv is not None else True)
            kde_by(ax, d[m], col, '_grp', order, pal, clip=clip)
            style(ax, title=ttl + (f'  ·  {fv}' if fv is not None else ''), xlabel='years')
    finish(fig, f'Demographics & tenure by {" × ".join(dims)}', 'Density curves; dashed line = median. Age uses Individuals only, business age uses Business only.', name=name)

def fig_counts(d, dims, name=None):
    cdims, facet = split_dims(dims)
    order, pal = group_spec(cdims)
    d = d.assign(_grp=make_label(d, cdims))
    facets = ORDER[facet] if facet else [None]
    acc = d['ActiveAccountsCount'].clip(upper=6).map(lambda v: '6+' if v >= 6 else f'{int(v)}' if not pd.isna(v) else v)
    ACC_ORDER = ['0', '1', '2', '3', '4', '5', '6+']
    panels = [('Active accounts per CIF', acc, ACC_ORDER), ('Product families held (DD / CD / LN)', d['ProductFamilies'].astype(str), ['0', '1', '2', '3']),
              ('Product mix', d['ProductMix'], MIX_ORDER)]
    if facet: nrows, ncols = len(panels), len(facets)
    else:     nrows, ncols = 1, len(panels)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.3 * ncols, 3.3 * nrows), squeeze=False)
    for i, (ttl, series, cat_order) in enumerate(panels):
        for j, fv in enumerate(facets):
            ax = axes[i, j] if facet else axes[0, i]
            m = is_(d[facet], fv) if fv is not None else pd.Series(True, index=d.index)
            ct = pd.crosstab(series[m], d.loc[m, '_grp'], normalize='columns') * 100
            ct = ct.reindex([c for c in cat_order if c in ct.index])
            grouped_bars(ax, ct, order, pal)
            style(ax, title=ttl + (f'  ·  {fv}' if fv is not None else ''), ylabel='% of CIFs in group', pct_y=True)
            if ttl == 'Product mix': ax.tick_params(axis='x', labelrotation=30)
    finish(fig, f'Product depth by {" × ".join(dims)}', 'Share of CIFs in each group. "Product families" counts only DD / CD / LN; other account types are not in the extract.', name=name)

def fig_balances(d, dims, cols, name=None, title='Balances'):
    cdims, facet = split_dims(dims)
    order, pal = group_spec(cdims)
    d = d.assign(_grp=make_label(d, cdims))
    facets = ORDER[facet] if facet else [None]
    nrows, ncols = len(facets), len(cols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.4 * ncols, 0.9 + 0.55 * len(order) * nrows + 1.6 * nrows), squeeze=False)
    for j, fv in enumerate(facets):
        for i, col in enumerate(cols):
            ax = axes[j, i]
            m = is_(d[facet], fv) if fv is not None else pd.Series(True, index=d.index)
            box_by(ax, d[m], col, '_grp', order, pal)
            style(ax, title=LABEL.get(col, col) + (f'  ·  {fv}' if fv is not None else ''))
            if i > 0: ax.set_yticklabels([])
    finish(fig, f'{title} by {" × ".join(dims)}', 'Among holders (balance > 0), log scale, no outliers drawn. Label = median; right = share of the group that holds the product.', name=name)
print('Helpers ready')

# 6. Portfolio overview — the customer base on one page

In [ ]:
n = len(df); ind_n = ind.sum(); biz_n = biz.sum()
dep_hold = df.loc[df['Deposits'] > 0, 'Deposits']
kpis = [
    ('Active CIFs', f'{n:,}', f'{ind_n:,} individuals · {biz_n:,} businesses', BLUE),
    ('Individuals', f'{ind_n / n:.0%}', f'{is_(df["ForD"], "Foreign")[ind].mean():.0%} of them foreign', BLUE),
    ('Businesses', f'{biz_n / n:.0%}', f'{is_(df["ForD"], "Foreign")[biz].mean():.0%} of them foreign', GREEN),
    ('Select', f'{is_(df["Select"], "Select").mean():.0%}', f'{is_(df["Select"], "Select")[ind].mean():.0%} of individuals · {is_(df["Select"], "Select")[biz].mean():.0%} of businesses', GREEN),
    ('Total deposits (DD + CD)', money(df['Deposits'].sum()), f'{df["Deposits"].gt(0).mean():.0%} of CIFs hold deposits', BLUE),
    ('Total loans (LN)', money(df['Loans'].sum()), f'{df["Loans"].gt(0).mean():.0%} of CIFs hold loans', GREEN),
    ('Median deposits (holders)', money(dep_hold.median()), f'mean {money(dep_hold.mean())} — heavy tail', BLUE),
    ('Countries', f'{df["Country"].nunique():,}', f'{is_(df["ForD"], "Foreign").mean():.0%} of CIFs are foreign', GREEN),
    ('Median age (individuals)', f'{df["Age"].median():.0f}', f'P10–P90: {df["Age"].quantile(.1):.0f}–{df["Age"].quantile(.9):.0f}', BLUE),
    ('Median business age', f'{df["BusinessAge"].median():.0f} y', 'years since incorporation', GREEN),
    ('Median tenure', f'{df["CIF_Tenure_Years"].median():.1f} y', f'individuals {df.loc[ind, "CIF_Tenure_Years"].median():.1f} · business {df.loc[biz, "CIF_Tenure_Years"].median():.1f}', BLUE),
    ('Single-family CIFs', f'{df["ProductFamilies"].le(1).mean():.0%}', 'hold at most one of DD / CD / LN', GREEN),
]
ncols = 4; nrows = math.ceil(len(kpis) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(3.3 * ncols, 1.45 * nrows))
for ax, it in zip(axes.flat, kpis + [None] * (nrows * ncols - len(kpis))):
    ax.axis('off')
    if it is None: continue
    label, value, sub, color = it
    ax.add_patch(mpl.patches.FancyBboxPatch((0.03, 0.06), 0.94, 0.88, boxstyle='round,pad=0.01,rounding_size=0.05', fc=INK['card'], ec=INK['grid'], transform=ax.transAxes))
    ax.plot([0.055, 0.055], [0.18, 0.82], color=color, lw=4, transform=ax.transAxes, solid_capstyle='round')
    ax.text(0.12, 0.76, label.upper(), fontsize=7.6, color=INK['muted'], transform=ax.transAxes, va='center')
    ax.text(0.12, 0.47, value, fontsize=16, fontweight='bold', color=INK['primary'], transform=ax.transAxes, va='center')
    ax.text(0.12, 0.2, sub, fontsize=7.6, color=INK['secondary'], transform=ax.transAxes, va='center')
finish(fig, 'Active customer base at a glance', f'Snapshot {SNAPSHOT.date()} · {n:,} active CIFs', name='06_kpis')

## 6.1 The 8-cell grid: IorB × ForD × Select

Counts say who the customers are; deposits and loans say where the money is. The two rarely agree — that gap is the story.

In [ ]:
grid = profile_table(df, ['IorB', 'ForD', 'Select'])
show_profile(grid)

labels = [SEP.join(ix) for ix in grid.index]
order4, pal4 = group_spec(['IorB', 'ForD'])
colors = [pal4[SEP.join(ix[:2])] for ix in grid.index]
hatches = ['' if ix[2] == 'Select' else '///' for ix in grid.index]
fig, axes = plt.subplots(1, 3, figsize=(15, 0.8 + 0.5 * len(grid)), sharey=True)
for ax, col, ttl in zip(axes, ['% CIFs', '% Deposits', '% Loans'], ['Share of CIFs', 'Share of deposits', 'Share of loans']):
    hbar(ax, labels, grid[col].values, colors, fmt=fmt_pct, hatches=hatches)
    style(ax, title=ttl); ax.xaxis.set_major_formatter(PCT0)
for ax in axes[1:]: ax.tick_params(labelleft=False)
finish(fig, 'Who the customers are vs. where the money is', 'Color = IorB × ForD · hatched = Non-Select. Each panel sums to 100%.', name='06_grid')

In [ ]:
# Composition of CIFs, deposits and loans by each primary dimension
fig, axes = plt.subplots(1, 3, figsize=(15, 3.2))
for ax, dim in zip(axes, ['IorB', 'ForD', 'Select']):
    order, pal = group_spec([dim])
    comp = pd.DataFrame({'CIFs': df.groupby(dim).size(), 'Deposits': df.groupby(dim)['Deposits'].sum(), 'Loans': df.groupby(dim)['Loans'].sum()}).reindex(order)
    comp = comp.div(comp.sum(axis=0), axis=1) * 100
    left = np.zeros(comp.shape[1])
    for g in order:
        vals = comp.loc[g].values
        ax.barh(comp.columns, vals, left=left, color=pal[g], label=g, height=0.55, edgecolor=INK['surface'], linewidth=1)
        for i, (l, v) in enumerate(zip(left, vals)):
            if v >= 7: ax.text(l + v / 2, i, f'{v:.0f}%', ha='center', va='center', fontsize=8.5, color='white')
        left += vals
    ax.invert_yaxis(); ax.set_xlim(0, 100); ax.xaxis.set_major_formatter(PCT0); ax.grid(axis='y', visible=False); ax.grid(axis='x', color=INK['grid'])
    style(ax, title=f'By {dim}'); ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.35), ncol=2)
finish(fig, 'Composition: customers vs. money', 'Each bar sums to 100%', name='06_composition')

**Observations §6** *(fill in after the run)*

- **Customers:** …
- **Money:** …
- **Biggest gaps between share of CIFs and share of money:** …

# 7. Segment deep-dive — every granularity × every feature

Seven granularities are run through the same three figures and the same profile table, so segments are always compared on identical axes:

`IorB` · `ForD` · `Select` · `IorB × ForD` · `IorB × Select` · `ForD × Select` · `IorB × ForD × Select`

For each: **(1)** profile table, **(2)** demographics & tenure, **(3)** product depth, **(4)** headline balances (deposits, loans, total), **(5)** product balances (DD, CD, LN). Color encodes up to two dimensions; the third becomes side-by-side panels.

In [ ]:
GRANULARITIES = [['IorB'], ['ForD'], ['Select'], ['IorB', 'ForD'], ['IorB', 'Select'], ['ForD', 'Select'], ['IorB', 'ForD', 'Select']]

for dims in GRANULARITIES:
    tag = ' × '.join(dims); slug = '07_' + '_'.join(dims)
    display(Markdown(f'### {tag}'))
    show_profile(profile_table(df, dims))
    fig_demographics(df, dims, name=f'{slug}_demographics')
    fig_counts(df, dims, name=f'{slug}_products')
    fig_balances(df, dims, HEADLINE_BAL, name=f'{slug}_balances', title='Headline balances')
    fig_balances(df, dims, PRODUCT_BAL, name=f'{slug}_product_balances', title='Product balances')

**Observations §7** *(fill in after the run)*

- **IorB:** …
- **ForD:** …
- **Select:** …
- **IorB × ForD:** …
- **IorB × Select:** …
- **ForD × Select:** …
- **IorB × ForD × Select:** …

# 8. Age

Age of **individuals** and age of **businesses** (years since incorporation) are different things, so they are shown separately. Bands make the comparison across segments easy; medians are on every curve.

In [ ]:
d_ind = df[ind]
fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
for ax, dims in zip(axes, [['ForD'], ['Select'], ['ForD', 'Select']]):
    order, pal = group_spec(dims)
    kde_by(ax, d_ind.assign(_grp=make_label(d_ind, dims)), 'Age', '_grp', order, pal, clip=(0, 100))
    style(ax, title='Individuals · age by ' + ' × '.join(dims), xlabel='years')
finish(fig, 'Age distribution of individual customers', 'Density; dashed = median', name='08_age_ind')

# Age bands: composition within each group (sequential blues = ordinal bands)
grp = make_label(d_ind, ['ForD', 'Select']); order4, _ = group_spec(['ForD', 'Select'])
ct = pd.crosstab(grp, d_ind['AgeBand'], normalize='index').reindex(index=order4, columns=AGE_BANDS).fillna(0) * 100
fig, ax = plt.subplots(figsize=(11, 3.2))
left = np.zeros(len(ct))
for band, colr in zip(AGE_BANDS, BLUES_7):
    vals = ct[band].values
    ax.barh(ct.index, vals, left=left, color=colr, label=band, height=0.6, edgecolor=INK['surface'], linewidth=1)
    for i, (l, v) in enumerate(zip(left, vals)):
        if v >= 6: ax.text(l + v / 2, i, f'{v:.0f}%', ha='center', va='center', fontsize=8, color='white' if band in AGE_BANDS[3:] else INK['primary'])
    left += vals
ax.invert_yaxis(); ax.set_xlim(0, 100); ax.xaxis.set_major_formatter(PCT0); ax.grid(axis='y', visible=False)
ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.5), ncol=7, title='Age band', title_fontsize=8.5)
finish(fig, 'Individuals · age-band mix by ForD × Select', name='08_age_bands')

age_tbl = pd.crosstab(d_ind['AgeBand'], [d_ind['ForD'], d_ind['Select']], margins=True, margins_name='All').reindex(AGE_BANDS + ['All'])
print('Individuals by age band (counts):'); display(age_tbl)

In [ ]:
# Age vs. money — median deposits among holders and share of Select, by age band
d_ind = df[ind]
fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
order2, pal2 = group_spec(['ForD'])
med, lo, hi = median_table(d_ind, 'Deposits', 'AgeBand', 'ForD', AGE_BANDS, order2, holders=True)
grouped_bars(axes[0], med, order2, pal2, fmt=money, lo=lo, hi=hi); style(axes[0], title='Median deposits among holders (95% CI)', money_y=True); axes[0].set_ylim(0, np.nanmax(hi.values) * 1.18)
sel = d_ind.groupby(['AgeBand', 'ForD'])['Select'].apply(lambda s: is_(s, 'Select').mean() * 100).unstack().reindex(index=AGE_BANDS, columns=order2)
grouped_bars(axes[1], sel, order2, pal2, fmt=lambda v: f'{v:.0f}%'); style(axes[1], title='Share of Select', pct_y=True); axes[1].set_ylim(0, 100)
ln = d_ind.groupby(['AgeBand', 'ForD'])['Has_LN'].mean().unstack().reindex(index=AGE_BANDS, columns=order2) * 100
grouped_bars(axes[2], ln, order2, pal2, fmt=lambda v: f'{v:.0f}%'); style(axes[2], title='Share with a loan', pct_y=True); axes[2].set_ylim(0, max(ln.max().max() * 1.3, 5))
finish(fig, 'Individuals · what changes with age', 'By age band and ForD · error bars = 95% bootstrap CI of the median', name='08_age_money')

In [ ]:
# Age pyramids: Domestic (left) vs Foreign (right). Each side sums to 100% so the shapes are comparable.
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (col, bands, ttl, sub) in zip(axes, [('AgeBand', AGE_BANDS, 'Individuals · age', df[ind]), ('BusinessAgeBand', YEAR_BANDS, 'Business · years since incorporation', df[biz])]):
    ct = pd.crosstab(sub[col], sub['ForD']).reindex(index=bands, columns=ORDER['ForD']).fillna(0)
    pct = ct / ct.sum(axis=0).replace(0, np.nan) * 100
    y = np.arange(len(bands))
    ax.barh(y, -pct['Domestic'].fillna(0).values, color=BLUE, height=0.7, label=f'Domestic (n={int(ct["Domestic"].sum()):,})', edgecolor=INK['surface'])
    ax.barh(y, pct['Foreign'].fillna(0).values, color=GREEN, height=0.7, label=f'Foreign (n={int(ct["Foreign"].sum()):,})', edgecolor=INK['surface'])
    for i, b in enumerate(bands):
        pd_, pf = pct.loc[b, 'Domestic'], pct.loc[b, 'Foreign']
        if not np.isnan(pd_): ax.text(-pd_ - 0.6, i, f'{pd_:.0f}% · {int(ct.loc[b, "Domestic"]):,}', ha='right', va='center', fontsize=8, color=INK['secondary'])
        if not np.isnan(pf):  ax.text(pf + 0.6, i, f'{pf:.0f}% · {int(ct.loc[b, "Foreign"]):,}', ha='left', va='center', fontsize=8, color=INK['secondary'])
    m = max(np.nanmax(pct.values) * 1.5, 10)
    ax.set_xlim(-m, m); ax.set_yticks(y); ax.set_yticklabels(bands); ax.axvline(0, color=INK['grid'])
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, p: f'{abs(v):.0f}%')); ax.grid(axis='y', visible=False)
    style(ax, title=ttl, xlabel='% of each side'); ax.legend(loc='lower right')
finish(fig, 'Age pyramids · Domestic vs Foreign', 'Labels = share of that side and count', name='08_pyramid')

# Deposit tier by age band — is the wealth effect of age the same at home and abroad?
d_ind = df[ind]
tier_colors = [CMAP_BLUE(x) for x in np.linspace(0.12, 1.0, len(TIER_BANDS))]
fig, axes = plt.subplots(1, 2, figsize=(14, 3.6))
for ax, fd in zip(axes, ORDER['ForD']):
    sub = d_ind[is_(d_ind['ForD'], fd)]
    ct = pd.crosstab(sub['AgeBand'], sub['DepositTier'], normalize='index').reindex(index=AGE_BANDS, columns=TIER_BANDS).fillna(0) * 100
    left = np.zeros(len(ct))
    for tier, colr in zip(TIER_BANDS, tier_colors):
        vals = ct[tier].values
        ax.barh(ct.index, vals, left=left, color=colr, label=tier, height=0.65, edgecolor=INK['surface'], linewidth=1)
        for i, (l, v) in enumerate(zip(left, vals)):
            if v >= 7: ax.text(l + v / 2, i, f'{v:.0f}%', ha='center', va='center', fontsize=7.8, color=text_on(colr))
        left += vals
    ax.invert_yaxis(); ax.set_xlim(0, 100); ax.xaxis.set_major_formatter(PCT0); ax.grid(axis='y', visible=False)
    style(ax, title=f'Individuals · {fd} · deposit tier by age band')
axes[1].tick_params(labelleft=False)
axes[1].legend(loc='lower center', bbox_to_anchor=(-0.1, -0.45), ncol=6, title='Deposit tier', title_fontsize=8.5)
finish(fig, 'Deposit tier by age band', 'Share of individual CIFs in each deposit tier', name='08_age_tier')

In [ ]:
d_biz = df[biz]
fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
for ax, dims in zip(axes, [['ForD'], ['Select'], ['ForD', 'Select']]):
    order, pal = group_spec(dims)
    kde_by(ax, d_biz.assign(_grp=make_label(d_biz, dims)), 'BusinessAge', '_grp', order, pal, clip=(0, 60))
    style(ax, title='Business · years since incorporation by ' + ' × '.join(dims), xlabel='years')
finish(fig, 'Business age (years since incorporation)', 'Density; dashed = median', name='08_age_biz')

grp = make_label(d_biz, ['ForD', 'Select']); order4, _ = group_spec(['ForD', 'Select'])
ct = pd.crosstab(grp, d_biz['BusinessAgeBand'], normalize='index').reindex(index=order4, columns=YEAR_BANDS).fillna(0) * 100
fig, ax = plt.subplots(figsize=(11, 3.2))
left = np.zeros(len(ct))
for band, colr in zip(YEAR_BANDS, [CMAP_GREEN(x) for x in np.linspace(0.2, 1.0, 6)]):
    vals = ct[band].values
    ax.barh(ct.index, vals, left=left, color=colr, label=band, height=0.6, edgecolor=INK['surface'], linewidth=1)
    for i, (l, v) in enumerate(zip(left, vals)):
        if v >= 6: ax.text(l + v / 2, i, f'{v:.0f}%', ha='center', va='center', fontsize=8, color='white' if band in YEAR_BANDS[3:] else INK['primary'])
    left += vals
ax.invert_yaxis(); ax.set_xlim(0, 100); ax.xaxis.set_major_formatter(PCT0); ax.grid(axis='y', visible=False)
ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.5), ncol=6, title='Years since incorporation', title_fontsize=8.5)
finish(fig, 'Business · age-band mix by ForD × Select', name='08_bizage_bands')
med = d_biz[d_biz['Deposits'] > 0].groupby(['BusinessAgeBand', 'ForD'])['Deposits'].median().unstack().reindex(index=YEAR_BANDS)
print('Business · median deposits among holders by age band × ForD:'); display(med.map(money))

**Observations §8** *(fill in after the run)*

- **Individuals:** …
- **Age and money:** …
- **Businesses:** …

# 9. Concentration & value — who holds the money

Everything here is computed **among holders** (balance > 0): only a minority of CIFs have loans, so "top 10% of CIFs hold 100% of loans" would only restate that. Then the whole base is ranked by total balance into value bands, the top 100 relationships are profiled, and the tail above fixed thresholds is sized.

In [ ]:
def lorenz(s):
    s = np.sort(np.asarray(s, dtype=float).clip(min=0))[::-1]
    if s.sum() <= 0: return None, None
    return np.arange(1, len(s) + 1) / len(s) * 100, np.cumsum(s) / s.sum() * 100
def gini(s):
    s = np.sort(np.asarray(s, dtype=float).clip(min=0))
    if s.sum() <= 0: return np.nan
    n = len(s); return float((2 * np.sum(np.arange(1, n + 1) * s) / (n * s.sum())) - (n + 1) / n)
def top_share(s, p):
    s = s[s > 0].sort_values(ascending=False); k = max(int(len(s) * p), 1)
    return s.iloc[:k].sum() / s.sum() * 100 if len(s) and s.sum() > 0 else np.nan
def holders_for(s, target=80):
    s = np.sort(s[s > 0].values)[::-1]
    if len(s) == 0 or s.sum() <= 0: return np.nan
    cum = np.cumsum(s) / s.sum() * 100
    return (np.searchsorted(cum, target) + 1) / len(s) * 100

order4, pal4 = group_spec(['IorB', 'ForD'])
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for ax, col in zip(axes, ['Deposits', 'Loans']):
    hold = df[df[col] > 0]
    x, y = lorenz(hold[col]); ax.plot(x, y, color=INK['muted'], lw=2, ls='--', label=f'All holders (n={len(hold):,})')
    for g in order4:
        sg = hold.loc[is_(hold['Group2'], g), col]; x, y = lorenz(sg)
        if x is not None: ax.plot(x, y, color=pal4[g], lw=2, label=f'{g} (n={len(sg):,})')
    p80, p50 = holders_for(df[col], 80), holders_for(df[col], 50)
    ax.axhline(80, color=INK['grid'], lw=1); ax.axvline(p80, color=RED, lw=1, ls=':')
    ax.annotate(f'{p80:.1f}% of holders = 80% of {col.lower()}\n{p50:.1f}% of holders = 50%', xy=(p80, 80), xytext=(min(p80 + 10, 55), 58), fontsize=8.5, color=INK['secondary'], arrowprops=dict(arrowstyle='-', color=INK['muted'], lw=0.8))
    ax.set_xlim(0, 100); ax.set_ylim(0, 101); ax.xaxis.set_major_formatter(PCT0); ax.yaxis.set_major_formatter(PCT0)
    style(ax, title=LABEL[col], xlabel='% of holders (largest first)', ylabel=f'cumulative % of {col.lower()}'); ax.legend(loc='lower right', fontsize=8)
finish(fig, 'Concentration among holders', 'Lorenz curves by IorB × ForD · red dotted line = share of holders that carries 80% of the balance', name='09_lorenz')

rows = []
for col in ['Deposits', 'Loans']:
    for g in ['All CIFs'] + order4:
        sg = df[col] if g == 'All CIFs' else df.loc[is_(df['Group2'], g), col]
        rows.append({'balance': col, 'segment': g, 'CIFs': len(sg), 'holders': int((sg > 0).sum()), '% holders': (sg > 0).mean() * 100, 'total $': sg.clip(lower=0).sum(),
                     'top 1% of holders': top_share(sg, .01), 'top 5%': top_share(sg, .05), 'top 10%': top_share(sg, .10), 'top 20%': top_share(sg, .20),
                     'holders for 80%': holders_for(sg, 80), 'Gini (holders)': gini(sg[sg > 0])})
conc = pd.DataFrame(rows).set_index(['balance', 'segment'])
display(styled(conc, pct_cols=['% holders', 'top 1% of holders', 'top 5%', 'top 10%', 'top 20%', 'holders for 80%'], money_cols=['total $'], int_cols=['CIFs', 'holders']).format({'Gini (holders)': '{:.2f}'}))
for col in ['Deposits', 'Loans']:
    sg = df[col].clip(lower=0).sort_values(ascending=False); tot = sg.sum()
    if tot > 0: print(f'{col}: top 10 CIFs = {sg.iloc[:10].sum() / tot:.1%} · top 50 = {sg.iloc[:50].sum() / tot:.1%} · top 100 = {sg.iloc[:100].sum() / tot:.1%} of the book')

## 9.1 Value pyramid — how many customers actually matter

Every active CIF ranked by total balance (deposits + loans) and cut into **Top 1% · Next 9% · Next 40% · Bottom 50%**. The left chart is the pyramid: how the customer count compares with the money. The right chart says who sits in each band.

In [ ]:
VB = ORDER['ValueBand']; vb_colors = dict(zip(VB, [CMAP_BLUE(x) for x in [1.0, 0.72, 0.42, 0.16]]))
vb = df.groupby('ValueBand').agg(CIFs=('CIF', 'size'), Deposits=('Deposits', 'sum'), Loans=('Loans', 'sum'), TotalBalance=('TotalBalance', 'sum'),
                                 EntryThreshold=('TotalBalance', 'min'), MedianBalance=('TotalBalance', 'median'), Select=('Select', lambda x: is_(x, 'Select').mean() * 100),
                                 Business=('IorB', lambda x: is_(x, 'Business').mean() * 100), Foreign=('ForD', lambda x: is_(x, 'Foreign').mean() * 100),
                                 Families=('ProductFamilies', 'mean'), Tenure=('CIF_Tenure_Years', 'median')).reindex(VB)
share = vb[['CIFs', 'Deposits', 'Loans']].div(vb[['CIFs', 'Deposits', 'Loans']].sum(axis=0).replace(0, np.nan), axis=1) * 100
vb['% CIFs'] = share['CIFs']; vb['% Deposits'] = share['Deposits']; vb['% Loans'] = share['Loans']; vb['% Total balance'] = vb['TotalBalance'] / vb['TotalBalance'].sum() * 100
vb = vb.rename(columns={'EntryThreshold': 'Entry threshold', 'MedianBalance': 'Median total balance', 'Select': '% Select', 'Business': '% Business', 'Foreign': '% Foreign', 'Families': 'Avg families', 'Tenure': 'Median tenure (y)'})
display(styled(vb[['CIFs', '% CIFs', 'Entry threshold', 'Median total balance', 'Deposits', '% Deposits', 'Loans', '% Loans', '% Total balance', '% Business', '% Foreign', '% Select', 'Avg families', 'Median tenure (y)']],
               pct_cols=['% CIFs', '% Deposits', '% Loans', '% Total balance', '% Business', '% Foreign', '% Select'], money_cols=['Entry threshold', 'Median total balance', 'Deposits', 'Loans'], int_cols=['CIFs'], one_dec=['Avg families', 'Median tenure (y)']))

fig, axes = plt.subplots(1, 2, figsize=(15, 3.6), gridspec_kw={'width_ratios': [1.1, 1]})
ax = axes[0]; left = np.zeros(3); bars_idx = ['CIFs', 'Deposits', 'Loans']
for band in VB:
    vals = share.loc[band, bars_idx].values
    ax.barh(bars_idx, vals, left=left, color=vb_colors[band], label=band, height=0.6, edgecolor=INK['surface'], linewidth=1)
    for i, (l, v) in enumerate(zip(left, vals)):
        if v >= 5: ax.text(l + v / 2, i, f'{v:.0f}%', ha='center', va='center', fontsize=8.5, color=text_on(vb_colors[band]))
    left += vals
ax.invert_yaxis(); ax.set_xlim(0, 100); ax.xaxis.set_major_formatter(PCT0); ax.grid(axis='y', visible=False)
style(ax, title='Share of customers vs. share of money, by value band'); ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.42), ncol=4)
ax = axes[1]
comp = pd.crosstab(df['ValueBand'], df['Group2'], normalize='index').reindex(index=VB, columns=order4).fillna(0) * 100
left = np.zeros(len(VB))
for g in order4:
    vals = comp[g].values
    ax.barh(VB, vals, left=left, color=pal4[g], label=g, height=0.6, edgecolor=INK['surface'], linewidth=1)
    for i, (l, v) in enumerate(zip(left, vals)):
        if v >= 7: ax.text(l + v / 2, i, f'{v:.0f}%', ha='center', va='center', fontsize=8.5, color='white')
    left += vals
ax.invert_yaxis(); ax.set_xlim(0, 100); ax.xaxis.set_major_formatter(PCT0); ax.grid(axis='y', visible=False)
style(ax, title='Who is in each band (IorB × ForD)'); ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.42), ncol=4)
finish(fig, 'Value pyramid', 'Bands = rank of every active CIF by total balance', name='09_pyramid')

# Where does each segment's population fall? (row = segment, sums to 100%)
seg_band = pd.crosstab(df['Group3'], df['ValueBand'], normalize='index').reindex(index=GROUP3_ORDER, columns=VB).fillna(0) * 100
fig, ax = plt.subplots(figsize=(9, 0.6 + 0.45 * len(seg_band)))
sns.heatmap(seg_band, annot=True, fmt='.0f', cmap=CMAP_BLUE, cbar=False, ax=ax, linewidths=1, linecolor=INK['surface'], annot_kws={'size': 9}, vmin=0, vmax=100)
for t in ax.texts: t.set_text(t.get_text() + '%')
ax.set_ylabel(''); ax.set_xlabel(''); ax.grid(False); ax.tick_params(axis='x', labelrotation=0)
finish(fig, 'Share of each segment that falls in each value band', 'Rows sum to 100%', name='09_segment_bands')

## 9.2 The top 100 relationships

With this level of concentration the top 100 depositors and the top 100 borrowers *are* the book. Who they are matters more than any average.

In [ ]:
def top_composition(col, k=100):
    t = df.nlargest(k, col)
    print(f'Top {k} by {col}: {t[col].sum() / df[col].clip(lower=0).sum():.1%} of the book · {is_(t["IorB"], "Business").mean():.0%} business · {is_(t["ForD"], "Foreign").mean():.0%} foreign · '
          f'{is_(t["Select"], "Select").mean():.0%} Select · {is_(t["Segment"], "Unassigned").sum()} with Unassigned segment · {t["BranchGroupName"].isna().sum()} without branch group')
    out = {}
    for dim, name in [('Group3', 'segment'), ('Country', 'country'), ('BranchGroupName', 'branch group'), ('OfficerCode', 'officer code'), ('Segment', 'segment code')]:
        g = t.groupby(t[dim].fillna('(missing)')).agg(CIFs=('CIF', 'size'), Amount=(col, 'sum')).sort_values('Amount', ascending=False)
        g['% of top-100 $'] = g['Amount'] / g['Amount'].sum() * 100
        out[name] = g.head(6)
    return t, out
tops = {}
for col in ['Deposits', 'Loans']:
    display(Markdown(f'#### Top 100 by {col}'))
    t, out = top_composition(col); tops[col] = t
    for name in ['segment', 'country', 'branch group', 'officer code']:
        print(f'by {name}:'); display(styled(out[name], money_cols=['Amount'], pct_cols=['% of top-100 $'], int_cols=['CIFs']))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))
for ax, col in zip(axes, ['Deposits', 'Loans']):
    t = tops[col]
    comp = pd.DataFrame({'CIFs': t['Group3'].value_counts(), '$': t.groupby('Group3')[col].sum()}).reindex(GROUP3_ORDER).fillna(0)
    comp = comp / comp.sum(axis=0).replace(0, np.nan) * 100
    left = np.zeros(2)
    for g in GROUP3_ORDER:
        vals = comp.loc[g].values; base = SEP.join(g.split(SEP)[:2]); hatch = '' if g.endswith('Select') and not g.endswith('Non-Select') else '///'
        ax.barh(['CIFs', '$'], vals, left=left, color=pal4[base], hatch=hatch, label=g, height=0.62, edgecolor=INK['surface'], linewidth=1)
        for i, (l, v) in enumerate(zip(left, vals)):
            if v >= 10: ax.text(l + v / 2, i, f'{v:.0f}%', ha='center', va='center', fontsize=8.5, color='white')
        left += vals
    ax.invert_yaxis(); ax.set_xlim(0, 100); ax.xaxis.set_major_formatter(PCT0); ax.grid(axis='y', visible=False)
    style(ax, title=f'Top 100 by {col.lower()} — by IorB × ForD × Select (hatched = Non-Select)')
axes[1].legend(loc='upper center', bbox_to_anchor=(-0.1, -0.22), ncol=4, fontsize=8)
finish(fig, 'Who the top 100 relationships are', name='09_top100')

## 9.3 The tail above fixed thresholds

In [ ]:
rows = []
for thr in [1e5, 1e6, 5e6, 10e6, 50e6]:
    m = df['TotalBalance'] >= thr
    rows.append({'Total balance ≥': money(thr), 'CIFs': int(m.sum()), '% of CIFs': m.mean() * 100, '% of deposits': df.loc[m, 'Deposits'].sum() / df['Deposits'].sum() * 100,
                 '% of loans': df.loc[m, 'Loans'].sum() / df['Loans'].sum() * 100 if df['Loans'].sum() else np.nan, '% Business': is_(df.loc[m, 'IorB'], 'Business').mean() * 100,
                 '% Foreign': is_(df.loc[m, 'ForD'], 'Foreign').mean() * 100, '% Select': is_(df.loc[m, 'Select'], 'Select').mean() * 100})
tail = pd.DataFrame(rows).set_index('Total balance ≥')
display(styled(tail, pct_cols=[c for c in tail.columns if c.startswith('%')], int_cols=['CIFs']))

**Observations §9** *(fill in after the run)*

- **Concentration:** …
- **Value pyramid:** …
- **Top 100 depositors:** …
- **Tail:** …

# 10. Balances in depth

Box plots summarize; histograms show the shape. Two humps inside one segment mean two customer types (transactional vs. wealth) that the segment labels do not separate. Then the deposit-tier mix per segment, and the loan side: how borrowing relates to deposits.

In [ ]:
order4, pal4 = group_spec(['IorB', 'ForD'])
fig, axes = plt.subplots(2, 4, figsize=(17, 6.4), sharey='row')
for r, col in enumerate(['Deposits', 'Loans']):
    for c, g in enumerate(order4):
        ax = axes[r, c]; sg = df.loc[is_(df['Group2'], g) & (df[col] > 0), col]
        if len(sg) < 10:
            ax.text(0.5, 0.5, f'n={len(sg)} — too few holders', ha='center', va='center', transform=ax.transAxes, color=INK['muted']); ax.set_xticks([]); style(ax, title=g, ylabel=f'{LABEL[col]}\n% of holders' if c == 0 else ''); continue
        sns.histplot(x=sg, log_scale=True, bins=40, stat='percent', color=pal4[g], edgecolor=INK['surface'], linewidth=0.4, ax=ax)
        ymax = ax.get_ylim()[1]
        for q, lab, ls in [(.5, 'P50', '--'), (.9, 'P90', ':'), (.99, 'P99', ':')]:
            v = sg.quantile(q); ax.axvline(v, color=INK['secondary'], lw=1, ls=ls)
            ax.text(v, ymax * 0.98, f' {lab} {money(v)}', rotation=90, ha='left', va='top', fontsize=7.2, color=INK['secondary'])
        ax.xaxis.set_major_formatter(MONEY); style(ax, title=f'{g}  (n={len(sg):,})', ylabel=f'{LABEL[col]}\n% of holders' if c == 0 else '')
finish(fig, 'Shape of the balance distributions among holders', 'Log scale · dashed = median, dotted = P90 and P99', name='10_hist')

In [ ]:
# Deposit-tier mix per segment: customers vs money
tier_colors = [CMAP_BLUE(x) for x in np.linspace(0.12, 1.0, len(TIER_BANDS))]
ct_n = pd.crosstab(df['Group3'], df['DepositTier'], normalize='index').reindex(index=GROUP3_ORDER, columns=TIER_BANDS).fillna(0) * 100
ct_d = df.pivot_table(index='Group3', columns='DepositTier', values='Deposits', aggfunc='sum').reindex(index=GROUP3_ORDER, columns=TIER_BANDS).fillna(0).clip(lower=0)
ct_d = ct_d.div(ct_d.sum(axis=1).replace(0, np.nan), axis=0) * 100
fig, axes = plt.subplots(1, 2, figsize=(16, 0.8 + 0.5 * len(GROUP3_ORDER)))
for ax, ct, ttl in zip(axes, [ct_n, ct_d], ['Share of CIFs by deposit tier', 'Share of deposits by deposit tier']):
    left = np.zeros(len(ct))
    for tier, colr in zip(TIER_BANDS, tier_colors):
        vals = ct[tier].values
        ax.barh(ct.index, vals, left=left, color=colr, label=tier, height=0.62, edgecolor=INK['surface'], linewidth=1)
        for i, (l, v) in enumerate(zip(left, vals)):
            if v >= 7: ax.text(l + v / 2, i, f'{v:.0f}%', ha='center', va='center', fontsize=7.8, color=text_on(colr))
        left += vals
    ax.invert_yaxis(); ax.set_xlim(0, 100); ax.xaxis.set_major_formatter(PCT0); ax.grid(axis='y', visible=False); style(ax, title=ttl)
axes[1].tick_params(labelleft=False)
axes[1].legend(loc='lower center', bbox_to_anchor=(-0.1, -0.3), ncol=6, title='Deposit tier', title_fontsize=8.5)
finish(fig, 'Deposit tiers by segment', 'Left: where the customers are · right: where the money is', name='10_tiers')

## 10.1 Loans vs. deposits — the shape of the credit relationship

Borrowers only. A point below the diagonal is a customer whose deposits exceed the loan; a point on the left edge is a **borrow-only** relationship with no deposits at all. The loan-to-deposit ratio per segment says which segments fund themselves.

In [ ]:
bor = df[df['Loans'] > 0].copy(); bor['borrow_only'] = bor['Deposits'] <= 0
FLOOR = 100.0
fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={'width_ratios': [1.35, 1]})
ax = axes[0]
for g in order4:
    sg = bor[is_(bor['Group2'], g)]
    if len(sg) == 0: continue
    ax.scatter(sg['Deposits'].clip(lower=FLOOR), sg['Loans'], s=14, alpha=0.55, color=pal4[g], edgecolors='none', label=f'{g} (n={len(sg):,} · {sg["borrow_only"].mean():.0%} borrow-only)')
hi = max(bor['Loans'].max(), bor['Deposits'].max()) * 1.6
ax.plot([FLOOR, hi], [FLOOR, hi], color=INK['muted'], lw=1, ls='--'); ax.text(hi, hi, 'loans = deposits ', ha='right', va='bottom', fontsize=8, color=INK['muted'])
ax.axvline(FLOOR * 1.05, color=INK['grid'], lw=1); ax.text(FLOOR * 1.1, bor['Loans'].min(), ' no deposits →', fontsize=7.5, color=INK['muted'], va='bottom')
ax.set_xscale('log'); ax.set_yscale('log'); ax.xaxis.set_major_formatter(MONEY); ax.yaxis.set_major_formatter(MONEY); ax.grid(axis='x', color=INK['grid'])
style(ax, title='Borrowers: deposits vs loans', xlabel='Deposits (log; zero-deposit borrowers on the left edge)', ylabel='Loans (log)'); ax.legend(loc='upper left', fontsize=8)
ltd = df.groupby('Group2').agg(Loans=('Loans', 'sum'), Deposits=('Deposits', 'sum')).reindex(order4)
ltd.loc['All CIFs'] = df[['Loans', 'Deposits']].sum()
ltd['LTD'] = ltd['Loans'] / ltd['Deposits'].replace(0, np.nan)
cols = [pal4.get(g, INK['muted']) for g in ltd.index]
hbar(axes[1], ltd.index.tolist(), ltd['LTD'].fillna(0).values, cols, fmt=lambda v: f'{v:.2f}')
axes[1].axvline(1, color=RED, lw=1, ls=':'); style(axes[1], title='Loan-to-deposit ratio (segment totals)', xlabel='loans ÷ deposits · red line = 1.0')
finish(fig, 'The credit relationship', name='10_ltd')

bt = bor.groupby('Group3').agg(Borrowers=('CIF', 'size'), BorrowOnly=('borrow_only', lambda x: x.mean() * 100), MedianLoan=('Loans', 'median')).reindex(GROUP3_ORDER)
with_dep = bor[bor['Deposits'] > 0]
bt['MedianLTD'] = (with_dep['Loans'] / with_dep['Deposits']).groupby(with_dep['Group3']).median().reindex(GROUP3_ORDER)
bt['% of segment that borrows'] = (df.groupby('Group3')['Loans'].apply(lambda x: (x > 0).mean() * 100)).reindex(GROUP3_ORDER)
bt = bt.rename(columns={'BorrowOnly': '% borrow-only (no deposits)', 'MedianLoan': 'Median loan', 'MedianLTD': 'Median loan ÷ deposits (CIFs with both)'})
display(styled(bt[['% of segment that borrows', 'Borrowers', '% borrow-only (no deposits)', 'Median loan', 'Median loan ÷ deposits (CIFs with both)']],
               pct_cols=['% of segment that borrows', '% borrow-only (no deposits)'], money_cols=['Median loan'], int_cols=['Borrowers']).format({'Median loan ÷ deposits (CIFs with both)': '{:.1f}'}))
print(f'Borrow-only relationships overall: {bor["borrow_only"].sum():,} of {len(bor):,} borrowers ({bor["borrow_only"].mean():.1%}), carrying {money(bor.loc[bor["borrow_only"], "Loans"].sum())} of loans')

**Observations §10** *(fill in after the run)*

- **Shape:** …
- **Deposit tiers:** …
- **Credit:** …

# 11. Time — vintages, survival and closures

The snapshot has four dates. Three views use them: **when** today's customers and money arrived (by year of CIF creation), which **vintages survived** (this is the only view that uses the inactive rows of the raw file), and **closure signals** inside the active base.

In [ ]:
order4, pal4 = group_spec(['IorB', 'ForD'])
yr = df['CohortYear']; ymax = int(yr.max()); ymin = max(int(yr.min()), ymax - 24); years = list(range(ymin, ymax + 1))
coh = pd.crosstab(yr, df['Group2']).reindex(years).fillna(0).reindex(columns=order4).fillna(0)
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
bottom = np.zeros(len(coh))
for g in order4:
    axes[0].bar(coh.index, coh[g].values, bottom=bottom, color=pal4[g], label=g, edgecolor=INK['surface'], linewidth=0.4); bottom += coh[g].values
style(axes[0], title='Active CIFs by year of CIF creation', ylabel='CIFs'); axes[0].legend(fontsize=8)
for ax, col in zip(axes[1:], ['Deposits', 'Loans']):
    byy = df.groupby(['CohortYear', 'IorB'])[col].sum().unstack().reindex(years).fillna(0).reindex(columns=ORDER['IorB']).fillna(0)
    bottom = np.zeros(len(byy))
    for g in ORDER['IorB']:
        ax.bar(byy.index, byy[g].values, bottom=bottom, color=COLOR['IorB'][g], label=g, edgecolor=INK['surface'], linewidth=0.4); bottom += byy[g].values
    style(ax, title=f'{col} held today, by CIF vintage', money_y=True); ax.legend(fontsize=8)
for ax in axes: ax.tick_params(axis='x', labelrotation=45)
before = (df['FirstAccountCreationDate'] < df['effectiveCIFCreationDate']).mean()
finish(fig, "Vintages: when today's customers and money arrived", f'Years {ymin}–{ymax} (earlier vintages not shown) · {before:.1%} of CIFs opened an account before their CIF date, so their real start is earlier', name='11_vintage')
cohort_tbl = df.groupby('CohortYear').agg(CIFs=('CIF', 'size'), Deposits=('Deposits', 'sum'), Loans=('Loans', 'sum'), Select=('Select', lambda x: is_(x, 'Select').mean() * 100), Foreign=('ForD', lambda x: is_(x, 'Foreign').mean() * 100)).reindex(years)
cohort_tbl['Deposits per CIF'] = cohort_tbl['Deposits'] / cohort_tbl['CIFs']
display(styled(cohort_tbl.rename(columns={'Select': '% Select', 'Foreign': '% Foreign'}).tail(12), money_cols=['Deposits', 'Loans', 'Deposits per CIF'], pct_cols=['% Select', '% Foreign'], int_cols=['CIFs']))

## 11.1 Survival by vintage — from the full file

Share of the CIFs created in each year that are still active today. Not a true survival curve (there is no CIF close date), but the closest thing the file allows, and the only place inactive rows are used.

In [ ]:
r = raw[['CIF_Status', 'effectiveCIFCreationDate', 'IorB']].copy()
r['year'] = pd.to_datetime(r['effectiveCIFCreationDate'], errors='coerce').dt.year
r['active'] = r['CIF_Status'].map(clean_text).eq(ACTIVE_STATUS)
r['IorB'] = r['IorB'].map(clean_text)
r = r[r['year'].between(ymin, ymax)]
vint = r.groupby('year').agg(CIFs=('active', 'size'), active=('active', 'sum')).reindex(years).fillna(0)
vint['active_rate'] = vint['active'] / vint['CIFs'].replace(0, np.nan) * 100
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
axes[0].bar(vint.index, vint['active'], color=BLUE, label='Still active', edgecolor=INK['surface'], linewidth=0.4)
axes[0].bar(vint.index, vint['CIFs'] - vint['active'], bottom=vint['active'], color=GRAY, label='Inactive / prospect', edgecolor=INK['surface'], linewidth=0.4)
style(axes[0], title='CIFs created per year and their status today', ylabel='CIFs'); axes[0].legend(fontsize=8)
for g in ORDER['IorB']:
    v = r[is_(r['IorB'], g)].groupby('year')['active'].agg(['mean', 'size']); v = v[v['size'] >= 30]
    axes[1].plot(v.index, v['mean'] * 100, color=COLOR['IorB'][g], lw=2, marker='o', ms=4, label=g)
axes[1].plot(vint.index, vint['active_rate'], color=INK['muted'], lw=1.5, ls='--', label='All')
axes[1].set_ylim(0, 100); axes[1].yaxis.set_major_formatter(PCT0)
style(axes[1], title='Share still active, by year of CIF creation', ylabel='% still active'); axes[1].legend(fontsize=8, loc='upper left')
for ax in axes: ax.tick_params(axis='x', labelrotation=45)
finish(fig, 'Survival by vintage', f'From the raw file ({len(raw):,} rows incl. inactive). Years with fewer than 30 CIFs of a type are not drawn.', name='11_survival')
survival_5y = vint['active_rate'].get(ymax - 5, np.nan)
print(f'Of the CIFs created in {ymax - 5}, {survival_5y:.0f}% are active today; created in {ymax - 10}: {vint["active_rate"].get(ymax - 10, np.nan):.0f}%; in {ymax - 1}: {vint["active_rate"].get(ymax - 1, np.nan):.0f}%')

## 11.2 Closure signals inside the active base

An active CIF that keeps closing accounts is a shrinking relationship. Share with any closure, and share with a closure in the last 12 months, by segment.

In [ ]:
cl = df.groupby('Group3').agg(CIFs=('CIF', 'size'), AnyClosure=('ClosedAccountsCount', lambda x: (x > 0).mean() * 100), ClosureRatio=('ClosureRatio', 'mean'),
                              Closed12=('MonthsSinceLastClose', lambda x: (x <= 12).mean() * 100), Closed36=('MonthsSinceLastClose', lambda x: (x <= 36).mean() * 100)).reindex(GROUP3_ORDER)
cl = cl.rename(columns={'AnyClosure': '% with any closure', 'ClosureRatio': 'Avg closed ÷ total accounts', 'Closed12': '% closed something in last 12m', 'Closed36': '% in last 36m'})
display(styled(cl, pct_cols=['% with any closure', '% closed something in last 12m', '% in last 36m'], int_cols=['CIFs']).format({'Avg closed ÷ total accounts': '{:.2f}'}))

order4, pal4 = group_spec(['IorB', 'ForD'])
cols8 = [pal4[SEP.join(g.split(SEP)[:2])] for g in GROUP3_ORDER]; hatch8 = ['' if g.endswith(SEP + 'Select') else '///' for g in GROUP3_ORDER]
fig, axes = plt.subplots(1, 3, figsize=(17, 3.9), gridspec_kw={'width_ratios': [1, 1, 1.1]})
hbar(axes[0], GROUP3_ORDER, cl['% with any closure'].values, cols8, fmt=fmt_pct, hatches=hatch8); style(axes[0], title='% with any closed account'); axes[0].xaxis.set_major_formatter(PCT0)
hbar(axes[1], GROUP3_ORDER, cl['% closed something in last 12m'].values, cols8, fmt=fmt_pct, hatches=hatch8); style(axes[1], title='% that closed an account in the last 12 months'); axes[1].xaxis.set_major_formatter(PCT0); axes[1].tick_params(labelleft=False)
ax = axes[2]
for g in ORDER['IorB']:
    m = df.loc[is_(df['IorB'], g), 'MonthsSinceLastClose'].dropna(); m = m[m <= 60]
    if len(m): sns.histplot(x=m, bins=30, stat='percent', element='step', fill=True, alpha=0.18, color=COLOR['IorB'][g], linewidth=1.8, ax=ax, label=f'{g} (n={len(m):,})')
ax.axvline(12, color=RED, lw=1, ls=':'); ax.legend(fontsize=8)
style(ax, title='Months since last closure (CIFs with a closure, last 5 years)', xlabel='months', ylabel='% of those CIFs')
finish(fig, 'Closure signals inside active CIFs', 'Hatched = Non-Select · red line = 12 months', name='11_closures')

**Observations §11** *(fill in after the run)*

- **Vintages:** …
- **Survival:** …
- **Closures:** …

# 12. Geography — country and region

Mostly a foreign-book story: which countries the customers come from, where the deposits and loans are, and how concentrated the country mix is. Domestic customers are split US vs. Puerto Rico.

In [ ]:
print(f'Country coverage: {df["Country"].notna().mean():.1%} of CIFs · {df["Country"].nunique()} distinct countries')
dom = df[is_(df['ForD'], 'Domestic')]
dom_tbl = dom.groupby(dom['Country'].fillna('(missing)')).agg(CIFs=('CIF', 'size'), Deposits=('Deposits', 'sum'), Loans=('Loans', 'sum')).sort_values('CIFs', ascending=False)
dom_tbl['% CIFs'] = dom_tbl['CIFs'] / dom_tbl['CIFs'].sum() * 100
print('\nDomestic CIFs by country:'); display(styled(dom_tbl.head(8), pct_cols=['% CIFs'], money_cols=['Deposits', 'Loans'], int_cols=['CIFs']))

fr = df[is_(df['ForD'], 'Foreign')].copy(); fr['Country'] = fr['Country'].fillna('(missing)')
ctry = fr.groupby('Country').agg(CIFs=('CIF', 'size'), Individuals=('IorB', lambda s: is_(s, 'Individual').sum()), Businesses=('IorB', lambda s: is_(s, 'Business').sum()),
                                 Select=('Select', lambda s: is_(s, 'Select').mean() * 100), Deposits=('Deposits', 'sum'), Loans=('Loans', 'sum'),
                                 MedianDeposits=('Deposits', lambda s: s[s > 0].median()), MedianAge=('Age', 'median'), MedianTenure=('CIF_Tenure_Years', 'median'))
ctry['% foreign CIFs'] = ctry['CIFs'] / ctry['CIFs'].sum() * 100
ctry['% foreign deposits'] = ctry['Deposits'] / ctry['Deposits'].sum() * 100
ctry['% foreign loans'] = ctry['Loans'] / ctry['Loans'].sum() * 100 if ctry['Loans'].sum() else np.nan
ctry = ctry.sort_values('CIFs', ascending=False)
top = ctry.head(TOP_COUNTRIES)
print(f'\nForeign CIFs: {len(fr):,} · top {TOP_COUNTRIES} countries = {top["CIFs"].sum() / len(fr):.1%} of foreign CIFs and {top["Deposits"].sum() / max(ctry["Deposits"].sum(), 1):.1%} of foreign deposits')
display(styled(top.rename(columns={'Select': '% Select', 'MedianDeposits': 'Median deposits (holders)', 'MedianAge': 'Median age (Ind.)', 'MedianTenure': 'Median tenure (y)'}),
               pct_cols=['% Select', '% foreign CIFs', '% foreign deposits', '% foreign loans'], money_cols=['Deposits', 'Loans', 'Median deposits (holders)'],
               int_cols=['CIFs', 'Individuals', 'Businesses'], one_dec=['Median age (Ind.)', 'Median tenure (y)'], gradient_cols=['% foreign CIFs', '% foreign deposits']))

fig, axes = plt.subplots(1, 3, figsize=(16, 0.8 + 0.4 * len(top)), sharey=True)
hbar(axes[0], top.index.tolist(), top['CIFs'].values, BLUE, fmt=lambda v: f'{v:,.0f} ({v / len(fr):.0%})'); style(axes[0], title='Foreign CIFs')
hbar(axes[1], top.index.tolist(), top['Deposits'].values, GREEN, fmt=money); style(axes[1], title='Deposits'); axes[1].xaxis.set_major_formatter(MONEY)
hbar(axes[2], top.index.tolist(), top['Loans'].values, GREEN_T, fmt=money); style(axes[2], title='Loans'); axes[2].xaxis.set_major_formatter(MONEY)
for ax in axes[1:]: ax.tick_params(labelleft=False)
finish(fig, f'Foreign book · top {TOP_COUNTRIES} countries', 'Sorted by number of CIFs. Country and money rankings differ — that is the point.', name='12_countries')

In [ ]:
# Country × IorB (stacked) and country concentration
fr = df[is_(df['ForD'], 'Foreign')].copy(); fr['Country'] = fr['Country'].fillna('(missing)')
top_c = fr['Country'].value_counts().head(TOP_COUNTRIES).index
ct = pd.crosstab(fr['Country'], fr['IorB']).reindex(top_c).reindex(columns=ORDER['IorB']).fillna(0)
fig, axes = plt.subplots(1, 2, figsize=(14, 0.8 + 0.4 * len(top_c)))
left = np.zeros(len(ct))
for g in ORDER['IorB']:
    axes[0].barh(ct.index, ct[g].values, left=left, color=COLOR['IorB'][g], label=g, height=0.62, edgecolor=INK['surface'], linewidth=0.8); left += ct[g].values
axes[0].invert_yaxis(); axes[0].grid(axis='y', visible=False); axes[0].grid(axis='x', color=INK['grid']); axes[0].legend(loc='lower right')
style(axes[0], title='Foreign CIFs by country and IorB', xlabel='CIFs')
x, y = lorenz(fr.groupby('Country')['Deposits'].sum()); axes[1].plot(x, y, color=GREEN, lw=2, label='Deposits')
x, y = lorenz(fr.groupby('Country').size().astype(float)); axes[1].plot(x, y, color=BLUE, lw=2, label='CIFs')
axes[1].set_xlim(0, 100); axes[1].set_ylim(0, 100); axes[1].xaxis.set_major_formatter(PCT0); axes[1].yaxis.set_major_formatter(PCT0); axes[1].legend(loc='lower right')
style(axes[1], title='Concentration across countries', xlabel='% of countries (largest first)', ylabel='cumulative %')
finish(fig, 'Foreign book · mix and concentration by country', name='12_country_mix')
top3 = fr['Country'].value_counts().head(3)
print(f'Top 3 countries = {top3.sum() / len(fr):.1%} of foreign CIFs: {top3.to_dict()}')

In [ ]:
# Region summary (all CIFs) — shares, Select rate, product depth
reg = df.groupby('Region').agg(CIFs=('CIF', 'size'), Individuals=('IorB', lambda s: is_(s, 'Individual').mean() * 100), Select=('Select', lambda s: is_(s, 'Select').mean() * 100),
                               Deposits=('Deposits', 'sum'), Loans=('Loans', 'sum'), MedianDeposits=('Deposits', lambda s: s[s > 0].median()),
                               HasLN=('Has_LN', lambda s: s.mean() * 100), MedianTenure=('CIF_Tenure_Years', 'median')).reindex(REGION_ORDER)
reg['% CIFs'] = reg['CIFs'] / reg['CIFs'].sum() * 100; reg['% Deposits'] = reg['Deposits'] / reg['Deposits'].sum() * 100
reg['% Loans'] = reg['Loans'] / reg['Loans'].sum() * 100 if reg['Loans'].sum() else np.nan
reg = reg.rename(columns={'Individuals': '% Individuals', 'Select': '% Select', 'MedianDeposits': 'Median deposits (holders)', 'HasLN': '% with loan', 'MedianTenure': 'Median tenure (y)'})
display(styled(reg, pct_cols=['% CIFs', '% Deposits', '% Loans', '% Individuals', '% Select', '% with loan'], money_cols=['Deposits', 'Loans', 'Median deposits (holders)'],
               int_cols=['CIFs'], one_dec=['Median tenure (y)'], gradient_cols=['% CIFs', '% Deposits', '% Loans']))

fig, axes = plt.subplots(1, 3, figsize=(15, 0.8 + 0.42 * len(reg)), sharey=True)
hbar(axes[0], reg.index.tolist(), reg['% CIFs'].values, BLUE, fmt=fmt_pct); style(axes[0], title='Share of CIFs'); axes[0].xaxis.set_major_formatter(PCT0)
hbar(axes[1], reg.index.tolist(), reg['% Deposits'].values, GREEN, fmt=fmt_pct); style(axes[1], title='Share of deposits'); axes[1].xaxis.set_major_formatter(PCT0)
hbar(axes[2], reg.index.tolist(), reg['% Loans'].fillna(0).values, GREEN_T, fmt=fmt_pct); style(axes[2], title='Share of loans'); axes[2].xaxis.set_major_formatter(PCT0)
for ax in axes[1:]: ax.tick_params(labelleft=False)
finish(fig, 'Customers and money by region', 'All active CIFs', name='12_regions')

## 12.1 The dominant country vs. the rest of the foreign book

When one country is most of the foreign base, "foreign" averages describe that country. `ForeignSplit` separates **Venezuela** from **Other foreign** so the rest of the book can be seen on its own.

In [ ]:
show_profile(profile_table(df, ['ForeignSplit']))
show_profile(profile_table(df, ['IorB', 'ForeignSplit']))
fig_demographics(df, ['ForeignSplit'], name='12_split_demographics')
fig_balances(df, ['ForeignSplit'], HEADLINE_BAL, name='12_split_balances', title='Headline balances')

## 12.2 Value per customer by country

Countries ranked by what a typical customer holds, not by how many customers there are. Small jurisdictions with very high balances per CIF are usually holding vehicles rather than people.

In [ ]:
fr = df[is_(df['ForD'], 'Foreign')]
cc = fr.groupby('Country').agg(CIFs=('CIF', 'size'), Deposits=('Deposits', 'sum'), MedianDep=('Deposits', lambda x: x[x > 0].median()),
                               Select=('Select', lambda x: is_(x, 'Select').mean() * 100), Business=('IorB', lambda x: is_(x, 'Business').mean() * 100))
cc = cc[cc['CIFs'] >= 30].copy(); cc['Deposits per CIF'] = cc['Deposits'] / cc['CIFs']
top_v = cc.sort_values('MedianDep', ascending=False).head(15)
fig, axes = plt.subplots(1, 2, figsize=(14, 0.8 + 0.4 * len(top_v)))
labels = [f'{c}  (n={int(n):,})' for c, n in zip(top_v.index, top_v['CIFs'])]
hbar(axes[0], labels, top_v['MedianDep'].fillna(0).values, BLUE, fmt=money); style(axes[0], title='Median deposits per holder'); axes[0].xaxis.set_major_formatter(MONEY)
hbar(axes[1], labels, top_v['Deposits per CIF'].values, GREEN, fmt=money); style(axes[1], title='Deposits per CIF (mean)'); axes[1].xaxis.set_major_formatter(MONEY); axes[1].tick_params(labelleft=False)
finish(fig, 'Value per customer by country', 'Countries with at least 30 foreign CIFs, ranked by median deposits per holder', name='12_value_by_country')
display(styled(top_v.rename(columns={'MedianDep': 'Median deposits (holders)', 'Select': '% Select', 'Business': '% Business'}), money_cols=['Deposits', 'Median deposits (holders)', 'Deposits per CIF'], pct_cols=['% Select', '% Business'], int_cols=['CIFs']))

In [ ]:
# Treemap: area = foreign deposits by country, color = median deposits per holder
frc = fr.groupby(fr['Country'].fillna('(missing)')).agg(Deposits=('Deposits', 'sum'), CIFs=('CIF', 'size'), MedianDep=('Deposits', lambda x: x[x > 0].median() if (x > 0).any() else np.nan))
frc = frc[frc['Deposits'] > 0].sort_values('Deposits', ascending=False)
top_t = frc.head(20).copy(); rest = frc.iloc[20:]
if len(rest): top_t.loc[f'OTHER ({len(rest)} countries)'] = [rest['Deposits'].sum(), rest['CIFs'].sum(), np.nan]
top_t = top_t.sort_values('Deposits', ascending=False)
vals = top_t['MedianDep'].fillna(top_t['MedianDep'].median()).values.astype(float)
norm = mpl.colors.Normalize(vmin=float(np.nanmin(vals)), vmax=float(np.nanquantile(vals, 0.95)) if len(vals) > 3 else float(np.nanmax(vals)))
colors = [CMAP_BLUE(0.08 + 0.92 * float(np.clip(norm(v), 0, 1))) for v in vals]
total = top_t['Deposits'].sum()
fig, ax = plt.subplots(figsize=(15, 6.8))
for (x, y, w, h), (cname, row), colr in zip(squarify(top_t['Deposits'].values, 0, 0, 100, 100), top_t.iterrows(), colors):
    ax.add_patch(mpl.patches.Rectangle((x, y), w, h, facecolor=colr, edgecolor=INK['surface'], linewidth=2))
    area = w * h; fs = 8.5 if area >= 6 else 7 if area >= 2.5 else 6
    maxc = int(w * (2.0 if fs >= 8 else 2.5 if fs >= 7 else 3.0))          # characters that fit on one line at this width
    lines = []
    if maxc >= 5: lines.append(cname if len(cname) <= maxc else cname[:maxc - 1] + '…')
    if maxc >= 11 and h >= 4: lines.append(f'{money(row["Deposits"])} · {row["Deposits"] / total:.0%}')
    if maxc >= 11 and h >= 7 and area >= 6: lines.append(f'{int(row["CIFs"]):,} CIFs')
    if lines: ax.text(x + w / 2, y + h / 2, '\n'.join(lines), ha='center', va='center', fontsize=fs, color=text_on(colr))
ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.invert_yaxis(); ax.axis('off')
sm = mpl.cm.ScalarMappable(norm=norm, cmap=CMAP_BLUE); cb = fig.colorbar(sm, ax=ax, fraction=0.025, pad=0.01)
cb.set_label('median deposits per holder', fontsize=8.5); cb.ax.yaxis.set_major_formatter(MONEY); cb.ax.tick_params(labelsize=8)
finish(fig, 'Foreign deposits by country', 'Area = deposits · color = median deposits per holder (darker = wealthier customers) · top 20 countries + rest', name='12_treemap')

**Observations §12** *(fill in after the run)*

- **The foreign book** …
- **Value per customer:** …
- **Domestic:** …
- **Venezuela vs the rest:** …
- **Data:** …

# 13. Branches, officers and segment codes

The distribution channel: which branch groups own which customers, how concentrated officer portfolios are, and whether the `Segment` code tells the same story as IorB / ForD.

In [ ]:
bg = df.groupby(df['BranchGroupName'].fillna('(missing)'))
br = pd.DataFrame({'CIFs': bg.size(), '% Individuals': bg['IorB'].apply(lambda s: is_(s, 'Individual').mean() * 100), '% Foreign': bg['ForD'].apply(lambda s: is_(s, 'Foreign').mean() * 100),
                   '% Select': bg['Select'].apply(lambda s: is_(s, 'Select').mean() * 100), 'Deposits': bg['Deposits'].sum(), 'Loans': bg['Loans'].sum(),
                   'Median deposits (holders)': bg['Deposits'].apply(lambda s: s[s > 0].median()), 'Avg families': bg['ProductFamilies'].mean()}).sort_values('CIFs', ascending=False)
br['% CIFs'] = br['CIFs'] / br['CIFs'].sum() * 100; br['% Deposits'] = br['Deposits'] / br['Deposits'].sum() * 100; br['% Loans'] = br['Loans'] / br['Loans'].sum() * 100 if br['Loans'].sum() else np.nan
display(styled(br, pct_cols=['% CIFs', '% Deposits', '% Loans', '% Individuals', '% Foreign', '% Select'], money_cols=['Deposits', 'Loans', 'Median deposits (holders)'], int_cols=['CIFs'], one_dec=['Avg families'], gradient_cols=['% CIFs', '% Deposits', '% Loans']))

ct = pd.crosstab(df['BranchGroupName'].fillna('(missing)'), df['IorB']).reindex(br.index).reindex(columns=ORDER['IorB']).fillna(0)
fig, axes = plt.subplots(1, 3, figsize=(16, 0.8 + 0.42 * len(br)), sharey=True)
left = np.zeros(len(ct))
for g in ORDER['IorB']:
    axes[0].barh(ct.index, ct[g].values, left=left, color=COLOR['IorB'][g], label=g, height=0.62, edgecolor=INK['surface'], linewidth=0.8); left += ct[g].values
axes[0].invert_yaxis(); axes[0].grid(axis='y', visible=False); axes[0].grid(axis='x', color=INK['grid']); axes[0].legend(loc='lower right'); style(axes[0], title='CIFs by branch group and IorB', xlabel='CIFs')
hbar(axes[1], br.index.tolist(), br['Deposits'].values, GREEN, fmt=money); style(axes[1], title='Deposits'); axes[1].xaxis.set_major_formatter(MONEY)
hbar(axes[2], br.index.tolist(), br['Loans'].values, GREEN_T, fmt=money); style(axes[2], title='Loans'); axes[2].xaxis.set_major_formatter(MONEY)
for ax in axes[1:]: ax.tick_params(labelleft=False)
finish(fig, 'Branch groups', name='13_branches')
print('\nTop 15 branches (BranchName):'); display(df['BranchName'].value_counts().head(15).to_frame('CIFs'))

In [ ]:
# Officer portfolios — concentration, not level-by-level plots (144+ codes)
off = df.groupby('OfficerCode').agg(CIFs=('CIF', 'size'), Individuals=('IorB', lambda s: is_(s, 'Individual').sum()), Deposits=('Deposits', 'sum'), Loans=('Loans', 'sum')).sort_values('CIFs', ascending=False)
off['% CIFs'] = off['CIFs'] / off['CIFs'].sum() * 100; off['% Deposits'] = off['Deposits'] / off['Deposits'].sum() * 100
print(f'{len(off):,} officer codes · median {off["CIFs"].median():.0f} CIFs per officer · top 5 codes hold {off["% CIFs"].head(5).sum():.1f}% of CIFs and {off["% Deposits"].head(5).sum():.1f}% of deposits')
print('Top 10 officer codes:'); display(styled(off.head(10), pct_cols=['% CIFs', '% Deposits'], money_cols=['Deposits', 'Loans'], int_cols=['CIFs', 'Individuals']))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
top_off = off.head(15)
hbar(axes[0], top_off.index.tolist(), top_off['CIFs'].values, BLUE, fmt=lambda v: f'{v:,.0f} ({v / len(df):.0%})'); style(axes[0], title='Top 15 officer codes by CIFs', xlabel='CIFs')
x, y = lorenz(off['CIFs'].astype(float)); axes[1].plot(x, y, color=BLUE, lw=2, label='CIFs')
x, y = lorenz(off['Deposits']); axes[1].plot(x, y, color=GREEN, lw=2, label='Deposits')
axes[1].set_xlim(0, 100); axes[1].set_ylim(0, 100); axes[1].xaxis.set_major_formatter(PCT0); axes[1].yaxis.set_major_formatter(PCT0); axes[1].legend(loc='lower right')
style(axes[1], title='Portfolio concentration across officers', xlabel='% of officers (largest first)', ylabel='cumulative %')
finish(fig, 'Officer portfolios', 'A few "house" codes usually hold most individual CIFs — check whether they are real relationship officers', name='13_officers')

In [ ]:
# Segment codes: size, money and consistency with IorB / ForD
sg = df.groupby(df['Segment'].fillna('(missing)'))
segt = pd.DataFrame({'CIFs': sg.size(), '% CIFs': sg.size() / len(df) * 100, 'Type (decoded)': df.groupby(df['Segment'].fillna('(missing)'))['Segment_Type'].first(),
                     'Geo (decoded)': df.groupby(df['Segment'].fillna('(missing)'))['Segment_Geo'].first(),
                     '% agree IorB': sg['Segment_vs_IorB_ok'].mean() * 100, '% agree ForD': sg['Segment_vs_ForD_ok'].mean() * 100,
                     '% Select': sg['Select'].apply(lambda s: is_(s, 'Select').mean() * 100), 'Deposits': sg['Deposits'].sum(),
                     'Median deposits (holders)': sg['Deposits'].apply(lambda s: s[s > 0].median()), 'Loans': sg['Loans'].sum(), 'Median tenure (y)': sg['CIF_Tenure_Years'].median()})
segt['% Deposits'] = segt['Deposits'] / segt['Deposits'].sum() * 100
segt = segt.sort_values(['Type (decoded)', 'Geo (decoded)', 'CIFs'], ascending=[True, True, False], na_position='last')
display(styled(segt, pct_cols=['% CIFs', '% Deposits', '% agree IorB', '% agree ForD', '% Select'], money_cols=['Deposits', 'Loans', 'Median deposits (holders)'], int_cols=['CIFs'], one_dec=['Median tenure (y)'], gradient_cols=['% CIFs', '% Deposits']))

fig, ax = plt.subplots(figsize=(11, 0.8 + 0.32 * len(segt)))
cols = [COLOR['IorB'].get(t, GRAY) if isinstance(t, str) else GRAY for t in segt['Type (decoded)']]
hbar(ax, segt.index.tolist(), segt['CIFs'].values, cols, fmt=lambda v: f'{v:,.0f}', height=0.66)
style(ax, xlabel='CIFs');
finish(fig, 'CIFs by segment code', 'Blue = personal (P) codes · green = business (B) codes · gray = Unassigned / undecodable', name='13_segments')

## 13.1 Branch group × segment — who owns which customers

In [ ]:
order4, pal4 = group_spec(['IorB', 'ForD'])
bgn = df['BranchGroupName'].fillna('(missing)')
ct_n = pd.crosstab(bgn, df['Group2']).reindex(columns=order4).fillna(0)
ct_n = ct_n.loc[ct_n.sum(axis=1).sort_values(ascending=False).index]
ct_d = df.assign(_b=bgn).pivot_table(index='_b', columns='Group2', values='Deposits', aggfunc='sum').reindex(index=ct_n.index, columns=order4).fillna(0)
fig, axes = plt.subplots(1, 2, figsize=(15, 0.7 + 0.42 * len(ct_n)), sharey=True)
for ax, ct, ttl, fmtf in zip(axes, [ct_n, ct_d], ['CIFs', 'Deposits'], [fmt_int, money]):
    colshare = ct.div(ct.sum(axis=0).replace(0, np.nan), axis=1) * 100
    sns.heatmap(colshare.astype(float), annot=ct.map(fmtf).values, fmt='', cmap=CMAP_BLUE, cbar=False, ax=ax, linewidths=1, linecolor=INK['surface'], annot_kws={'size': 8.5}, vmin=0, vmax=100)
    ax.set_ylabel(''); ax.set_xlabel(''); ax.grid(False); ax.tick_params(axis='x', labelrotation=15); style(ax, title=f'{ttl} — color = share of the segment (column)')
finish(fig, 'Branch group × segment', 'Each column of color sums to 100%: where each segment is booked. Cells show the actual count / amount.', name='13_branch_segment')
if (bgn == '(missing)').any():
    mb = df[bgn == '(missing)']
    print(f'CIFs without branch group: {len(mb):,} · deposits {money(mb["Deposits"].sum())} ({mb["Deposits"].sum() / df["Deposits"].sum():.1%} of the book) · loans {money(mb["Loans"].sum())} ({mb["Loans"].sum() / max(df["Loans"].sum(), 1):.1%})')
    mbt = pd.DataFrame({'CIFs': mb['Group3'].value_counts(), 'Deposits': mb.groupby('Group3')['Deposits'].sum(), 'Loans': mb.groupby('Group3')['Loans'].sum()}).reindex(GROUP3_ORDER).dropna(how='all')
    display(styled(mbt, money_cols=['Deposits', 'Loans'], int_cols=['CIFs']))
    print('Most frequent segment codes / officer codes among them:')
    display(pd.concat([mb['Segment'].fillna('(missing)').value_counts().head(5).rename('CIFs by Segment'), mb['OfficerCode'].fillna('(missing)').value_counts().head(5).rename('CIFs by OfficerCode')], axis=1))

## 13.2 House codes vs. relationship officers

Officer codes that each hold more than 3% of all CIFs are flagged as **house codes** (pooled portfolios). If they look like the mass market — small balances, one product, few Select — those customers are effectively unmanaged.

In [ ]:
off = df.groupby('OfficerCode').agg(CIFs=('CIF', 'size'), Deposits=('Deposits', 'sum'))
off['% CIFs'] = off['CIFs'] / len(df) * 100
HOUSE = off[off['% CIFs'] > 3].sort_values('CIFs', ascending=False).index.tolist()
df['OfficerType'] = np.where(df['OfficerCode'].isin(HOUSE), 'House code', 'Relationship officer')
house_m = df['OfficerType'].eq('House code')
print(f'House codes (> 3% of CIFs each): {HOUSE} → {house_m.mean():.1%} of CIFs, {df.loc[house_m, "Deposits"].sum() / df["Deposits"].sum():.1%} of deposits, {df.loc[house_m, "Loans"].sum() / max(df["Loans"].sum(), 1):.1%} of loans')
OT_ORDER = ['Relationship officer', 'House code']
ot = df.groupby('OfficerType').agg(CIFs=('CIF', 'size'), Deposits=('Deposits', 'sum'), Loans=('Loans', 'sum'), DepPerCIF=('Deposits', 'mean'), MedianDep=('Deposits', lambda x: x[x > 0].median()),
                                   Select=('Select', lambda x: is_(x, 'Select').mean() * 100), Families=('ProductFamilies', 'mean'), HasLN=('Has_LN', lambda x: x.mean() * 100),
                                   ForeignInd=('Group2', lambda x: is_(x, 'Individual' + SEP + 'Foreign').mean() * 100), Tenure=('CIF_Tenure_Years', 'median')).reindex(OT_ORDER)
ot = ot.rename(columns={'DepPerCIF': 'Deposits per CIF', 'MedianDep': 'Median deposits (holders)', 'Select': '% Select', 'Families': 'Avg families', 'HasLN': '% with loan', 'ForeignInd': '% foreign individuals', 'Tenure': 'Median tenure (y)'})
display(styled(ot, money_cols=['Deposits', 'Loans', 'Deposits per CIF', 'Median deposits (holders)'], pct_cols=['% Select', '% with loan', '% foreign individuals'], int_cols=['CIFs'], one_dec=['Avg families', 'Median tenure (y)']))
fig, axes = plt.subplots(1, 4, figsize=(16, 3.1))
for ax, col, fmtf in zip(axes, ['Deposits per CIF', 'Median deposits (holders)', '% Select', 'Avg families'], [money, money, fmt_pct, fmt_1]):
    v = ot[col].astype(float).values
    bars = ax.bar(OT_ORDER, np.nan_to_num(v), color=[BLUE, GRAY], width=0.55, edgecolor=INK['surface'])
    for b_, val in zip(bars, v):
        if not np.isnan(val): ax.text(b_.get_x() + b_.get_width() / 2, val, fmtf(val), ha='center', va='bottom', fontsize=8.5, color=INK['secondary'])
    ax.set_ylim(0, max(np.nanmax(v) * 1.25, 1e-9)); style(ax, title=col)
    if col.startswith('%'): ax.yaxis.set_major_formatter(PCT0)
    elif col != 'Avg families': ax.yaxis.set_major_formatter(MONEY)
finish(fig, 'House codes vs relationship officers', 'Gray = house codes (pooled portfolios)', name='13_house_codes')

**Observations §13** *(fill in after the run)*

- **Branch groups:** …
- **Officers:** …
- **Segment codes:** …

# 14. Relationships & tests — what moves with what

Correlations first (Spearman, so heavy tails do not dominate), then the relationships that matter for the customer story: age, tenure and product depth against balances, the Select uplift measured within segment with confidence intervals, effect sizes for every comparison, and the product-penetration map.

In [ ]:
corr_cols = ['Age', 'BusinessAge', 'CIF_Tenure_Years', 'ActiveAccountsCount', 'ProductFamilies', 'OtherAccountsActive_c', 'Deposits', 'Loans', 'DD Accounts Balance', 'CD Accounts Balance', 'LN Accounts Balance']
fig, axes = plt.subplots(1, 2, figsize=(15, 5.6))
for ax, g in zip(axes, ORDER['IorB']):
    cc = [c for c in corr_cols if not (g == 'Individual' and c == 'BusinessAge') and not (g == 'Business' and c == 'Age')]
    corr = df.loc[is_(df['IorB'], g), cc].corr(method='spearman')
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap=CMAP_DIV, center=0, vmin=-1, vmax=1, cbar=False, ax=ax, annot_kws={'size': 7.5}, linewidths=0.5, linecolor=INK['surface'])
    ax.set_title(g); ax.tick_params(axis='x', labelrotation=45); ax.grid(False)
finish(fig, 'Spearman correlation between measures', 'Red = negative, blue = positive. Balances include zeros (non-holders).', name='14_corr')

In [ ]:
# Tenure and product depth vs. money, by IorB
order2, pal2 = group_spec(['IorB'])
fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
med, lo, hi = median_table(df, 'Deposits', 'TenureBand', 'IorB', YEAR_BANDS, order2, holders=True)
grouped_bars(axes[0], med, order2, pal2, fmt=money, lo=lo, hi=hi); style(axes[0], title='Median deposits (holders) by tenure band · 95% CI', money_y=True); axes[0].set_ylim(0, np.nanmax(hi.values) * 1.18)
fam = df.groupby(['TenureBand', 'IorB'])['ProductFamilies'].mean().unstack().reindex(index=YEAR_BANDS, columns=order2)
grouped_bars(axes[1], fam, order2, pal2, fmt=lambda v: f'{v:.2f}'); style(axes[1], title='Avg product families by tenure band'); axes[1].set_ylim(0, 3)
sel = df.groupby(['TenureBand', 'IorB'])['Select'].apply(lambda s: is_(s, 'Select').mean() * 100).unstack().reindex(index=YEAR_BANDS, columns=order2)
grouped_bars(axes[2], sel, order2, pal2, fmt=lambda v: f'{v:.0f}%'); style(axes[2], title='Share of Select by tenure band', pct_y=True); axes[2].set_ylim(0, 100)
for ax in axes: ax.set_xlabel('CIF tenure (years)')
finish(fig, 'What changes with tenure', name='14_tenure')

# Product mix vs. money
mix = df.groupby(['ProductMix', 'IorB']).agg(CIFs=('CIF', 'size'), MedDep=('Deposits', lambda s: s[s > 0].median()), MedLoan=('Loans', lambda s: s[s > 0].median()), Select=('Select', lambda s: is_(s, 'Select').mean() * 100))
mix = mix.unstack('IorB').reindex(MIX_ORDER).dropna(how='all')
share = mix['CIFs'].div(mix['CIFs'].sum(axis=0), axis=1) * 100
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
grouped_bars(axes[0], share.reindex(columns=order2), order2, pal2, fmt=lambda v: f'{v:.0f}%'); style(axes[0], title='Share of CIFs by product mix', pct_y=True); axes[0].tick_params(axis='x', labelrotation=30)
medm, lom, him = median_table(df, 'Deposits', 'ProductMix', 'IorB', list(mix.index), order2, holders=True)
grouped_bars(axes[1], medm, order2, pal2, fmt=money, lo=lom, hi=him); style(axes[1], title='Median deposits (holders) by product mix · 95% CI', money_y=True); axes[1].tick_params(axis='x', labelrotation=30)
axes[1].set_ylim(0, np.nanmax(him.values) * 1.18)
finish(fig, 'Product mix', 'Families = DD / CD / LN only', name='14_mix')
mix.columns = [f'{a} · {b}' for a, b in mix.columns]
display(styled(mix, money_cols=[c for c in mix.columns if c.startswith('Med')], pct_cols=[c for c in mix.columns if c.startswith('Select')], int_cols=[c for c in mix.columns if c.startswith('CIFs')]))

In [ ]:
# Select uplift: Select vs Non-Select within each IorB × ForD
rows = []
for g in group_spec(['IorB', 'ForD'])[0]:
    d = df[is_(df['Group2'], g)]
    for s in ORDER['Select']:
        e = d[is_(d['Select'], s)]
        rows.append({'segment': g, 'Select': s, 'CIFs': len(e), 'Median deposits (holders)': e.loc[e['Deposits'] > 0, 'Deposits'].median(), 'Mean deposits': e['Deposits'].mean(),
                     '% with loan': e['Has_LN'].mean() * 100, 'Median loans (holders)': e.loc[e['Loans'] > 0, 'Loans'].median(), 'Avg families': e['ProductFamilies'].mean(),
                     'Avg active accts': e['ActiveAccountsCount'].mean(), 'Median tenure (y)': e['CIF_Tenure_Years'].median(), 'Median age (Ind.)': e['Age'].median()})
up = pd.DataFrame(rows).set_index(['segment', 'Select'])
display(styled(up, money_cols=['Median deposits (holders)', 'Mean deposits', 'Median loans (holders)'], pct_cols=['% with loan'], int_cols=['CIFs'], one_dec=['Avg families', 'Avg active accts', 'Median tenure (y)', 'Median age (Ind.)']))
ratio = (up.xs('Select', level='Select') / up.xs('Non-Select', level='Select')).drop(columns=['CIFs'])
print('Select ÷ Non-Select ratio (>1 = Select is higher):'); display(ratio.round(2))

In [ ]:
# Association between dimensions (Cramér's V)
cat_cols = [c for c in ['IorB', 'ForD', 'Select', 'Region', 'BranchGroupName', 'ProductMix', 'TenureBand', 'AgeBand', 'DepositTier', 'Segment_Tier'] if c in df.columns]
V = pd.DataFrame(index=cat_cols, columns=cat_cols, dtype=float)
for a in cat_cols:
    for b in cat_cols:
        V.loc[a, b] = 1.0 if a == b else cramers_v(df[a].fillna('(missing)'), df[b].fillna('(missing)'))
fig, ax = plt.subplots(figsize=(8.5, 6))
sns.heatmap(V.astype(float), annot=True, fmt='.2f', cmap=CMAP_BLUE, vmin=0, vmax=1, cbar=False, ax=ax, annot_kws={'size': 8}, linewidths=0.5, linecolor=INK['surface'])
ax.grid(False); ax.tick_params(axis='x', labelrotation=45)
finish(fig, "Association between dimensions (Cramér's V)", '0 = independent, 1 = one determines the other', name='14_cramers')

In [ ]:
# Within-segment uplift with confidence intervals — the aggregate ratio mixes segments (Simpson's paradox)
order4, pal4 = group_spec(['IorB', 'ForD']); orderS, palS = group_spec(['Select'])
medS, loS, hiS = median_table(df, 'Deposits', 'Group2', 'Select', order4, orderS, holders=True)
fig, axes = plt.subplots(1, 2, figsize=(15, 3.9), gridspec_kw={'width_ratios': [1.5, 1]})
grouped_bars(axes[0], medS, orderS, palS, fmt=money, lo=loS, hi=hiS); style(axes[0], title='Median deposits among holders · Select vs Non-Select within segment (95% CI)', money_y=True)
top_y = np.nanmax(hiS.values); axes[0].set_ylim(0, top_y * 1.35)
ratio = medS['Select'] / medS['Non-Select']
for i, g in enumerate(order4): axes[0].text(i, top_y * 1.2, f'×{ratio[g]:.1f}', ha='center', fontsize=9.5, fontweight='bold', color=INK['primary'])
sel_h = df.loc[is_(df['Select'], 'Select') & (df['Deposits'] > 0), 'Deposits']; non_h = df.loc[is_(df['Select'], 'Non-Select') & (df['Deposits'] > 0), 'Deposits']
agg_ratio = sel_h.median() / non_h.median()
hbar(axes[1], ['All CIFs (mix effect)'] + order4, [agg_ratio] + [ratio[g] for g in order4], [INK['muted']] + [pal4[g] for g in order4], fmt=lambda v: f'×{v:.1f}')
axes[1].axvline(1, color=RED, lw=1, ls=':'); style(axes[1], title='Select ÷ Non-Select median deposits', xlabel='ratio (red line = no uplift)')
finish(fig, 'The Select uplift, measured honestly', 'The all-CIF ratio is inflated by segment mix (Simpson\'s paradox); the within-segment ratios are the real effect', name='14_uplift')
print(f'Aggregate ratio ×{agg_ratio:.1f} vs within-segment: ' + ' · '.join(f'{g} ×{ratio[g]:.1f}' for g in order4))

## 14.1 Are the differences real? Effect sizes and tests

Mann-Whitney U tests whether one group tends to hold more than the other; **Cliff's δ** says how much (−1…+1: the probability that a random member of A exceeds a random member of B, minus the reverse; |δ| < 0.15 negligible, < 0.33 small, < 0.47 medium, else large). With thousands of CIFs almost everything is "significant", so the effect size is the number to read.

In [ ]:
from scipy.stats import mannwhitneyu
def compare(a, b):
    a = pd.Series(a).dropna(); b = pd.Series(b).dropna()
    if len(a) < 10 or len(b) < 10: return np.nan, np.nan
    u = mannwhitneyu(a, b, alternative='two-sided')
    return float(u.pvalue), float(2 * u.statistic / (len(a) * len(b)) - 1)
def effect_label(d):
    if pd.isna(d): return 'n too small'
    a = abs(d); return 'negligible' if a < 0.147 else 'small' if a < 0.33 else 'medium' if a < 0.474 else 'large'
rows = []
def add_test(comparison, segment, measure, a, b, holders=False):
    if holders: a = a[a > 0]; b = b[b > 0]
    pval, d = compare(a, b)
    rows.append({'comparison': comparison, 'segment': segment, 'measure': measure + (' (holders)' if holders else ''), 'n A': int(a.notna().sum()), 'n B': int(b.notna().sum()),
                 'median A': a.median(), 'median B': b.median(), 'p (Mann-Whitney)': pval, "Cliff's δ (A vs B)": d, 'effect': effect_label(d)})
for g in order4:
    d_ = df[is_(df['Group2'], g)]; sel, non = is_(d_['Select'], 'Select'), is_(d_['Select'], 'Non-Select')
    add_test('A = Select, B = Non-Select', g, 'Deposits', d_.loc[sel, 'Deposits'], d_.loc[non, 'Deposits'], holders=True)
    add_test('A = Select, B = Non-Select', g, 'Tenure (years)', d_.loc[sel, 'CIF_Tenure_Years'], d_.loc[non, 'CIF_Tenure_Years'])
    add_test('A = Select, B = Non-Select', g, 'Active accounts', d_.loc[sel, 'ActiveAccountsCount'], d_.loc[non, 'ActiveAccountsCount'])
for g in ORDER['IorB']:
    d_ = df[is_(df['IorB'], g)]; dom, forn = is_(d_['ForD'], 'Domestic'), is_(d_['ForD'], 'Foreign'); agec = 'Age' if g == 'Individual' else 'BusinessAge'
    add_test('A = Domestic, B = Foreign', g, 'Deposits', d_.loc[dom, 'Deposits'], d_.loc[forn, 'Deposits'], holders=True)
    add_test('A = Domestic, B = Foreign', g, LABEL[agec], d_.loc[dom, agec], d_.loc[forn, agec])
    add_test('A = Domestic, B = Foreign', g, 'Tenure (years)', d_.loc[dom, 'CIF_Tenure_Years'], d_.loc[forn, 'CIF_Tenure_Years'])
tests = pd.DataFrame(rows).set_index(['comparison', 'segment', 'measure'])
def fmt_med(v): return '' if pd.isna(v) else (money(v) if abs(v) >= 1000 else f'{v:,.1f}')
def fmt_p(v): return '' if pd.isna(v) else ('<0.001' if v < 0.001 else f'{v:.3f}')
display(tests.style.format({'median A': fmt_med, 'median B': fmt_med, 'p (Mann-Whitney)': fmt_p, "Cliff's δ (A vs B)": '{:+.2f}', 'n A': fmt_int, 'n B': fmt_int}, na_rep='')
        .background_gradient(cmap=CMAP_DIV, subset=["Cliff's δ (A vs B)"], vmin=-1, vmax=1))

# Product holding vs Select, within IorB (chi-square + Cramér's V)
rows = []
for g in ORDER['IorB']:
    d_ = df[is_(df['IorB'], g)]
    for f in FAM + ['Other']:
        ct = pd.crosstab(d_['Select'], d_[f'Has_{f}'])
        pval = chi2_contingency(ct)[1] if ct.shape == (2, 2) else np.nan
        rows.append({'IorB': g, 'product': f, '% holding · Select': d_.loc[is_(d_['Select'], 'Select'), f'Has_{f}'].mean() * 100,
                     '% holding · Non-Select': d_.loc[is_(d_['Select'], 'Non-Select'), f'Has_{f}'].mean() * 100, 'p (chi-square)': pval, "Cramér's V": cramers_v(d_['Select'], d_[f'Has_{f}'])})
chi = pd.DataFrame(rows).set_index(['IorB', 'product'])
display(chi.style.format({'% holding · Select': fmt_pct, '% holding · Non-Select': fmt_pct, 'p (chi-square)': fmt_p, "Cramér's V": '{:.2f}'}, na_rep=''))

## 14.2 Product penetration — the cross-sell map

In [ ]:
pen_cols = {'DD': 'Has_DD', 'CD': 'Has_CD', 'LN': 'Has_LN', 'Other (not in extract)': 'Has_Other'}
pen = pd.DataFrame({k: df.groupby('Group3')[v].mean() * 100 for k, v in pen_cols.items()}).reindex(GROUP3_ORDER)
pen.loc['All CIFs'] = [df[v].mean() * 100 for v in pen_cols.values()]
bgn = df['BranchGroupName'].fillna('(missing)'); bg_top = bgn.value_counts().head(8).index
pen_b = pd.DataFrame({k: df.assign(_b=bgn).groupby('_b')[v].mean() * 100 for k, v in pen_cols.items()}).reindex(bg_top)
fig, axes = plt.subplots(1, 2, figsize=(15, 0.7 + 0.45 * max(len(pen), len(pen_b))))
for ax, t, ttl in zip(axes, [pen, pen_b], ['By segment', 'By branch group (top 8 by CIFs)']):
    sns.heatmap(t.astype(float), annot=True, fmt='.0f', cmap=CMAP_BLUE, cbar=False, ax=ax, linewidths=1, linecolor=INK['surface'], annot_kws={'size': 9}, vmin=0, vmax=100)
    for tx in ax.texts: tx.set_text(tx.get_text() + '%')
    ax.set_ylabel(''); ax.set_xlabel(''); ax.grid(False); ax.tick_params(axis='x', labelrotation=0); style(ax, title=ttl)
finish(fig, 'Product penetration — share of CIFs holding each family', 'Light cells = cross-sell whitespace · "Other" = account types not in the extract', name='14_penetration')

**Observations §14** *(fill in after the run)*

- **Strongest relationships:** …
- **Tenure:** …
- **Select uplift:** …
- **Surprises:** …

# 15. What makes a valuable customer — explanatory models

Two questions, answered with models that are used only to *explain*, never to score anyone:

1. **Who ends up in the top 10% by total balance?** Features: demographics, tenure, channel, country, product holdings. Balances and the Select flag are excluded so the answer is not circular.
2. **What characterizes a Select customer?** The same features plus (log) balances, to see whether Select is a wealth tier or something else.

For each: the cross-validated AUC of a gradient-boosting model (how predictable the outcome is at all), permutation importance (which features carry the signal), and a depth-3 decision tree (the simplest rules a person can repeat).

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.inspection import permutation_importance

def design(d, extra_num=()):
    X = pd.DataFrame(index=d.index)
    for c in ['Age', 'BusinessAge', 'CIF_Tenure_Years', 'ActiveAccountsCount', 'ClosedAccountsCount', 'ProductFamilies', 'OtherAccountsActive_c', 'Segment_Tier'] + list(extra_num):
        X[c] = d[c].astype(float)
    for c in ['Has_DD', 'Has_CD', 'Has_LN']: X[c] = d[c].astype(float)
    cats = pd.DataFrame({'IorB': d['IorB'], 'ForD': d['ForD'], 'Country': lump(d['Country'].fillna('(missing)'), 5), 'Branch': lump(d['BranchGroupName'].fillna('(missing)'), 6), 'Region': lump(d['Region'], 6)})
    X = pd.concat([X, pd.get_dummies(cats, dtype=float)], axis=1)
    X.columns = [str(c).replace(SEP, '_').replace(' ', '_') for c in X.columns]
    return X

def explain(target_name, y, X, name):
    y = np.asarray(y).astype(int)
    hgb = HistGradientBoostingClassifier(max_depth=4, max_iter=250, learning_rate=0.06, random_state=1)
    auc = cross_val_score(hgb, X, y, cv=5, scoring='roc_auc')
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=1, stratify=y)
    hgb.fit(Xtr, ytr)
    imp = permutation_importance(hgb, Xte, yte, n_repeats=5, scoring='roc_auc', random_state=1)
    imp_s = pd.Series(imp.importances_mean, index=X.columns).sort_values(ascending=False).head(12)
    tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=max(50, len(X) // 200), random_state=1).fit(X.fillna(-1), y)
    fig = plt.figure(figsize=(18, 8.5)); gs = fig.add_gridspec(1, 2, width_ratios=[1, 2.2])
    ax0 = fig.add_subplot(gs[0]); hbar(ax0, imp_s.index.tolist(), imp_s.values.clip(min=0), BLUE, fmt=lambda v: f'{v:.3f}')
    style(ax0, title='Permutation importance\n(AUC lost when the feature is shuffled)')
    ax1 = fig.add_subplot(gs[1])
    arts = plot_tree(tree, feature_names=list(X.columns), class_names=['No', 'Yes'], filled=True, proportion=True, impurity=False, rounded=True, fontsize=8, ax=ax1, precision=2)
    # Recolor nodes in brand blue by their share of "Yes" (parsed from each node's own text, so edge labels are ignored)
    nodes = [(a, re.search(r'value = \[([\d.]+), ([\d.]+)\]', a.get_text())) for a in arts if a.get_bbox_patch() is not None]
    rates = [float(m.group(2)) for a, m in nodes if m]; rmax = max(rates) if rates else 1.0
    for a, m in nodes:
        if not m: continue
        r = float(m.group(2)) / max(rmax, 1e-9)
        a.get_bbox_patch().set_facecolor(CMAP_BLUE(0.06 + 0.94 * r)); a.get_bbox_patch().set_edgecolor(INK['grid'])
        a.set_color('white' if r > 0.55 else INK['primary'])
    style(ax1, title=f'Simplest rules (depth-3 tree)\ndarker = higher share of "{target_name}" · value = [No, Yes] shares · samples = share of CIFs')
    finish(fig, f'What predicts "{target_name}"', f'Gradient boosting 5-fold AUC = {auc.mean():.3f} ± {auc.std():.3f} (0.5 = coin flip, 1 = perfect) · base rate {y.mean():.1%} · n = {len(y):,}', name=name)
    return imp_s, auc

X_top = design(df); y_top = df['TopDecile']
imp_top, auc_top = explain('top 10% by total balance', y_top, X_top, '15_top_decile')
X_sel = design(df, extra_num=['log_Deposits', 'log_Loans']); y_sel = is_(df['Select'], 'Select')
imp_sel, auc_sel = explain('Select', y_sel, X_sel, '15_select')

**Observations §15** *(fill in after the run)*

- **Top 10% by total balance:** …
- **Select:** …

# 16. Segment profiles — who the customer is, final version

The 8 segments (IorB × ForD × Select) on the same metrics, plus a "fingerprint" heatmap where every column is scaled 0–1 across segments so the reader sees at a glance which segment is oldest, longest-tenured, deepest, richest.

In [ ]:
prof = profile_table(df, ['IorB', 'ForD', 'Select'])
show_profile(prof)

fp_cols = ['% CIFs', 'Median age (Ind.)', 'Median business age', 'Median tenure (y)', 'Avg families', '% has DD', '% has CD', '% has LN', '% Deposits', 'Median deposits (holders)', '% Loans', 'Median loans (holders)']
fp = prof[fp_cols].copy()
scaled = (fp - fp.min()) / (fp.max() - fp.min())
annot = fp.copy().astype(object)
for c in fp_cols:
    annot[c] = fp[c].map(money if c in MONEY_COLS else (fmt_pct if c in PCT_COLS else fmt_1))
fig, ax = plt.subplots(figsize=(15, 0.6 + 0.55 * len(fp)))
sns.heatmap(scaled.astype(float), annot=annot.values, fmt='', cmap=CMAP_BLUE, cbar=False, ax=ax, linewidths=1, linecolor=INK['surface'], annot_kws={'size': 8.5})
ax.set_yticklabels([SEP.join(ix) for ix in fp.index], rotation=0); ax.set_ylabel(''); ax.tick_params(axis='x', labelrotation=30); ax.grid(False)
finish(fig, 'Segment fingerprint — IorB × ForD × Select', 'Each column scaled 0–1 across the 8 segments (darker = higher). Cells show the actual value.', name='16_fingerprint')

# Auto-narrative: one sentence per segment
print('\nSegment one-liners (paste into the findings):')
for ix, r in prof.iterrows():
    name = SEP.join(ix); age = f'median age {r["Median age (Ind.)"]:.0f}' if ix[0] == 'Individual' else f'median business age {r["Median business age"]:.0f}y'
    print(f'• {name}: {r["CIFs"]:,.0f} CIFs ({r["% CIFs"]:.1f}%), {age}, tenure {r["Median tenure (y)"]:.1f}y, {r["Avg families"]:.2f} families, '
          f'holds {r["% Deposits"]:.1f}% of deposits (median {money(r["Median deposits (holders)"])}) and {r["% Loans"]:.1f}% of loans.')

**Observations §16** *(fill in after the run — one sentence per segment)*

- **Individual · Domestic · Select** …
- **Individual · Domestic · Non-Select** …
- **Individual · Foreign · Select** …
- **Individual · Foreign · Non-Select** …
- **Business · Domestic · Select** …
- **Business · Domestic · Non-Select** …
- **Business · Foreign · Select** …
- **Business · Foreign · Non-Select** …

# 17. Personas (optional) — k-means per IorB

Descriptive only, never for production. Run separately for individuals and businesses because their features mean different things. Inputs: log deposits, log loans, tenure, age (or business age), active accounts, product families — standardized.

In [ ]:
if RUN_PERSONAS:
    from sklearn.preprocessing import StandardScaler
    from sklearn.cluster import KMeans
    from sklearn.decomposition import PCA
    df['Persona'] = pd.Series([None] * len(df), index=df.index, dtype=object)
    PERSONA_COLORS = [BLUE, GREEN, BLUE_T, GREEN_T, GRAY, '#1E5A8A', '#2F8F6B', '#A3B4CC'][:K_PERSONAS]
    for g, age_col in [('Individual', 'Age'), ('Business', 'BusinessAge')]:
        d = df[is_(df['IorB'], g)].copy()
        feats = pd.DataFrame({'log_deposits': np.log1p(d['Deposits'].clip(lower=0)), 'log_loans': np.log1p(d['Loans'].clip(lower=0)), 'tenure': d['CIF_Tenure_Years'],
                              age_col: d[age_col], 'active_accts': d['ActiveAccountsCount'].clip(upper=20), 'families': d['ProductFamilies']})
        feats = feats.fillna(feats.median())
        Xs = StandardScaler().fit_transform(feats)
        km = KMeans(n_clusters=K_PERSONAS, n_init=10, random_state=1).fit(Xs)
        # Label personas by descending median deposits so P1 is always the richest
        lab = pd.Series(km.labels_, index=d.index)
        rank = d.assign(_l=lab).groupby('_l')['Deposits'].median().sort_values(ascending=False).index
        remap = {old: f'{g[0]}{i + 1}' for i, old in enumerate(rank)}
        df.loc[d.index, 'Persona'] = lab.map(remap)
        d['Persona'] = lab.map(remap)
        order_p = [f'{g[0]}{i + 1}' for i in range(K_PERSONAS)]; pal_p = dict(zip(order_p, PERSONA_COLORS))
        pt = d.groupby('Persona').agg(CIFs=('CIF', 'size'), MedDeposits=('Deposits', 'median'), MedLoans=('Loans', 'median'), HasLN=('Has_LN', lambda s: s.mean() * 100),
                                      Tenure=('CIF_Tenure_Years', 'median'), AgeOrBizAge=(age_col, 'median'), Accts=('ActiveAccountsCount', 'median'), Families=('ProductFamilies', 'mean'),
                                      Foreign=('ForD', lambda s: is_(s, 'Foreign').mean() * 100), Select=('Select', lambda s: is_(s, 'Select').mean() * 100)).reindex(order_p)
        pt['% CIFs'] = pt['CIFs'] / pt['CIFs'].sum() * 100
        pt = pt.rename(columns={'MedDeposits': 'Median deposits', 'MedLoans': 'Median loans', 'HasLN': '% with loan', 'Tenure': 'Median tenure (y)', 'AgeOrBizAge': f'Median {age_col}', 'Accts': 'Median active accts', 'Families': 'Avg families', 'Foreign': '% Foreign', 'Select': '% Select'})
        display(Markdown(f'### {g} personas (k = {K_PERSONAS})'))
        display(styled(pt, money_cols=['Median deposits', 'Median loans'], pct_cols=['% CIFs', '% with loan', '% Foreign', '% Select'], int_cols=['CIFs'], one_dec=['Median tenure (y)', f'Median {age_col}', 'Median active accts', 'Avg families']))
        pcs = PCA(n_components=2, random_state=1).fit_transform(Xs)
        fig, ax = plt.subplots(figsize=(7, 4.5))
        samp = np.random.default_rng(1).choice(len(d), min(len(d), 6000), replace=False)
        for p in order_p:
            m = (d['Persona'].values[samp] == p)
            ax.scatter(pcs[samp][m, 0], pcs[samp][m, 1], s=9, alpha=0.45, color=pal_p[p], label=f'{p} (n={int(pt.loc[p, "CIFs"]):,})', edgecolors='none')
        ax.grid(False); ax.set_xticks([]); ax.set_yticks([]); ax.legend(markerscale=2.2)
        style(ax, title=f'{g} · personas in PCA space', xlabel='component 1', ylabel='component 2')
        finish(fig, f'{g} personas', 'k-means on standardized log balances, tenure, age, accounts, families · sample of points shown', name=f'17_personas_{g}')
else:
    print('RUN_PERSONAS = False — skipped')

**Observations §17** *(fill in after the run, k = 4 per IorB)*

- **Individuals:** …
- **Businesses:** …

# 18. Key findings

Written for someone who will not open the code. The cell below prints the headline numbers; the template after it is where the narrative goes.

In [ ]:
dep = df['Deposits']; loans = df['Loans']
fr_m = is_(df['ForD'], 'Foreign'); sel_m = is_(df['Select'], 'Select')
top3 = df.loc[fr_m, 'Country'].value_counts().head(3)
lines = [
    f'Population: {len(df):,} active CIFs — {ind.mean():.0%} individuals, {biz.mean():.0%} businesses; {fr_m.mean():.0%} foreign; {sel_m.mean():.0%} Select.',
    f'Money: {money(dep.sum())} deposits and {money(loans.sum())} loans. Businesses are {biz.mean():.0%} of CIFs but hold {dep[biz].sum() / dep.sum():.0%} of deposits and {loans[biz].sum() / max(loans.sum(), 1):.0%} of loans.',
    f'Foreign customers are {fr_m.mean():.0%} of CIFs and hold {dep[fr_m].sum() / dep.sum():.0%} of deposits / {loans[fr_m].sum() / max(loans.sum(), 1):.0%} of loans; top 3 countries = {top3.sum() / fr_m.sum():.0%} of foreign CIFs ({", ".join(top3.index.astype(str))}).',
    f'Typical individual: median age {df["Age"].median():.0f}, tenure {df.loc[ind, "CIF_Tenure_Years"].median():.1f} y, {df.loc[ind, "ActiveAccountsCount"].median():.0f} active account(s), median deposits among holders {money(dep[ind & (dep > 0)].median())}.',
    f'Typical business: median {df["BusinessAge"].median():.0f} years since incorporation, tenure {df.loc[biz, "CIF_Tenure_Years"].median():.1f} y, median deposits among holders {money(dep[biz & (dep > 0)].median())}; {df.loc[biz, "Has_LN"].mean():.0%} have a loan vs {df.loc[ind, "Has_LN"].mean():.0%} of individuals.',
    f'Select uplift: Select customers hold median deposits of {money(dep[sel_m & (dep > 0)].median())} vs {money(dep[~sel_m & (dep > 0)].median())} for Non-Select ({dep[sel_m & (dep > 0)].median() / max(dep[~sel_m & (dep > 0)].median(), 1):.1f}×).',
    f'Concentration: top 1% of CIFs hold {top_share(dep, .01):.0f}% of deposits, top 10% hold {top_share(dep, .10):.0f}%; top 100 CIFs = {dep.clip(lower=0).nlargest(100).sum() / dep.clip(lower=0).sum():.0%}. Loans: top 10% hold {top_share(loans, .10):.0f}%.',
    f'Product depth: {df["ProductFamilies"].le(1).mean():.0%} of CIFs hold at most one of DD / CD / LN; {df["OtherAccountsActive"].gt(0).mean():.0%} hold account types that are not in the extract ({acct_share_pct.loc["Other (not in extract)", "Active accounts"]:.0f}% of active accounts).',
    f'Data quality: {(df["OtherAccountsActive"] < 0).sum():,} CIFs with a negative account gap, {(df["DD Accounts Balance"] < 0).sum():,} overdrawn DDAs ({money(df.loc[df["DD Accounts Balance"] < 0, "DD Accounts Balance"].sum())}), {(ind & (df["AGE"] > 100)).sum():,} implausible ages, {df["Country_vs_ForD_ok"].eq(0).sum():,} country/ForD conflicts, {(~decodable).sum():,} undecodable segment codes.',
    f'Value pyramid: the top 1% of CIFs hold {vb.loc["Top 1%", "% Total balance"]:.0f}% of total balance and the bottom 50% hold {vb.loc["Bottom 50%", "% Total balance"]:.1f}%; the top 100 depositors are {is_(tops["Deposits"]["IorB"], "Business").mean():.0%} business and {is_(tops["Deposits"]["ForD"], "Foreign").mean():.0%} foreign.',
    f'Credit relationship: loan-to-deposit ratio {ltd.loc["All CIFs", "LTD"]:.2f} overall ({ltd.loc["Business" + SEP + "Domestic", "LTD"]:.2f} for domestic business); {bor["borrow_only"].mean():.0%} of borrowers hold no deposits.',
    f'Time: of the CIFs created in {ymax - 5}, {survival_5y:.0f}% are still active; {(df["MonthsSinceLastClose"] <= 12).mean():.1%} of active CIFs closed an account in the last 12 months.',
    f'Dominant country: {is_(df["ForeignSplit"], "Venezuela").sum():,} Venezuelan CIFs = {is_(df["ForeignSplit"], "Venezuela").sum() / max(fr_m.sum(), 1):.0%} of the foreign book and {df.loc[is_(df["ForeignSplit"], "Venezuela"), "Deposits"].sum() / max(df.loc[fr_m, "Deposits"].sum(), 1):.0%} of foreign deposits.',
    f'Channel: {len(HOUSE)} house officer codes hold {house_m.mean():.0%} of CIFs and {df.loc[house_m, "Deposits"].sum() / df["Deposits"].sum():.0%} of deposits; {(df["BranchGroupName"].isna()).sum():,} CIFs have no branch group.',
    f'Models: top-10% status is predictable from who the customer is (no balances) at AUC {auc_top.mean():.2f}; Select at AUC {auc_sel.mean():.2f} once balances are included.',
]
print('\n'.join(f'{i + 1}. {l}' for i, l in enumerate(lines)))

**Key findings** *(fill in after the run — the numbers come from the cell above)*

1. **Who.** …
2. **Where the money is.** …
3. **Foreign vs domestic.** …
4. **Select.** …
5. **Concentration and data risk.** …

# 19. Open questions, data to request, next steps

**Data to request (highest value first)**

1. **All account families, not just DD / CD / LN** — savings, money market, time deposits, credit cards, safe-deposit, trade finance / letters of credit. §4.3 shows how many accounts are currently invisible; until they arrive, "product depth" is understated.
2. **Average balances (monthly average, last 3–12 months)** instead of one overnight snapshot — DDA balances swing daily, so the snapshot exaggerates outliers and noise.
3. **Activity:** last transaction date, transactions per month, incoming / outgoing wires and ACH, debit-card usage. "Active" by status is not "active" by behaviour; a dormancy definition needs this.
4. **Loan detail:** loan type (mortgage, CRE, C&I, consumer), original amount, rate, maturity, delinquency (days past due), risk rating, collateral. Today loans are one number per CIF.
5. **Deposit detail:** CD term and maturity date, rate, interest-bearing vs. non-interest DDA. Needed for cost-of-funds and rollover views.
6. **Profitability / revenue per CIF:** fees, net interest contribution. The final "who is our best customer" needs money made, not just money held.
7. **Digital channel:** online / mobile enrollment, e-statements, login frequency.
8. **Demographics (individuals):** gender, occupation / industry, income band, residence state / ZIP (domestic), nationality vs. country of residence.
9. **Firmographics (businesses):** industry (NAICS), annual revenue, employees, legal entity type.
10. **Household linkage:** `HouseholdID` exists — household-level aggregation shows the real relationship size.
11. **Risk / KYC:** risk rating, PEP flag (aggregate only) — relevant for the foreign book.
12. **History:** monthly snapshots (12+ months) to turn this static picture into trends; inactive CIFs with close date and reason for an attrition view (they are the majority of the raw file).

**Country analysis — yes, and here is what it adds.** Country is the only geography for the foreign book. §12 shows customer count vs. deposits vs. loans by country, the Select rate per country, value per customer by country, the country concentration curve, and the dominant country split out from the rest. Next steps: (a) map countries to the bank's own risk / market tiers instead of the generic region map, (b) add nationality vs. residence when available, (c) track the country mix over time — a shift in the top-3 countries is an early warning for the whole foreign franchise.

**Hypotheses to test next**
- Do foreign individuals with Select hold the deposits, while domestic businesses hold the loans? (§6.1, §16)
- Is tenure the real driver of product depth, or is it the segment code? (§14, §15)
- Are the "house" officer codes hiding un-managed relationships? (§13.2)
- Which vintages are eroding fastest, and is that a country story or a product story? (§11)

**What feeds the Power BI dashboard.** The clean CIF-level table exported in §5 (`CLEAN_PATH`) with the derived fields: `Select`, `Deposits`, `Loans`, `TotalBalance`, `ProductMix`, `AgeBand`, `TenureBand`, `DepositTier`, `Region`, `ForeignSplit`, `ValueBand`, `CohortYear`, `Group2`, `Group3`, `OtherAccountsActive`, `OfficerType`, `Persona`.

# 20. Attrition — the CIFs that are no longer active

Sections 3–19 describe the customers the bank *has*. This section describes the ones it *lost*: the rows §3 removed. It rebuilds them from the raw file with **the same cleaning and feature rules as §2.1 and §5**, so IorB, ForD and the country split mean exactly the same thing here as in the rest of the notebook.

**Only attributes that do not change over a customer's life are used.** `IorB`, `ForD`, country (and from it `ForeignSplit` = Domestic / Venezuela / Other foreign, and `Region`), birth / incorporation date, the account counts and the dates. **`Banesco Select` and `Segment` are left out on purpose:** the file only says who is Select (and in which segment tier) *today*, not who was Select when they left, so any attrition split by them would compare today's label with a past customer.

**Definitions** (stated once, used everywhere in §20–§22)

| Term | Rule |
|---|---|
| **Churned CIF** | `CIF_Status` in `CHURN_STATUSES` (default `Inactive`). |
| **Prospect** | `CIF_Status` in `PROSPECT_STATUSES`. Never onboarded, so **not churn**; counted in §20.1 and excluded afterwards. |
| **Churn date** | `LastAccountCloseDate` of a churned CIF: the relationship ends when its last account closes (**assumption** — the file has no CIF close date). |
| **Lifetime** | Relationship start (§5: earlier of CIF creation and first account) → churn date. For active CIFs it runs to the snapshot and is *censored* (they have not left yet). |
| **Ever held** | `Active + Closed` accounts of a family. A churned CIF has no active accounts, so "had a loan / CD / DDA" is read from the closed counts. |
| **Exit profile** | What the customer held over its life: `DD only`, `CD only`, `LN only`, `DD+CD`, `Deposits + LN`, `Other only` (only families outside the extract), `No accounts`. A `LN only` exit is usually a loan paid off or matured; a `CD only` exit a CD that matured and was not renewed. Neither is the same business problem as a lost DDA relationship. |
| **Attrition share** | Churned ÷ (active + churned): the share of everyone the bank ever onboarded in a segment that is gone today. A *stock* measure. |
| **Annual attrition rate** | CIFs that churned in year *Y* ÷ CIFs alive on 1 January *Y*: a *flow* measure, comparable across years (§20.3). |

| Sub-section | Question |
|---|---|
| 20.1 | How big is the lost base, and is the data on it usable? |
| 20.2 | Which segments lose the most customers? (IorB, ForD, Domestic / Venezuela / other foreign, IorB × ForD) |
| 20.3 | When did they leave? Annual attrition rate by segment |
| 20.4 | **Tenure** — how long did they stay? Survival curves and lifetime bands |
| 20.5 | **Accounts and products** — how many accounts, and how many had loans, CDs, deposits |
| 20.6 | Who leaves vs. who stays: attrition share by age, depth, products, geography, channel, vintage |
| 20.7 | Churned vs. active, tested: effect sizes |
| 20.8 | Recent leavers (last 12 / 24 months) |
| 20.9 | Export and key numbers |

In [ ]:
# ============================ ATTRITION CONFIGURATION ============================
CHURN_STATUSES    = ['Inactive']        # CIF_Status values that mean "the customer left"
PROSPECT_STATUSES = ['Prospect']        # never onboarded → not churn
CHURN_YEARS       = 10                  # years shown in the annual attrition view (§20.3)
EXPORT_CHURN      = True                # True → churned CIF-level table written to CHURN_PATH
CHURN_PATH        = 'churned_cifs.csv'

# Splits used in §20–§22: only attributes that do not change over the customer's life (no Select, no Segment)
CH_GRAN       = [['IorB'], ['ForD'], ['ForeignSplit'], ['IorB', 'ForD']]
STATUS_ORDER  = ['Active', 'Churned']
EXIT_ORDER    = ['DD only', 'CD only', 'LN only', 'DD+CD', 'Deposits + LN', 'Other only', 'No accounts']
EXIT_COLOR    = dict(zip(EXIT_ORDER, [BLUE, BLUE_T, GREEN, '#2E5F94', GREEN_T, GRAY, '#C9CED6']))
ACC_EVER_BANDS = ['0', '1', '2', '3', '4', '5', '6+']
CMAP_RED      = LinearSegmentedColormap.from_list('brand_red', ['#FFFFFF', RED])
order4, pal4  = group_spec(['IorB', 'ForD'])

def clean_frame(frame):
    """The type enforcement of §2.1, applied to any slice of the raw file (here: all statuses)."""
    f = frame.copy()
    for c in DIMS + ['CIF_Status', ID_COL]:
        if c in f.columns: f[c] = f[c].map(clean_text).astype(object)
    if 'Country' in f.columns:
        f['Country'] = f['Country'].map(lambda v: v.upper() if isinstance(v, str) else v)
    for c in DIMS + ['CIF_Status']:
        if c not in f.columns: continue
        ph = f[c].map(lambda v: isinstance(v, str) and v.strip().upper() in PLACEHOLDERS)
        f.loc[ph, c] = np.nan
    for c in DATES:
        f[c] = pd.to_datetime(f[c], errors='coerce', utc=True).dt.tz_localize(None)
    for c in MEASURES:
        s = f[c]
        if not pd.api.types.is_numeric_dtype(s):
            s = s.astype(str).str.replace(',', '', regex=False).str.replace('$', '', regex=False).str.strip()
        f[c] = pd.to_numeric(s, errors='coerce').astype('float64').replace([np.inf, -np.inf], np.nan)
    return f

def exit_profile(r):
    dd, cd, ln, oth, tot = r['Had_DD'], r['Had_CD'], r['Had_LN'], r['Had_OtherEver'], r['TotalAccountsEver']
    if ln and (dd or cd): return 'Deposits + LN'
    if ln:                return 'LN only'
    if dd and cd:         return 'DD+CD'
    if dd:                return 'DD only'
    if cd:                return 'CD only'
    if oth:               return 'Other only'
    return 'No accounts' if (pd.isna(tot) or tot == 0) else 'Other only'

def add_churn_features(f):
    """The §5 rules that do not depend on today's labels, plus the lifetime / ever-held fields this section needs."""
    st = f['CIF_Status']
    f['Status'] = np.select([is_(st, ACTIVE_STATUS), st.isin(CHURN_STATUSES), st.isin(PROSPECT_STATUSES)], ['Active', 'Churned', 'Prospect'], 'Other status')
    i_, b_ = is_(f['IorB'], 'Individual'), is_(f['IorB'], 'Business')
    f['Age'] = f['AGE'].where(i_); f.loc[f['Age'] > 105, 'Age'] = np.nan
    f['BusinessAge'] = f['AGE'].where(b_)
    f['Deposits'] = f['DD Accounts Balance'].fillna(0) + f['CD Accounts Balance'].fillna(0)
    f['Loans'] = f['LN Accounts Balance'].fillna(0)
    f['TotalBalance'] = f['Deposits'] + f['Loans']
    f['Group2'] = make_label(f, ['IorB', 'ForD'])
    f['ForeignSplit'] = np.where(is_(f['ForD'], 'Domestic'), 'Domestic', np.where(is_(f['Country'], 'VENEZUELA'), 'Venezuela', 'Other foreign'))
    f['Region'] = f['Country'].map(region_of)
    # ---- ever held = active + closed, per family ----
    for fam in FAM:
        f[f'{fam}_Ever'] = f[f'{fam} Accounts Active'].fillna(0) + f[f'{fam} Accounts Closed'].fillna(0)
        f[f'Had_{fam}'] = f[f'{fam}_Ever'].gt(0)
    f['Had_Deposit'] = f['Had_DD'] | f['Had_CD']
    f['OtherEver'] = (f['TotalAccountsEver'] - f[[f'{fam}_Ever' for fam in FAM]].sum(axis=1)).clip(lower=0)
    f['Had_OtherEver'] = f['OtherEver'].gt(0)
    f['FamiliesEver'] = f[[f'Had_{fam}' for fam in FAM]].sum(axis=1).astype(int)
    f['MixEver'] = f[[f'Had_{fam}' for fam in FAM]].apply(lambda r: '+'.join([fam for fam in FAM if r[f'Had_{fam}']]) or 'None', axis=1)
    f['ExitProfile'] = f.apply(exit_profile, axis=1)
    f['AccountsEverBand'] = f['TotalAccountsEver'].map(lambda v: np.nan if pd.isna(v) else ('6+' if v >= 6 else f'{int(v)}'))
    # ---- time ----
    f['RelationshipStart'] = f[['effectiveCIFCreationDate', 'FirstAccountCreationDate']].min(axis=1)
    f['CohortYear'] = f['effectiveCIFCreationDate'].dt.year
    churned = f['Status'].eq('Churned')
    f['ChurnDate'] = f['LastAccountCloseDate'].where(churned)
    f['ChurnYear'] = f['ChurnDate'].dt.year
    f['EndDate'] = f['ChurnDate'].where(churned, pd.Series(SNAPSHOT, index=f.index).where(f['Status'].eq('Active')))
    f['EndDate'] = pd.to_datetime(f['EndDate'])
    f['LifetimeYears'] = (f['EndDate'] - f['RelationshipStart']).dt.days / 365.25
    f.loc[f['LifetimeYears'] < 0, 'LifetimeYears'] = np.nan        # close before start = data error (§20.1)
    f['LifetimeBand'] = pd.cut(f['LifetimeYears'], [0, 1, 3, 6, 11, 21, np.inf], right=False, labels=YEAR_BANDS).astype(object)
    f['YearsSinceChurn'] = (SNAPSHOT - f['ChurnDate']).dt.days / 365.25
    f['MonthsSinceChurn'] = (SNAPSHOT - f['ChurnDate']).dt.days / 30.44
    # Age at the end of the relationship: at churn for churned CIFs, at the snapshot for active ones
    age_end = (f['EndDate'] - f['BIRTHDATE_TEXT']).dt.days / 365.25
    age_end = age_end.fillna(f['AGE'] - f['YearsSinceChurn'].fillna(0))
    f['AgeAtEnd'] = age_end.where(i_ & f['Age'].notna())
    f['AgeAtEndBand'] = pd.cut(f['AgeAtEnd'], [0, 25, 35, 45, 55, 65, 75, np.inf], right=False, labels=AGE_BANDS).astype(object)
    f['BusinessAgeBand'] = pd.cut(f['BusinessAge'], [0, 1, 3, 6, 11, 21, np.inf], right=False, labels=YEAR_BANDS).astype(object)
    f['Churned'] = churned.astype(int)
    return f

full = add_churn_features(clean_frame(raw))
other = full.loc[full['Status'].eq('Other status'), 'CIF_Status'].fillna('(missing)').value_counts()
if len(other): print('⚠ CIF_Status values not mapped to Active / Churned / Prospect (excluded from §20 — add them to CHURN_STATUSES or PROSPECT_STATUSES):', other.to_dict())
dup_full = int(full[ID_COL].duplicated().sum())
if dup_full: print(f'⚠ {dup_full:,} duplicated CIFs across statuses — a CIF appears more than once in the file')

base = full[full['Status'].isin(STATUS_ORDER)].copy()          # Active + Churned: the population §20–§22 work on
ch   = base[base['Status'].eq('Churned')].copy()                 # churned only
act_ = base[base['Status'].eq('Active')]
assert len(ch), f'No churned CIFs found — check CHURN_STATUSES against the CIF_Status values in the file: {full["CIF_Status"].value_counts().to_dict()}'
print(f'Full file {len(full):,} CIFs · active {len(act_):,} · churned {len(ch):,} · prospects {full["Status"].eq("Prospect").sum():,} · other {full["Status"].eq("Other status").sum():,}')
print(f'Attrition share (churned ÷ active + churned): {len(ch) / len(base):.1%}')

## 20.1 The lost base — size and data checks

How many CIFs left, against the active base and the prospects, and whether the rows are good enough to read: a churned CIF should have **no active accounts**, **no balance**, a **close date**, and that date should fall after the relationship started.

In [ ]:
# ---- Status by segment (all statuses) ----
st_tbl = pd.crosstab(full['Group2'], full['Status']).reindex(index=order4, columns=['Active', 'Churned', 'Prospect', 'Other status']).fillna(0).astype(int)
st_tbl = st_tbl.loc[:, st_tbl.sum() > 0]
st_fs = pd.crosstab(full['ForeignSplit'], full['Status']).reindex(index=ORDER['ForeignSplit'], columns=st_tbl.columns).fillna(0).astype(int)
st_fs.index = ['Foreign split · ' + i for i in st_fs.index]
st_tbl = pd.concat([st_tbl, st_fs]); st_tbl.loc['All CIFs'] = st_tbl.loc[order4].sum()
st_tbl['Attrition share %'] = st_tbl['Churned'] / (st_tbl['Active'] + st_tbl['Churned']).replace(0, np.nan) * 100
display(styled(st_tbl, int_cols=[c for c in st_tbl.columns if c != 'Attrition share %'], pct_cols=['Attrition share %'], gradient_cols=['Attrition share %'], cmap=CMAP_RED))

# ---- Data checks on churned CIFs ----
rows = []
def add_chk(name, mask, note=''):
    mask = pd.Series(mask, index=ch.index).fillna(False).astype(bool)
    rows.append({'check (churned CIFs)': name, 'CIFs': int(mask.sum()), '% of churned': mask.mean() * 100 if len(ch) else np.nan, 'note': note})
add_chk('Still has active accounts (ActiveAccountsCount > 0)', ch['ActiveAccountsCount'] > 0, 'status and accounts disagree')
add_chk('Residual balance ≠ 0 (DD, CD or LN)', ch[BAL_COLS].fillna(0).abs().sum(axis=1) > 0, f'total {money(ch[BAL_COLS].fillna(0).sum().sum())}')
add_chk('No LastAccountCloseDate → no churn date', ch['LastAccountCloseDate'].isna(), 'excluded from timing and tenure views')
add_chk('Close date after the snapshot', ch['LastAccountCloseDate'] > SNAPSHOT)
add_chk('Close date before relationship start', ch['LastAccountCloseDate'] < ch['RelationshipStart'], 'lifetime set to missing')
add_chk('TotalAccountsEver = 0 (never had an account)', ch['TotalAccountsEver'].fillna(0).eq(0), 'closer to a prospect than to churn')
add_chk('Holds account types outside the extract (ever)', ch['Had_OtherEver'])
add_chk('IorB missing', ch['IorB'].isna()); add_chk('ForD missing', ch['ForD'].isna()); add_chk('Country missing', ch['Country'].isna())
chk = pd.DataFrame(rows).set_index('check (churned CIFs)')
display(styled(chk, pct_cols=['% of churned'], int_cols=['CIFs']))

pro = full[full['Status'].eq('Prospect')]
if len(pro): print(f'Prospects: {len(pro):,} · with any account ever: {pro["TotalAccountsEver"].fillna(0).gt(0).sum():,} (should be 0)')

# Is CIF_Tenure_Years measured to the snapshot or to the churn date for churned CIFs?
to_snap = (SNAPSHOT - ch['effectiveCIFCreationDate']).dt.days / 365.25
to_churn = (ch['ChurnDate'] - ch['effectiveCIFCreationDate']).dt.days / 365.25
m_snap, m_churn = (ch['CIF_Tenure_Years'] - to_snap).abs().median(), (ch['CIF_Tenure_Years'] - to_churn).abs().median()
print(f'CIF_Tenure_Years of churned CIFs matches "creation → snapshot" (median gap {m_snap:.2f} y) vs "creation → churn date" (median gap {m_churn:.2f} y). '
      f'§20 uses LifetimeYears (start → churn date) either way.')

## 20.2 Which segments lose the most customers

Attrition share = churned ÷ (active + churned) per segment, by IorB, ForD, the country split (Domestic / Venezuela / Other foreign) and IorB × ForD. The second chart compares the mix of the lost base with the mix of today's base: a segment whose red bar is longer than its blue bar is over-represented among the leavers.

In [ ]:
def attrition_table(d, dims):
    g = d.groupby(dims, dropna=False)
    t = pd.DataFrame({'Active': g['Status'].apply(lambda s: int(s.eq('Active').sum())), 'Churned': g['Status'].apply(lambda s: int(s.eq('Churned').sum()))})
    t['Ever onboarded'] = t['Active'] + t['Churned']
    t['Attrition share %'] = t['Churned'] / t['Ever onboarded'].replace(0, np.nan) * 100
    t['% of churned base'] = t['Churned'] / t['Churned'].sum() * 100
    t['% of active base'] = t['Active'] / t['Active'].sum() * 100
    t['Over-representation (pp)'] = t['% of churned base'] - t['% of active base']
    t['Median lifetime of churned (y)'] = d[d['Status'].eq('Churned')].groupby(dims)['LifetimeYears'].median()
    t['% churned in first year'] = d[d['Status'].eq('Churned')].groupby(dims)['LifetimeYears'].apply(lambda s: (s < 1).mean() * 100)
    idx = pd.Index(ORDER[dims[0]], name=dims[0]) if len(dims) == 1 else pd.MultiIndex.from_product([ORDER[c] for c in dims], names=dims)
    return t.reindex(idx).dropna(subset=['Ever onboarded'])

def flat_labels(idx): return [SEP.join(i) if isinstance(i, tuple) else i for i in idx]

ATTR_PCT = ['Attrition share %', '% of churned base', '% of active base', '% churned in first year']
attr_tables = {}
for dims in CH_GRAN:
    key = ' × '.join(dims); display(Markdown(f'#### {key}'))
    t = attrition_table(base, dims); attr_tables[key] = t
    display(styled(t, pct_cols=ATTR_PCT, int_cols=['Active', 'Churned', 'Ever onboarded'], one_dec=['Over-representation (pp)', 'Median lifetime of churned (y)'],
                   gradient_cols=['Attrition share %'], cmap=CMAP_RED))

overall = len(ch) / len(base) * 100
fig, axes = plt.subplots(1, 4, figsize=(18, 3.5), gridspec_kw={'width_ratios': [1, 1, 1.1, 1.4]})
for ax, dims in zip(axes, CH_GRAN):
    key = ' × '.join(dims); t = attr_tables[key]; labs = flat_labels(t.index); _, pal = group_spec(dims)
    hbar(ax, labs, t['Attrition share %'].values, [pal[l] for l in labs], fmt=fmt_pct)
    ax.axvline(overall, color=RED, lw=1, ls=':'); style(ax, title=f'By {key}'); ax.xaxis.set_major_formatter(PCT0)
finish(fig, 'Attrition share by segment', f'Churned ÷ (active + churned). Red dotted line = all CIFs ({overall:.1f}%).', name='20_attrition_share')

t4 = attr_tables['IorB × ForD']; labels4 = flat_labels(t4.index)
fig, ax = plt.subplots(figsize=(11, 0.9 + 0.55 * len(t4)))
y = np.arange(len(t4)); h = 0.38
ax.barh(y - h / 2, t4['% of active base'].values, height=h, color=BLUE, label='Active base')
ax.barh(y + h / 2, t4['% of churned base'].values, height=h, color=RED, label='Churned base')
for yi, (a, c) in enumerate(zip(t4['% of active base'].values, t4['% of churned base'].values)):
    ax.text(a + 0.3, yi - h / 2, fmt_pct(a), va='center', fontsize=7.8, color=INK['secondary']); ax.text(c + 0.3, yi + h / 2, fmt_pct(c), va='center', fontsize=7.8, color=INK['secondary'])
ax.set_yticks(y); ax.set_yticklabels(labels4); ax.invert_yaxis(); ax.grid(axis='y', visible=False); ax.grid(axis='x', color=INK['grid']); ax.xaxis.set_major_formatter(PCT0)
ax.legend(loc='lower right'); style(ax, xlabel='% of the base')
finish(fig, 'Who the leavers are vs. who stays', 'Each color sums to 100% across IorB × ForD', name='20_mix')

## 20.3 When they left — annual attrition rate

For each calendar year *Y*: CIFs that churned during *Y* ÷ CIFs alive on 1 January *Y* (relationship started before *Y* and either still active or churned on/after 1 January *Y*). The snapshot year is partial and is annualized. **Caveat:** if the source system purges old inactive CIFs, early years are understated — a rising curve can be partly an artefact of what the file keeps.

In [ ]:
def annual_rates(d, years):
    d = d[d['RelationshipStart'].notna() & (d['Status'].eq('Active') | d['ChurnDate'].notna())]
    out = []
    for Y in years:
        y0 = pd.Timestamp(Y, 1, 1); y1 = min(pd.Timestamp(Y + 1, 1, 1), SNAPSHOT + pd.Timedelta(days=1))   # the snapshot day itself counts
        alive = (d['RelationshipStart'] < y0) & (d['Status'].eq('Active') | (d['ChurnDate'] >= y0))
        left = alive & d['ChurnDate'].ge(y0) & d['ChurnDate'].lt(y1)
        frac = (y1 - y0).days / 365.25
        out.append({'year': Y, 'alive 1 Jan': int(alive.sum()), 'churned in year': int(left.sum()),
                    'annual attrition %': left.sum() / alive.sum() * 100 / frac if alive.sum() and frac > 0 else np.nan})
    return pd.DataFrame(out).set_index('year')

y_end = SNAPSHOT.year; CH_YEARS = list(range(y_end - CHURN_YEARS + 1, y_end + 1))
rates_all = annual_rates(base, CH_YEARS)
display(styled(rates_all, int_cols=['alive 1 Jan', 'churned in year'], pct_cols=['annual attrition %']))
print(f'{y_end} is partial (to {SNAPSHOT.date()}): its rate is annualized.')

fig, axes = plt.subplots(1, 4, figsize=(19, 3.8), gridspec_kw={'width_ratios': [1.25, 1, 1, 1]})
cy = pd.crosstab(ch['ChurnYear'], ch['Group2']).reindex(index=CH_YEARS, columns=order4).fillna(0)
bottom = np.zeros(len(cy))
for g in order4:
    axes[0].bar(cy.index, cy[g].values, bottom=bottom, color=pal4[g], label=g, edgecolor=INK['surface'], linewidth=0.4); bottom += cy[g].values
style(axes[0], title='CIFs churned per year', ylabel='CIFs'); axes[0].legend(fontsize=7.5)
for ax, dim in zip(axes[1:], ['IorB', 'ForD', 'ForeignSplit']):
    order, pal = group_spec([dim])
    for g in order:
        r_ = annual_rates(base[is_(base[dim], g)], CH_YEARS); r_ = r_[r_['alive 1 Jan'] >= 30]
        ax.plot(r_.index, r_['annual attrition %'], color=pal[g], lw=2, marker='o', ms=4, label=g)
    ax.plot(rates_all.index, rates_all['annual attrition %'], color=INK['muted'], lw=1.3, ls='--', label='All')
    ax.set_ylim(0, None); ax.yaxis.set_major_formatter(PCT0); style(ax, title=f'Annual attrition rate by {dim}'); ax.legend(fontsize=7.5)
for ax in axes: ax.tick_params(axis='x', labelrotation=45)
finish(fig, 'When customers left', f'Churn date = last account close date · last {CHURN_YEARS} years · {y_end} annualized · points with < 30 CIFs alive are not drawn', name='20_annual')

rates_seg = {g: annual_rates(base[is_(base['Group2'], g)], CH_YEARS)['annual attrition %'] for g in order4}
rates_seg.update({f'Foreign split · {g}': annual_rates(base[is_(base['ForeignSplit'], g)], CH_YEARS)['annual attrition %'] for g in ORDER['ForeignSplit']})
rates_seg['All CIFs'] = rates_all['annual attrition %']
rates_seg = pd.DataFrame(rates_seg).T
fig, ax = plt.subplots(figsize=(1.1 * len(CH_YEARS) + 3, 0.6 + 0.45 * len(rates_seg)))
sns.heatmap(rates_seg.astype(float), annot=True, fmt='.1f', cmap=CMAP_RED, cbar=False, ax=ax, linewidths=1, linecolor=INK['surface'], annot_kws={'size': 8.5}, vmin=0)
ax.set_ylabel(''); ax.set_xlabel(''); ax.grid(False)
finish(fig, 'Annual attrition rate (%) by segment', f'Row = segment, column = year · {y_end} annualized', name='20_annual_heatmap')

## 20.4 Tenure — how long did the leavers stay?

Two views. **Survival curves** (Kaplan–Meier) use everyone, active and churned: active CIFs count as "still here" up to the snapshot, so the curve is not biased toward short relationships the way a median over churned CIFs alone is. The **lifetime bands** describe only the churned: how long the customers we lost had been with the bank. A large `<1` year band is an onboarding problem; a large `11–20` band is a legacy book running off.

In [ ]:
def kaplan_meier(t, e):
    """Kaplan–Meier survival: t = years observed, e = 1 if churned (event), 0 if still active (censored)."""
    t = np.asarray(t, dtype=float); e = np.asarray(e, dtype=bool); ok = ~np.isnan(t); t, e = t[ok], e[ok]
    if len(t) == 0 or e.sum() == 0: return np.array([0.0]), np.array([1.0])
    ts = np.sort(t); times, d = np.unique(t[e], return_counts=True)
    at_risk = len(ts) - np.searchsorted(ts, times, side='left')
    s = np.cumprod(1 - d / at_risk)
    return np.r_[0.0, times], np.r_[1.0, s]
def km_median(x, s):
    below = np.where(s <= 0.5)[0]
    return float(x[below[0]]) if len(below) else np.nan
def km_at(x, s, yrs):
    return float(s[np.searchsorted(x, yrs, side='right') - 1]) * 100

surv = base[base['LifetimeYears'].notna()]
km_rows = []
fig, axes = plt.subplots(1, 4, figsize=(19, 3.9))
for ax, dims in zip(axes, CH_GRAN):
    order, pal = group_spec(dims); lab = make_label(surv, dims)
    for g in order:
        sg = surv[lab.eq(g)]
        if len(sg) < 30: continue
        x, s = kaplan_meier(sg['LifetimeYears'], sg['Churned'])
        ax.step(x, s * 100, where='post', color=pal[g], lw=2, label=f'{g} (n={len(sg):,})')
        km_rows.append({'split': ' × '.join(dims), 'segment': g, 'CIFs': len(sg), 'churned': int(sg['Churned'].sum()),
                        'still here after 1y %': km_at(x, s, 1), 'after 3y %': km_at(x, s, 3), 'after 5y %': km_at(x, s, 5), 'after 10y %': km_at(x, s, 10),
                        'median relationship life (y)': km_median(x, s)})
    ax.axhline(50, color=INK['grid'], lw=1); ax.set_xlim(0, 30); ax.set_ylim(0, 101); ax.yaxis.set_major_formatter(PCT0)
    style(ax, title=f'By {" × ".join(dims)}', xlabel='years since relationship start', ylabel='% still a customer'); ax.legend(fontsize=7.5, loc='lower left')
finish(fig, 'Survival curves — how long relationships last', 'Kaplan–Meier over active + churned CIFs; active CIFs are censored at the snapshot', name='20_km')
x_all, s_all = kaplan_meier(surv['LifetimeYears'], surv['Churned'])
km_rows.append({'split': 'All', 'segment': 'All CIFs', 'CIFs': len(surv), 'churned': int(surv['Churned'].sum()), 'still here after 1y %': km_at(x_all, s_all, 1),
                'after 3y %': km_at(x_all, s_all, 3), 'after 5y %': km_at(x_all, s_all, 5), 'after 10y %': km_at(x_all, s_all, 10), 'median relationship life (y)': km_median(x_all, s_all)})
km_tbl = pd.DataFrame(km_rows).set_index(['split', 'segment'])
display(styled(km_tbl, int_cols=['CIFs', 'churned'], pct_cols=['still here after 1y %', 'after 3y %', 'after 5y %', 'after 10y %'], one_dec=['median relationship life (y)']))

In [ ]:
# ---- Lifetime of the churned: table, bands per segment, distribution ----
def life_row(s, n):
    return [n, s.median(), s.quantile(.25), s.quantile(.75), s.mean(), (s < 1).mean() * 100, (s < 3).mean() * 100, (s >= 10).mean() * 100]
LIFE_COLS = ['Churned', 'Median lifetime (y)', 'P25 (y)', 'P75 (y)', 'Mean lifetime (y)', '% left in < 1 y', '% left in < 3 y', '% stayed 10+ y']
ten = pd.DataFrame({g: life_row(ch.loc[is_(ch['Group2'], g), 'LifetimeYears'], int(is_(ch['Group2'], g).sum())) for g in order4}, index=LIFE_COLS).T
for g in ORDER['ForeignSplit']:
    ten.loc[f'Foreign split · {g}'] = life_row(ch.loc[is_(ch['ForeignSplit'], g), 'LifetimeYears'], int(is_(ch['ForeignSplit'], g).sum()))
ten.loc['All churned'] = life_row(ch['LifetimeYears'], len(ch))
ten['Median years since leaving'] = [ch.loc[is_(ch['Group2'], g), 'YearsSinceChurn'].median() for g in order4] + \
                                    [ch.loc[is_(ch['ForeignSplit'], g), 'YearsSinceChurn'].median() for g in ORDER['ForeignSplit']] + [ch['YearsSinceChurn'].median()]
print('Lifetime of churned CIFs (relationship start → last account closed):')
display(styled(ten, int_cols=['Churned'], one_dec=['Median lifetime (y)', 'P25 (y)', 'P75 (y)', 'Mean lifetime (y)', 'Median years since leaving'],
               pct_cols=['% left in < 1 y', '% left in < 3 y', '% stayed 10+ y'], gradient_cols=['% left in < 1 y'], cmap=CMAP_RED))

band_colors = [CMAP_BLUE(x) for x in np.linspace(0.18, 1.0, len(YEAR_BANDS))]
act_band = pd.cut(act_['LifetimeYears'], [0, 1, 3, 6, 11, 21, np.inf], right=False, labels=YEAR_BANDS).astype(object)
fig, axes = plt.subplots(1, 2, figsize=(16, 0.9 + 0.55 * len(order4)))
for ax, (d_, bands_, ttl) in zip(axes, [(ch, ch['LifetimeBand'], 'Churned · lifetime when they left'), (act_, act_band, 'Active · relationship age today')]):
    ct = pd.crosstab(d_['Group2'], bands_, normalize='index').reindex(index=order4, columns=YEAR_BANDS).fillna(0) * 100
    left = np.zeros(len(ct))
    for band, colr in zip(YEAR_BANDS, band_colors):
        vals = ct[band].values
        ax.barh(ct.index, vals, left=left, color=colr, label=band, height=0.62, edgecolor=INK['surface'], linewidth=1)
        for i, (l, v) in enumerate(zip(left, vals)):
            if v >= 7: ax.text(l + v / 2, i, f'{v:.0f}%', ha='center', va='center', fontsize=7.8, color=text_on(colr))
        left += vals
    ax.invert_yaxis(); ax.set_xlim(0, 100); ax.xaxis.set_major_formatter(PCT0); ax.grid(axis='y', visible=False); style(ax, title=ttl)
axes[1].tick_params(labelleft=False)
axes[1].legend(loc='lower center', bbox_to_anchor=(-0.1, -0.45), ncol=6, title='Years', title_fontsize=8.5)
finish(fig, 'Tenure of the customers who left vs. the ones who stayed', 'Rows sum to 100%', name='20_lifetime_bands')

fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
for ax, dims in zip(axes, [['IorB'], ['ForD'], ['ForeignSplit']]):
    order, pal = group_spec(dims)
    kde_by(ax, ch.assign(_grp=make_label(ch, dims)), 'LifetimeYears', '_grp', order, pal, clip=(0, 35))
    style(ax, title=f'Churned · lifetime by {" × ".join(dims)}', xlabel='years')
finish(fig, 'How long the leavers stayed', 'Churned CIFs only · dashed = median', name='20_lifetime_kde')

## 20.5 Accounts and products — what the leavers had

Read from the **closed** counts (a churned CIF has no active accounts). For every segment: how many accounts they ever had, and **how many had a DDA, a CD, a loan, any deposit (DD or CD)**, or only account types outside the extract. Active CIFs are shown next to them on the same "ever held" basis so the comparison is fair.

In [ ]:
def holdings_table(d, by):
    g = d.groupby(by)
    t = pd.DataFrame({'CIFs': g.size()})
    t['Avg accounts ever'] = g['TotalAccountsEver'].mean(); t['Median accounts ever'] = g['TotalAccountsEver'].median()
    t['% 1 account only'] = g['TotalAccountsEver'].apply(lambda s: s.eq(1).mean() * 100)
    t['Avg DD/CD/LN families'] = g['FamiliesEver'].mean()
    for col, lab in [('Had_Deposit', 'deposits (DD or CD)'), ('Had_DD', 'DDA'), ('Had_CD', 'CD'), ('Had_LN', 'loan'), ('Had_OtherEver', 'other (not in extract)')]:
        t[f'# had {lab}'] = g[col].sum().astype(int)
        t[f'% had {lab}'] = g[col].mean() * 100
    t['Avg DD accts'] = g['DD_Ever'].mean(); t['Avg CD accts'] = g['CD_Ever'].mean(); t['Avg LN accts'] = g['LN_Ever'].mean()
    return t

_cols = holdings_table(base.head(50), 'Status').columns
HOLD_PCT = ['% 1 account only'] + [c for c in _cols if c.startswith('% had')]
HOLD_INT = ['CIFs'] + [c for c in _cols if c.startswith('# had')]
HOLD_DEC = ['Avg accounts ever', 'Median accounts ever', 'Avg DD/CD/LN families', 'Avg DD accts', 'Avg CD accts', 'Avg LN accts']
def show_hold(t): display(styled(t, pct_cols=HOLD_PCT, int_cols=HOLD_INT, one_dec=HOLD_DEC, gradient_cols=['% had deposits (DD or CD)', '% had CD', '% had loan']))

display(Markdown('#### Churned vs. active — whole base'))
show_hold(holdings_table(base, 'Status').reindex(STATUS_ORDER))
for dims in CH_GRAN:
    display(Markdown(f'#### Churned CIFs by {" × ".join(dims)}'))
    t = holdings_table(ch, dims)
    idx = pd.Index(ORDER[dims[0]], name=dims[0]) if len(dims) == 1 else pd.MultiIndex.from_product([ORDER[c] for c in dims], names=dims)
    show_hold(t.reindex(idx).dropna(subset=['CIFs']))

cmp_rows = []
for g in order4:
    for stt in STATUS_ORDER:
        d_ = base[is_(base['Group2'], g) & base['Status'].eq(stt)]
        if len(d_) == 0: continue
        cmp_rows.append({'segment': g, 'status': stt, 'CIFs': len(d_), **{f'% had {f}': d_[f'Had_{f}'].mean() * 100 for f in FAM}, '% had other': d_['Had_OtherEver'].mean() * 100,
                         'Avg accounts ever': d_['TotalAccountsEver'].mean()})
display(Markdown('#### Ever held, churned vs. active, per IorB × ForD'))
display(styled(pd.DataFrame(cmp_rows).set_index(['segment', 'status']), pct_cols=[f'% had {f}' for f in FAM] + ['% had other'], int_cols=['CIFs'], one_dec=['Avg accounts ever']))

In [ ]:
prod_cols = {'Deposits (DD or CD)': 'Had_Deposit', 'DDA': 'Had_DD', 'CD': 'Had_CD', 'Loan': 'Had_LN', 'Other (not in extract)': 'Had_OtherEver'}
fig, axes = plt.subplots(1, 3, figsize=(17, 3.8))
for ax, dim in zip(axes, ['IorB', 'ForD', 'ForeignSplit']):
    order, pal = group_spec([dim])
    ct = pd.DataFrame({k: ch.groupby(dim)[v].mean() * 100 for k, v in prod_cols.items()}).T.reindex(columns=order)
    grouped_bars(ax, ct, order, pal, fmt=lambda v: f'{v:.0f}%'); ax.set_ylim(0, 115); ax.yaxis.set_major_formatter(PCT0)
    style(ax, title=f'Churned · % that ever had each product, by {dim}'); ax.tick_params(axis='x', labelrotation=15)
finish(fig, 'What the leavers had', 'Share of churned CIFs that ever held each product (active + closed accounts)', name='20_products_churned')

fig, axes = plt.subplots(1, 2, figsize=(15, 3.6))
for ax, (series, cats, ttl) in zip(axes, [(ch['AccountsEverBand'], ACC_EVER_BANDS, 'Accounts ever opened'), (ch['FamiliesEver'].astype(str), ['0', '1', '2', '3'], 'DD / CD / LN families ever held')]):
    ct = pd.crosstab(series, ch['Group2'], normalize='columns').reindex(index=[c for c in cats if c in set(series.dropna())], columns=order4) * 100
    grouped_bars(ax, ct, order4, pal4); style(ax, title=f'Churned · {ttl}', ylabel='% of churned CIFs in group', pct_y=True)
finish(fig, 'How many accounts the leavers had', 'By IorB × ForD · share of churned CIFs in each group', name='20_accounts_churned')

ex = pd.crosstab(ch['Group2'], ch['ExitProfile'], normalize='index').reindex(index=order4, columns=EXIT_ORDER).fillna(0) * 100
ex_n = pd.crosstab(ch['Group2'], ch['ExitProfile']).reindex(index=order4, columns=EXIT_ORDER).fillna(0).astype(int)
ex_n.loc['All churned'] = ex_n.sum()
fig, ax = plt.subplots(figsize=(13, 0.9 + 0.55 * len(ex)))
left = np.zeros(len(ex))
for p in EXIT_ORDER:
    vals = ex[p].values
    ax.barh(ex.index, vals, left=left, color=EXIT_COLOR[p], label=p, height=0.62, edgecolor=INK['surface'], linewidth=1)
    for i, (l, v) in enumerate(zip(left, vals)):
        if v >= 7: ax.text(l + v / 2, i, f'{v:.0f}%', ha='center', va='center', fontsize=7.8, color=text_on(EXIT_COLOR[p]))
    left += vals
ax.invert_yaxis(); ax.set_xlim(0, 100); ax.xaxis.set_major_formatter(PCT0); ax.grid(axis='y', visible=False)
ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.45), ncol=7)
finish(fig, 'Exit profile — which relationship ended', 'What each churned CIF ever held. LN only ≈ loan paid off / matured; CD only ≈ CD matured, not renewed.', name='20_exit_profile')
print('Churned CIFs by exit profile (counts):'); display(ex_n)

ex_life = ch.groupby('ExitProfile').agg(CIFs=('CIF', 'size'), MedianLife=('LifetimeYears', 'median'), Under1=('LifetimeYears', lambda s: (s < 1).mean() * 100),
                                         Accounts=('TotalAccountsEver', 'mean'), Business=('IorB', lambda s: is_(s, 'Business').mean() * 100),
                                         Foreign=('ForD', lambda s: is_(s, 'Foreign').mean() * 100)).reindex(EXIT_ORDER).dropna(subset=['CIFs'])
ex_life = ex_life.rename(columns={'MedianLife': 'Median lifetime (y)', 'Under1': '% left in < 1 y', 'Accounts': 'Avg accounts ever', 'Business': '% Business', 'Foreign': '% Foreign'})
display(styled(ex_life, int_cols=['CIFs'], one_dec=['Median lifetime (y)', 'Avg accounts ever'], pct_cols=['% left in < 1 y', '% Business', '% Foreign']))

## 20.6 Who leaves vs. who stays — attrition share by characteristic

Attrition share (churned ÷ active + churned) cut by every stable characteristic the file has. Age is measured **at the end of the relationship** (at churn for the leavers, at the snapshot for the active), so a customer who left at 40 is not counted as 50 today. Bars below the dotted line retain better than average; above it, worse. Branch group is the one on file today (for a churned CIF, its last one).

In [ ]:
def rate_by(d, col, order=None, min_n=30):
    t = d.groupby(d[col].fillna('(missing)'))['Churned'].agg(['mean', 'size'])
    t = t[t['size'] >= min_n]
    if order is not None: t = t.reindex([o for o in order if o in t.index] + [o for o in t.index if o not in order])
    return t.assign(rate=t['mean'] * 100)

base['HadLoanTxt'] = np.where(base['Had_LN'], 'Had a loan', 'No loan')
base['Families'] = base['FamiliesEver'].astype(str) + ' famil' + np.where(base['FamiliesEver'].eq(1), 'y', 'ies')
base['Branch group'] = lump(base['BranchGroupName'].fillna('(missing)'), 8)
COHORT_BANDS = ['< 2000', '2000–04', '2005–09', '2010–14', '2015–19', '2020+']
base['Cohort'] = pd.cut(base['CohortYear'], [0, 2000, 2005, 2010, 2015, 2020, 2100], right=False, labels=COHORT_BANDS).astype(object)
FAM_BANDS = ['0 families', '1 family', '2 families', '3 families']
panels = [('AgeAtEndBand', 'Individuals · age at end', AGE_BANDS, is_(base['IorB'], 'Individual')),
          ('BusinessAgeBand', 'Business · years since incorporation', YEAR_BANDS, is_(base['IorB'], 'Business')),
          ('AccountsEverBand', 'Accounts ever opened', ACC_EVER_BANDS, None),
          ('Families', 'DD / CD / LN families ever held', FAM_BANDS, None),
          ('ExitProfile', 'Products ever held (profile)', EXIT_ORDER, None),
          ('HadLoanTxt', 'Ever had a loan', ['Had a loan', 'No loan'], None),
          ('ForeignSplit', 'Domestic / Venezuela / other foreign', ORDER['ForeignSplit'], None),
          ('Region', 'Region', REGION_ORDER, None),
          ('Branch group', 'Branch group (top 8, as on file)', None, None),
          ('Cohort', 'Vintage (CIF creation year)', COHORT_BANDS, None)]
ncols = 2; nrows = math.ceil(len(panels) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 2.9 * nrows))
for ax, (col, ttl, order, mask) in zip(axes.flat, panels):
    d_ = base if mask is None else base[mask]
    t = rate_by(d_, col, order)
    if len(t) == 0: ax.axis('off'); continue
    hbar(ax, [f'{i}  (n={n:,})' for i, n in zip(t.index, t['size'])], t['rate'].values, [RED if r > overall * 1.25 else BLUE if r < overall * 0.8 else GRAY for r in t['rate']], fmt=fmt_pct)
    ax.axvline(overall, color=RED, lw=1, ls=':'); ax.xaxis.set_major_formatter(PCT0); style(ax, title=ttl)
for ax in list(axes.flat)[len(panels):]: ax.axis('off')
finish(fig, 'Attrition share by characteristic', f'Churned ÷ (active + churned) · red = > 1.25× the average ({overall:.1f}%), blue = < 0.8×, gray = around it · groups with < 30 CIFs hidden', name='20_drivers')

for col, order in [('AccountsEverBand', ACC_EVER_BANDS), ('Families', FAM_BANDS), ('ExitProfile', EXIT_ORDER)]:
    pv = base.pivot_table(index=col, columns='Group2', values='Churned', aggfunc='mean').reindex(index=[o for o in order if o in set(base[col])], columns=order4) * 100
    nn = base.pivot_table(index=col, columns='Group2', values='Churned', aggfunc='size').reindex_like(pv)
    print(f'Attrition share % by {col} × IorB × ForD (cells with < 30 CIFs blanked):')
    display(pv.where(nn >= 30).style.format(fmt_pct, na_rep='').background_gradient(cmap=CMAP_RED, vmin=0, vmax=100))

## 20.7 Churned vs. active — are the differences real?

Same method as §14.1: Mann-Whitney U for direction, **Cliff's δ** for size (A = churned, B = active; δ > 0 means leavers tend to have *more*). Lifetime of active CIFs is censored, so the tenure row understates how long active relationships will last — read it together with the survival curves in §20.4.

In [ ]:
from scipy.stats import mannwhitneyu
def _cliff(a, b):
    a = pd.Series(a).dropna(); b = pd.Series(b).dropna()
    if len(a) < 10 or len(b) < 10: return np.nan, np.nan
    u = mannwhitneyu(a, b, alternative='two-sided')
    return float(u.pvalue), float(2 * u.statistic / (len(a) * len(b)) - 1)
def _effect(d):
    if pd.isna(d): return 'n too small'
    a = abs(d); return 'negligible' if a < 0.147 else 'small' if a < 0.33 else 'medium' if a < 0.474 else 'large'
def _p(v): return '' if pd.isna(v) else ('<0.001' if v < 0.001 else f'{v:.3f}')

rows = []
measures = [('LifetimeYears', 'Relationship years (to churn / to snapshot)'), ('TotalAccountsEver', 'Accounts ever'), ('FamiliesEver', 'DD / CD / LN families ever'),
            ('DD_Ever', 'DDAs ever'), ('CD_Ever', 'CDs ever'), ('LN_Ever', 'Loans ever'), ('AgeAtEnd', 'Age at end (Individuals)'), ('BusinessAge', 'Years since incorporation (Business)')]
for g in ['All CIFs'] + order4:
    d_ = base if g == 'All CIFs' else base[is_(base['Group2'], g)]
    a, b = d_[d_['Status'].eq('Churned')], d_[d_['Status'].eq('Active')]
    for col, lab in measures:
        pval, dlt = _cliff(a[col], b[col])
        if pd.isna(dlt): continue
        rows.append({'segment': g, 'measure': lab, 'n churned': int(a[col].notna().sum()), 'n active': int(b[col].notna().sum()),
                     'median churned': a[col].median(), 'median active': b[col].median(), 'mean churned': a[col].mean(), 'mean active': b[col].mean(),
                     'p (Mann-Whitney)': pval, "Cliff's δ (churned vs active)": dlt, 'effect': _effect(dlt)})
ctests = pd.DataFrame(rows).set_index(['segment', 'measure'])
display(ctests.style.format({'median churned': fmt_1, 'median active': fmt_1, 'mean churned': '{:.2f}', 'mean active': '{:.2f}', 'n churned': fmt_int, 'n active': fmt_int,
                             'p (Mann-Whitney)': _p, "Cliff's δ (churned vs active)": '{:+.2f}'}, na_rep='')
        .background_gradient(cmap=CMAP_DIV, subset=["Cliff's δ (churned vs active)"], vmin=-1, vmax=1))

rows = []
for g in ORDER['IorB']:
    d_ = base[is_(base['IorB'], g)]
    for f, col in [('DD', 'Had_DD'), ('CD', 'Had_CD'), ('LN', 'Had_LN'), ('Other', 'Had_OtherEver')]:
        ct = pd.crosstab(d_['Status'], d_[col])
        pval = chi2_contingency(ct)[1] if ct.shape == (2, 2) else np.nan
        rows.append({'IorB': g, 'product (ever)': f, '% holding · churned': d_.loc[d_['Status'].eq('Churned'), col].mean() * 100,
                     '% holding · active': d_.loc[d_['Status'].eq('Active'), col].mean() * 100, 'p (chi-square)': pval, "Cramér's V": cramers_v(d_['Status'], d_[col])})
display(pd.DataFrame(rows).set_index(['IorB', 'product (ever)']).style.format({'% holding · churned': fmt_pct, '% holding · active': fmt_pct,
        'p (chi-square)': _p, "Cramér's V": '{:.2f}'}, na_rep=''))

## 20.8 Recent leavers — the last 12 and 24 months

The customers whose last account closed recently are the ones still worth a call: the relationship is fresh, the officer is probably still there, and the reason can still be asked. Residual balances on churned CIFs (if any) are listed because they usually mean a closure that was not completed.

In [ ]:
rec = ch[ch['MonthsSinceChurn'].notna()]
r12, r24 = rec[rec['MonthsSinceChurn'] <= 12], rec[rec['MonthsSinceChurn'] <= 24]
print(f'Churned in the last 12 months: {len(r12):,} CIFs · last 24 months: {len(r24):,} CIFs (of {len(ch):,} churned)')
rt = pd.DataFrame({'Last 12m': r12['Group2'].value_counts(), 'Last 24m': r24['Group2'].value_counts()}).reindex(order4).fillna(0).astype(int)
for lab, d_ in [('12m', r12), ('24m', r24)]:
    g = d_.groupby('Group2')
    rt[f'Median lifetime (y) · {lab}'] = g['LifetimeYears'].median()
    rt[f'Avg accounts ever · {lab}'] = g['TotalAccountsEver'].mean()
    rt[f'% had loan · {lab}'] = g['Had_LN'].mean() * 100
    rt[f'% had CD · {lab}'] = g['Had_CD'].mean() * 100
    rt[f'% had deposits · {lab}'] = g['Had_Deposit'].mean() * 100
rt.loc['All churned'] = [len(r12), len(r24)] + [np.nan] * (rt.shape[1] - 2)
display(styled(rt, int_cols=['Last 12m', 'Last 24m'], one_dec=[c for c in rt.columns if c.startswith(('Median', 'Avg'))], pct_cols=[c for c in rt.columns if c.startswith('%')]))

if len(r24):
    mon = r24.assign(m=r24['ChurnDate'].dt.to_period('M').dt.to_timestamp()).groupby(['m', 'IorB']).size().unstack().reindex(columns=ORDER['IorB']).fillna(0)
    fig, ax = plt.subplots(figsize=(14, 3.4)); bottom = np.zeros(len(mon))
    for g in ORDER['IorB']:
        ax.bar(mon.index, mon[g].values, bottom=bottom, width=20, color=COLOR['IorB'][g], label=g, edgecolor=INK['surface'], linewidth=0.4); bottom += mon[g].values
    ax.legend(fontsize=8); style(ax, title='CIFs whose last account closed, per month', ylabel='CIFs'); ax.tick_params(axis='x', labelrotation=45)
    finish(fig, 'Recent attrition — last 24 months', f'Snapshot {SNAPSHOT.date()} · the last month may be incomplete', name='20_recent')

resid = ch[ch[BAL_COLS].fillna(0).abs().sum(axis=1) > 0]
if len(resid):
    print(f'Churned CIFs with a residual balance: {len(resid):,} · DD {money(resid["DD Accounts Balance"].sum())} · CD {money(resid["CD Accounts Balance"].sum())} · LN {money(resid["LN Accounts Balance"].sum())}')
    display(resid.nlargest(10, 'Deposits')[[ID_COL, 'IorB', 'ForD', 'Country', 'BranchGroupName', 'LastAccountCloseDate'] + BAL_COLS])

## 20.9 Export and key numbers

In [ ]:
CHURN_EXPORT_COLS = [ID_COL, 'CIF_Status', 'Status', 'IorB', 'ForD', 'Group2', 'BranchGroupName', 'BranchName', 'OfficerCode', 'Country', 'Region', 'ForeignSplit',
                     'Age', 'AgeAtEnd', 'AgeAtEndBand', 'BusinessAge', 'BusinessAgeBand', 'RelationshipStart', 'CohortYear', 'ChurnDate', 'ChurnYear', 'LifetimeYears', 'LifetimeBand',
                     'YearsSinceChurn', 'TotalAccountsEver', 'ClosedAccountsCount', 'DD_Ever', 'CD_Ever', 'LN_Ever', 'OtherEver', 'Had_DD', 'Had_CD', 'Had_LN', 'Had_Deposit',
                     'Had_OtherEver', 'FamiliesEver', 'MixEver', 'ExitProfile'] + BAL_COLS
if EXPORT_CHURN:
    ch[[c for c in CHURN_EXPORT_COLS if c in ch.columns]].to_csv(CHURN_PATH, index=False)
    print(f'Churned CIF table exported → {CHURN_PATH} ({len(ch):,} rows)')

def _share(mask): return f'{mask.sum():,} ({mask.mean():.0%})'
def _rate(key, lvl):
    t = attr_tables[key]; return f'{t.loc[lvl, "Attrition share %"]:.1f}%' if lvl in t.index else 'n/a'
worst = t4['Attrition share %'].idxmax(); best = t4['Attrition share %'].idxmin()
last_full = y_end - 1
churn_lines = [
    f'Lost base: {len(ch):,} churned CIFs vs {len(act_):,} active → attrition share {len(ch) / len(base):.1%} of everyone ever onboarded ({full["Status"].eq("Prospect").sum():,} prospects excluded).',
    f'By IorB: Individual {_rate("IorB", "Individual")} · Business {_rate("IorB", "Business")}. By ForD: Domestic {_rate("ForD", "Domestic")} · Foreign {_rate("ForD", "Foreign")}. '
    f'By country split: Venezuela {_rate("ForeignSplit", "Venezuela")} · Other foreign {_rate("ForeignSplit", "Other foreign")}.',
    f'Highest attrition share: {SEP.join(worst)} ({t4.loc[worst, "Attrition share %"]:.1f}%); lowest: {SEP.join(best)} ({t4.loc[best, "Attrition share %"]:.1f}%).',
    f'Annual attrition rate {last_full}: {rates_all.loc[last_full, "annual attrition %"]:.1f}% ({rates_all.loc[last_full, "churned in year"]:,} CIFs); {y_end} annualized: {rates_all.loc[y_end, "annual attrition %"]:.1f}%.',
    f'Tenure: churned CIFs stayed a median {ch["LifetimeYears"].median():.1f} years (P25–P75 {ch["LifetimeYears"].quantile(.25):.1f}–{ch["LifetimeYears"].quantile(.75):.1f}); '
    f'{(ch["LifetimeYears"] < 1).mean():.0%} left within the first year and {(ch["LifetimeYears"] >= 10).mean():.0%} after 10+ years. '
    f'Survival curve: {km_at(x_all, s_all, 5):.0f}% of relationships last 5 years.',
    f'Accounts: churned CIFs had on average {ch["TotalAccountsEver"].mean():.1f} accounts ever (active CIFs {act_["TotalAccountsEver"].mean():.1f}); {ch["TotalAccountsEver"].eq(1).mean():.0%} had a single account.',
    f'Products ever held by the churned: deposits (DD or CD) {_share(ch["Had_Deposit"])} · DDA {_share(ch["Had_DD"])} · CD {_share(ch["Had_CD"])} · loan {_share(ch["Had_LN"])} · only non-extract products {_share(ch["ExitProfile"].eq("Other only"))}.',
    f'Exit profile: ' + ' · '.join(f'{p} {v:.0%}' for p, v in ch['ExitProfile'].value_counts(normalize=True).reindex(EXIT_ORDER).dropna().items()) + '.',
    f'Recent: {len(r12):,} CIFs left in the last 12 months, {len(r24):,} in the last 24.',
]
print('\n'.join(f'{i + 1}. {l}' for i, l in enumerate(churn_lines)))

**Observations §20** *(fill in after the run — one line per question; the numbers come from the cell above and the tables in 20.1–20.8)*

- **Size of the lost base:** … churned CIFs, … % of everyone ever onboarded. Data checks: …
- **Which segments churn (IorB / ForD / Venezuela vs other foreign):** …
- **When:** annual attrition rate … ; trend …
- **Tenure:** median lifetime of the leavers … ; % who left in year one … ; survival at 5 years by segment …
- **Accounts:** average accounts ever … ; % single-account …
- **Loans / CDs / deposits:** of the churned, … had deposits, … a DDA, … a CD, … a loan. Exit profile: … % `LN only` (natural run-off), … % `CD only` (maturity), … % DDA relationships (the regrettable part).
- **Who leaves vs who stays:** strongest cuts …
- **Recent leavers:** … in the last 12 months, mostly …

**Data to request for attrition** (adds to §19)

1. **CIF close / inactivation date and reason code** — the churn date here is a proxy (last account closed).
2. **Balance at closure** (or the last 12 monthly average balances before it) — the file shows how many customers left, not how much money left with them.
3. **Loan maturity / payoff flag and CD maturity / rollover flag** — to separate natural run-off from customers who chose to leave.
4. **Whether old inactive CIFs are purged** from the source system, and from when — needed to trust the early years of §20.3 and the survival curves.
5. **History of `Banesco Select` and `Segment`** (monthly or at least at closure) — without it, attrition cannot be split by Select or segment tier, which is why §20 does not use them.

# 21. The customer over time — base bridge, how the customer has changed, cohort retention

§20 looked at the leavers. This section puts leavers and stayers on one timeline and **rebuilds the customer base year by year**: a CIF counts as a customer on a date if its relationship had started and it had not churned yet. The rebuild gives three views a single snapshot cannot:

1. **Base bridge** — customers on 1 January + new − lost = customers on 31 December, per year. The banking "book walk", in customers.
2. **Who the customer has been** — the mix of the base (business, foreign, Venezuelan), the median age of individuals and the median tenure *at each point in time*; and the profile of the customers acquired each year.
3. **Cohort retention** — of the customers who started in year *Y*, how many were still customers 1, 2, … 10 years later.

Like §20, only attributes that do not change are used (IorB, ForD, country, birth date, dates). **Caveat:** CIFs that the source system purged before the extract are invisible, so early years may look smaller and better-retained than they were.

In [ ]:
HIST_YEARS = 15                     # years shown in the bridge and the cohort views
hist = base[base['RelationshipStart'].notna() & (base['Status'].eq('Active') | base['ChurnDate'].notna())].copy()
bad_dates = hist['ChurnDate'].lt(hist['RelationshipStart']) | hist['ChurnDate'].gt(SNAPSHOT) | hist['RelationshipStart'].gt(SNAPSHOT)
hist = hist[~bad_dates]
print(f'History rebuilt from {len(hist):,} of {len(base):,} active + churned CIFs ({len(base) - len(hist):,} without a usable start / churn date are left out)')

T_NOW = SNAPSHOT + pd.Timedelta(days=1)
def alive_at(d, t):
    """Customer on date t: relationship started before t and not churned before t."""
    return (d['RelationshipStart'] < t) & (d['Status'].eq('Active') | (d['ChurnDate'] >= t))

def bridge(d, years):
    rows = []
    for Y in years:
        t0 = pd.Timestamp(Y, 1, 1); t1 = min(pd.Timestamp(Y + 1, 1, 1), T_NOW)
        new = d['RelationshipStart'].ge(t0) & d['RelationshipStart'].lt(t1)
        lost = d['ChurnDate'].ge(t0) & d['ChurnDate'].lt(t1)
        rows.append({'year': Y, 'Customers 1 Jan': int(alive_at(d, t0).sum()), 'New': int(new.sum()), 'Lost': int(lost.sum()), 'Customers end of year': int(alive_at(d, t1).sum())})
    t = pd.DataFrame(rows).set_index('year')
    t['Net change'] = t['New'] - t['Lost']
    t['Net growth %'] = t['Net change'] / t['Customers 1 Jan'].replace(0, np.nan) * 100
    t['Acquisition rate %'] = t['New'] / t['Customers 1 Jan'].replace(0, np.nan) * 100
    t['Loss rate %'] = t['Lost'] / t['Customers 1 Jan'].replace(0, np.nan) * 100
    return t

YEARS_H = list(range(SNAPSHOT.year - HIST_YEARS + 1, SNAPSHOT.year + 1))
br = bridge(hist, YEARS_H)
gap = (br['Customers 1 Jan'] + br['New'] - br['Lost'] - br['Customers end of year']).abs().sum()
print('Bridge identity (start + new − lost = end) holds for every year' if gap == 0 else f'⚠ Bridge identity off by {gap:,} CIFs — check dates')
print(f'{SNAPSHOT.year} runs to the snapshot ({SNAPSHOT.date()}). Loss rate here includes customers who joined and left in the same year, so it is slightly above the §20.3 rate.')
display(styled(br, int_cols=['Customers 1 Jan', 'New', 'Lost', 'Customers end of year', 'Net change'], pct_cols=['Net growth %', 'Acquisition rate %', 'Loss rate %']))

br_seg = {g: bridge(hist[is_(hist['Group2'], g)], YEARS_H) for g in order4}
print('Net growth % per year by IorB × ForD:')
display(pd.DataFrame({g: t['Net growth %'] for g, t in br_seg.items()}).T.style.format(fmt_pct, na_rep='').background_gradient(cmap=CMAP_DIV, vmin=-30, vmax=30, axis=None))

In [ ]:
# ---- Figures: the base over time and the yearly flows ----
pts = [pd.Timestamp(Y, 1, 1) for Y in YEARS_H] + [T_NOW]; pt_lab = [str(Y) for Y in YEARS_H] + ['Today']
comp_t = pd.DataFrame({lab: hist.loc[alive_at(hist, t), 'Group2'].value_counts() for lab, t in zip(pt_lab, pts)}).T.reindex(columns=order4).fillna(0)
fig, axes = plt.subplots(1, 3, figsize=(19, 4), gridspec_kw={'width_ratios': [1.1, 1.1, 1.2]})
x = np.arange(len(comp_t)); bottom = np.zeros(len(comp_t))
for g in order4:
    axes[0].bar(x, comp_t[g].values, bottom=bottom, color=pal4[g], label=g, edgecolor=INK['surface'], linewidth=0.4); bottom += comp_t[g].values
axes[0].set_xticks(x); axes[0].set_xticklabels(comp_t.index, rotation=45); style(axes[0], title='Customers on 1 January (and today)', ylabel='CIFs'); axes[0].legend(fontsize=7.5)
share_t = comp_t.div(comp_t.sum(axis=1).replace(0, np.nan), axis=0) * 100; bottom = np.zeros(len(share_t))
for g in order4:
    axes[1].bar(x, share_t[g].values, bottom=bottom, color=pal4[g], edgecolor=INK['surface'], linewidth=0.4); bottom += share_t[g].values
axes[1].set_xticks(x); axes[1].set_xticklabels(share_t.index, rotation=45); axes[1].set_ylim(0, 100); style(axes[1], title='Mix of the base', pct_y=True)
axes[2].bar(br.index, br['New'], color=BLUE, label='New')
axes[2].bar(br.index, -br['Lost'], color=RED, label='Lost')
axes[2].plot(br.index, br['Net change'], color=INK['primary'], marker='o', ms=4, lw=1.5, label='Net change')
axes[2].axhline(0, color=INK['muted'], lw=0.8); style(axes[2], title='New vs lost customers per year', ylabel='CIFs'); axes[2].legend(fontsize=7.5); axes[2].tick_params(axis='x', labelrotation=45)
finish(fig, 'The customer base over time', f'Rebuilt from start and churn dates · {SNAPSHOT.year} to the snapshot', name='21_bridge')

fig, axes = plt.subplots(1, len(order4), figsize=(4.4 * len(order4), 3.4), sharey=True)
for ax, g in zip(axes, order4):
    t = br_seg[g]
    ax.bar(t.index, t['New'], color=pal4[g]); ax.bar(t.index, -t['Lost'], color=RED, alpha=0.85)
    ax.plot(t.index, t['Net change'], color=INK['primary'], marker='o', ms=3, lw=1.2); ax.axhline(0, color=INK['muted'], lw=0.8)
    style(ax, title=g); ax.tick_params(axis='x', labelrotation=45)
finish(fig, 'New (up) vs lost (down, red) by IorB × ForD', 'Line = net change', name='21_bridge_segments')

In [ ]:
# ---- Who the customer has been: the base and the new customers, year by year ----
birth = hist['BIRTHDATE_TEXT'].where(hist['BIRTHDATE_TEXT'].notna(), SNAPSHOT - pd.to_timedelta(hist['AGE'] * 365.25, unit='D'))
h_ind = is_(hist['IorB'], 'Individual') & hist['Age'].notna()
rows = []
for lab, t in zip(pt_lab, pts):
    m = alive_at(hist, t); d_ = hist[m]
    rows.append({'point': lab, 'Customers': int(m.sum()), '% Business': is_(d_['IorB'], 'Business').mean() * 100, '% Foreign': is_(d_['ForD'], 'Foreign').mean() * 100,
                 '% Venezuela': is_(d_['ForeignSplit'], 'Venezuela').mean() * 100,
                 'Median age (Ind.)': ((t - birth[m & h_ind]).dt.days / 365.25).median(), 'Median tenure (y)': ((t - d_['RelationshipStart']).dt.days / 365.25).median()})
evo = pd.DataFrame(rows).set_index('point')
print('The customer base at each point in time:')
display(styled(evo, int_cols=['Customers'], pct_cols=['% Business', '% Foreign', '% Venezuela'], one_dec=['Median age (Ind.)', 'Median tenure (y)']))

acq = hist[hist['RelationshipStart'].dt.year.isin(YEARS_H)].copy(); acq['StartYear'] = acq['RelationshipStart'].dt.year
acq['AgeAtStart'] = ((acq['RelationshipStart'] - birth.reindex(acq.index)).dt.days / 365.25).where(h_ind.reindex(acq.index))
acq['BizAgeAtStart'] = (acq['BusinessAge'] - (SNAPSHOT - acq['RelationshipStart']).dt.days / 365.25).where(is_(acq['IorB'], 'Business')).clip(lower=0)
acq_tbl = acq.groupby('StartYear').agg(New=('CIF', 'size'), Business=('IorB', lambda s: is_(s, 'Business').mean() * 100), Foreign=('ForD', lambda s: is_(s, 'Foreign').mean() * 100),
                                       Venezuela=('ForeignSplit', lambda s: is_(s, 'Venezuela').mean() * 100), AgeAtStart=('AgeAtStart', 'median'),
                                       BizAgeAtStart=('BizAgeAtStart', 'median'), Accounts=('TotalAccountsEver', 'mean'), Loan=('Had_LN', lambda s: s.mean() * 100),
                                       StillHere=('Status', lambda s: s.eq('Active').mean() * 100)).reindex(YEARS_H)
acq_tbl = acq_tbl.rename(columns={'New': 'New customers', 'Business': '% Business', 'Foreign': '% Foreign', 'Venezuela': '% Venezuela', 'AgeAtStart': 'Median age at start (Ind.)',
                                  'BizAgeAtStart': 'Median business age at start', 'Accounts': 'Avg accounts ever', 'Loan': '% ever had a loan', 'StillHere': '% still a customer today'})
print('The customers acquired each year:')
display(styled(acq_tbl, int_cols=['New customers'], pct_cols=['% Business', '% Foreign', '% Venezuela', '% ever had a loan', '% still a customer today'],
               one_dec=['Median age at start (Ind.)', 'Median business age at start', 'Avg accounts ever'], gradient_cols=['% still a customer today']))

fig, axes = plt.subplots(2, 3, figsize=(18, 7.2))
xs = np.arange(len(evo))
for col, c in [('% Business', GREEN), ('% Foreign', BLUE), ('% Venezuela', BLUE_T)]:
    axes[0, 0].plot(xs, evo[col], color=c, marker='o', ms=3.5, lw=2, label=col.replace('% ', ''))
axes[0, 0].set_ylim(0, 100); axes[0, 0].yaxis.set_major_formatter(PCT0); style(axes[0, 0], title='Mix of the base'); axes[0, 0].legend(fontsize=7.5)
axes[0, 1].plot(xs, evo['Median age (Ind.)'], color=BLUE, marker='o', ms=3.5, lw=2); style(axes[0, 1], title='Median age of individual customers', ylabel='years')
axes[0, 2].plot(xs, evo['Median tenure (y)'], color=GREEN, marker='o', ms=3.5, lw=2); style(axes[0, 2], title='Median tenure of the base', ylabel='years')
for ax in axes[0]: ax.set_xticks(xs); ax.set_xticklabels(evo.index, rotation=45)
na = pd.crosstab(acq['StartYear'], acq['Group2']).reindex(index=YEARS_H, columns=order4).fillna(0); bottom = np.zeros(len(na))
for g in order4:
    axes[1, 0].bar(na.index, na[g].values, bottom=bottom, color=pal4[g], label=g, edgecolor=INK['surface'], linewidth=0.4); bottom += na[g].values
style(axes[1, 0], title='New customers per year', ylabel='CIFs'); axes[1, 0].legend(fontsize=7)
for col, c in [('% Business', GREEN), ('% Foreign', BLUE), ('% Venezuela', BLUE_T)]:
    axes[1, 1].plot(acq_tbl.index, acq_tbl[col], color=c, marker='o', ms=3.5, lw=2, label=col.replace('% ', ''))
axes[1, 1].set_ylim(0, 100); axes[1, 1].yaxis.set_major_formatter(PCT0); style(axes[1, 1], title='Mix of the new customers'); axes[1, 1].legend(fontsize=7.5)
axes[1, 2].plot(acq_tbl.index, acq_tbl['Median age at start (Ind.)'], color=BLUE, marker='o', ms=3.5, lw=2, label='Individuals · age at start')
axes[1, 2].plot(acq_tbl.index, acq_tbl['Median business age at start'], color=GREEN, marker='o', ms=3.5, lw=2, label='Business · years since incorporation at start')
style(axes[1, 2], title='Who we acquired: age at start', ylabel='years'); axes[1, 2].legend(fontsize=7.5)
for ax in axes[1]: ax.tick_params(axis='x', labelrotation=45)
finish(fig, 'Who the customer has been', 'Top row: the base on 1 January of each year · bottom row: the customers acquired in each year', name='21_evolution')

In [ ]:
# ---- Cohort retention ----
hist['StartYear'] = hist['RelationshipStart'].dt.year
MAX_K = 10
def retention_matrix(d, years, max_k=MAX_K):
    """% of each start-year cohort still a customer k years after its own start; blank where not every member has reached k."""
    m = pd.DataFrame(np.nan, index=years, columns=range(1, max_k + 1)); n = {}
    for Y in years:
        c = d[d['StartYear'].eq(Y)]; n[Y] = len(c)
        if len(c) == 0: continue
        for k in range(1, max_k + 1):
            if pd.Timestamp(Y + k + 1, 1, 1) > T_NOW: break
            anniv = c['RelationshipStart'] + pd.to_timedelta(k * 365.25, unit='D')
            m.loc[Y, k] = (c['Status'].eq('Active') | (c['ChurnDate'] >= anniv)).mean() * 100
    return m, pd.Series(n)

COH_YEARS = list(range(SNAPSHOT.year - HIST_YEARS, SNAPSHOT.year))
ret, ret_n = retention_matrix(hist, COH_YEARS)
ret.index = [f'{Y}  (n={ret_n[Y]:,})' for Y in COH_YEARS]
fig, ax = plt.subplots(figsize=(11, 0.6 + 0.38 * len(ret)))
sns.heatmap(ret.astype(float), annot=True, fmt='.0f', cmap=CMAP_BLUE, cbar=False, ax=ax, linewidths=1, linecolor=INK['surface'], annot_kws={'size': 8.5}, vmin=0, vmax=100)
ax.set_xlabel('years since the cohort started'); ax.set_ylabel('start year'); ax.grid(False)
finish(fig, 'Cohort retention — % still a customer k years after starting', 'Each row is the customers who started that year · blank = the cohort has not reached that age yet', name='21_cohorts')

# Retention at 1 / 3 / 5 / 10 years by cohort period and IorB × ForD (each CIF counted only once it has reached the anniversary)
def retention_at(d, k):
    anniv = d['RelationshipStart'] + pd.to_timedelta(k * 365.25, unit='D')
    elig = anniv <= SNAPSHOT
    if elig.sum() < 30: return np.nan
    return (d.loc[elig, 'Status'].eq('Active') | (d.loc[elig, 'ChurnDate'] >= anniv[elig])).mean() * 100
PERIODS = ['< 2005', '2005–09', '2010–14', '2015–19', '2020+']
hist['StartPeriod'] = pd.cut(hist['StartYear'], [0, 2005, 2010, 2015, 2020, 2100], right=False, labels=PERIODS).astype(object)
rows = []
for g in order4 + ['All CIFs']:
    d_g = hist if g == 'All CIFs' else hist[is_(hist['Group2'], g)]
    for p in PERIODS:
        d_ = d_g[d_g['StartPeriod'].eq(p)]
        rows.append({'segment': g, 'start period': p, 'CIFs': len(d_), **{f'retained after {k}y %': retention_at(d_, k) for k in [1, 3, 5, 10]}})
ret_p = pd.DataFrame(rows).set_index(['segment', 'start period'])
display(styled(ret_p, int_cols=['CIFs'], pct_cols=[c for c in ret_p.columns if c.startswith('retained')], gradient_cols=[c for c in ret_p.columns if c.startswith('retained')]))

fig, axes = plt.subplots(1, len(order4), figsize=(4.4 * len(order4), 3.5), sharey=True)
per_colors = dict(zip(PERIODS, [CMAP_BLUE(v) for v in np.linspace(0.3, 1.0, len(PERIODS))]))
for ax, g in zip(axes, order4):
    d_g = hist[is_(hist['Group2'], g)]
    for p in PERIODS:
        d_ = d_g[d_g['StartPeriod'].eq(p)]
        ys = [retention_at(d_, k) for k in range(1, MAX_K + 1)]
        if np.all(np.isnan(ys)): continue
        ax.plot(range(1, MAX_K + 1), ys, color=per_colors[p], marker='o', ms=3, lw=2, label=p)
    ax.set_ylim(0, 100); ax.yaxis.set_major_formatter(PCT0); style(ax, title=g, xlabel='years since start'); ax.legend(fontsize=7, title='started', title_fontsize=7)
finish(fig, 'Are newer cohorts retained better?', 'Retention curve per start period · darker = more recent', name='21_cohort_periods')

**Observations §21** *(fill in after the run)*

- **Base bridge:** the base grew / shrank from … to … ; best year … ; years with more customers lost than won …
- **Who the customer has been:** share of business … → … ; foreign … → … ; Venezuela … → … ; median age of individuals … → …
- **Who we acquired:** the newest customers are … (age, business share, country) vs. the ones acquired 10 years ago …
- **Cohort retention:** after 5 years … % of a cohort is still here; newer cohorts retain better / worse …

# 22. BCG matrix — which segments grow and which hold the money

The Boston Consulting Group growth-share matrix, adapted to a customer book:

- **x-axis — share:** the segment's share of today's total balance (deposits + loans) — how much of the book it holds. The vertical line is the "fair share" (100% ÷ number of segments).
- **y-axis — growth:** the annual growth of the segment's **number of customers** over the last `BCG_YEARS` years, rebuilt from start and churn dates (§21). The horizontal line is the growth of the whole bank.
- **Bubble size:** customers today.

| Quadrant | Meaning for a bank | Typical action |
|---|---|---|
| **Star** — high share, high growth | the book of the future, already large | invest, protect pricing and service |
| **Cash cow** — high share, low growth | holds the money but is not growing | retain, defend balances, cross-sell |
| **Question mark** — low share, high growth | growing fast, little money yet | decide: deepen the relationships or stop chasing |
| **Dog** — low share, low growth | small and shrinking | serve efficiently, review cost-to-serve |

Growth uses customer counts because the file has no balance history (a balance-growth BCG needs monthly balances — see §19). The table adds a money signal: the **new-money share**, the part of today's balance brought by customers who joined inside the window. Segments are built only from stable attributes, so a churned customer lands in the same segment it had when it left. Branch group is the one on file today and is shown with that caveat.

In [ ]:
BCG_YEARS   = 3                     # growth window (years)
BCG_MIN_CIFS = 30                   # segments smaller than this at the start of the window are not plotted
BCG_TOP_N   = 15                    # max segments per chart (largest by customers today)
QUAD_COLOR  = {'Star': GREEN, 'Cash cow': BLUE, 'Question mark': BLUE_T, 'Dog': GRAY}
T_THEN = SNAPSHOT - pd.DateOffset(years=BCG_YEARS)

def cagr(now, then): return ((now / then) ** (1 / BCG_YEARS) - 1) * 100
bank_growth = cagr(alive_at(hist, T_NOW).sum(), alive_at(hist, T_THEN).sum())
print(f'Whole bank: customers {alive_at(hist, T_THEN).sum():,} on {T_THEN.date()} → {alive_at(hist, T_NOW).sum():,} today = {bank_growth:+.1f}% a year')

def bcg_table(unit):
    """unit: Series aligned with `base` giving the segment of each CIF."""
    h = hist.assign(_u=unit.reindex(hist.index).fillna('(missing)'))
    a = base[base['Status'].eq('Active')].assign(_u=unit.reindex(base.index).fillna('(missing)'))
    t = pd.DataFrame({'CIFs then': h[alive_at(h, T_THEN)].groupby('_u').size(), 'CIFs today': h[alive_at(h, T_NOW)].groupby('_u').size(),
                      'New in window': h[h['RelationshipStart'].ge(T_THEN)].groupby('_u').size(), 'Lost in window': h[h['ChurnDate'].ge(T_THEN)].groupby('_u').size(),
                      'Balance today': a.groupby('_u')['TotalBalance'].sum(), 'Deposits today': a.groupby('_u')['Deposits'].sum(), 'Loans today': a.groupby('_u')['Loans'].sum(),
                      'New money': a[a['RelationshipStart'].ge(T_THEN)].groupby('_u')['TotalBalance'].sum()}).fillna(0)
    t['Share of balance %'] = t['Balance today'].clip(lower=0) / t['Balance today'].clip(lower=0).sum() * 100
    t = t[t['CIFs then'] >= BCG_MIN_CIFS].sort_values('CIFs today', ascending=False).head(BCG_TOP_N)
    t['CIF growth % a year'] = cagr(t['CIFs today'], t['CIFs then'])
    t['Acquisition % a year'] = t['New in window'] / t['CIFs then'] / BCG_YEARS * 100
    t['Loss % a year'] = t['Lost in window'] / t['CIFs then'] / BCG_YEARS * 100
    t['Relative share'] = t['Share of balance %'] / t['Share of balance %'].max()
    t['New-money share %'] = t['New money'] / t['Balance today'].replace(0, np.nan) * 100
    fair = 100 / max(len(t), 1)
    hi_s, hi_g = t['Share of balance %'] >= fair, t['CIF growth % a year'] >= bank_growth
    t['Quadrant'] = np.select([hi_s & hi_g, hi_s & ~hi_g, ~hi_s & hi_g], ['Star', 'Cash cow', 'Question mark'], 'Dog')
    t.attrs['fair'] = fair
    return t

def bcg_plot(t, title, name):
    fig, ax = plt.subplots(figsize=(12, 6.2))
    xv = t['Share of balance %'].clip(lower=0.05); yv = t['CIF growth % a year']
    sizes = 120 + 2400 * t['CIFs today'] / t['CIFs today'].max()
    ax.scatter(xv, yv, s=sizes, c=[QUAD_COLOR[q] for q in t['Quadrant']], alpha=0.72, edgecolors=INK['surface'], linewidths=1.2, zorder=3)
    for lab, xx, yy in zip(t.index, xv, yv):
        ax.annotate(lab, (xx, yy), xytext=(6, 6), textcoords='offset points', fontsize=7.8, color=INK['primary'], zorder=4)
    ax.set_xscale('log'); pctf = FuncFormatter(lambda v, p: f'{v:g}%'); ax.xaxis.set_major_formatter(pctf); ax.xaxis.set_minor_formatter(pctf)
    ax.tick_params(axis='x', which='minor', labelsize=7.5, labelcolor=INK['muted']); ax.set_xlim(xv.min() / 1.6, xv.max() * 2.2)
    ypad = max((yv.max() - yv.min()) * 0.15, 1); ax.set_ylim(min(yv.min(), bank_growth) - ypad, max(yv.max(), bank_growth) + ypad)
    ax.axvline(t.attrs['fair'], color=INK['muted'], lw=1, ls='--'); ax.axhline(bank_growth, color=INK['muted'], lw=1, ls='--')
    lo, hi = ax.get_xlim(); ylo, yhi = ax.get_ylim(); pad = (yhi - ylo) * 0.04
    for q, (xx, yy, ha, va) in {'Star': (hi, yhi - pad, 'right', 'top'), 'Cash cow': (hi, ylo + pad, 'right', 'bottom'),
                                'Question mark': (lo, yhi - pad, 'left', 'top'), 'Dog': (lo, ylo + pad, 'left', 'bottom')}.items():
        ax.text(xx, yy, q.upper(), ha=ha, va=va, fontsize=10, fontweight='bold', color=QUAD_COLOR[q], alpha=0.9)
    ax.grid(axis='x', color=INK['grid'])
    style(ax, xlabel='share of today\'s total balance (log scale) · dashed = fair share', ylabel=f'customer growth, % a year (last {BCG_YEARS} y) · dashed = bank')
    finish(fig, title, f'Bubble = customers today · {T_THEN.date()} → {SNAPSHOT.date()} · segments with < {BCG_MIN_CIFS} customers at the start are left out', name=name)
    show = t[['Quadrant', 'CIFs then', 'CIFs today', 'CIF growth % a year', 'Acquisition % a year', 'Loss % a year', 'Balance today', 'Share of balance %',
              'Relative share', 'Deposits today', 'Loans today', 'New-money share %']]
    display(styled(show, int_cols=['CIFs then', 'CIFs today'], money_cols=['Balance today', 'Deposits today', 'Loans today'],
                   pct_cols=['CIF growth % a year', 'Acquisition % a year', 'Loss % a year', 'Share of balance %', 'New-money share %']).format({'Relative share': '{:.2f}'}))

BCG_UNITS = {
    'IorB × Domestic / Venezuela / other foreign': make_label(base, ['IorB', 'ForeignSplit']),
    'IorB × Region': make_label(base, ['IorB', 'Region']),
    'Country': base['Country'].fillna('(missing)'),
    'Branch group (as on file today)': base['BranchGroupName'].fillna('(missing)'),
}
bcg_results = {}
for k, (title, unit) in enumerate(BCG_UNITS.items()):
    display(Markdown(f'#### BCG · {title}'))
    t = bcg_table(unit); bcg_results[title] = t
    if len(t) >= 2: bcg_plot(t, f'BCG matrix · {title}', f'22_bcg_{k}')
    else: print('Not enough segments above the minimum size.')
bcg2 = bcg_table(base['Group2'])            # used in the synthesis (§26)

**Observations §22** *(fill in after the run)*

- **Stars:** … — growing and already large: where to invest.
- **Cash cows:** … — hold the money, not growing: retention and cross-sell.
- **Question marks:** … — growing customer counts with little balance: are they worth deepening?
- **Dogs:** … — small and shrinking.
- **New-money share:** the segments where most of today's balance arrived in the last … years are …

# 23. Customer value matrix — value × relationship depth

The classic relationship-banking grid for the **active** customers: how much the customer is worth to the bank today (value) against how deep the relationship is (number of active accounts, all types, including those outside the extract). Each of the nine cells gets a standard action. Here Select can be used again, because it describes today.

| | **Single** (1 account) | **Moderate** (2–3) | **Deep** (4+) |
|---|---|---|---|
| **High value** (top 10%) | Protect & deepen — large balance hanging on one product | Deepen | Retain — core |
| **Mid value** (next 40%) | Grow | Develop | Maintain |
| **Low value** (bottom 50%) | Serve efficiently | Activate / reprice | Review cost-to-serve |

Value is ranked **within IorB** by total balance (deposits + loans), so individuals are compared with individuals and businesses with businesses (`VALUE_WITHIN_IORB`).

In [ ]:
VALUE_WITHIN_IORB = True
DEPTH_CUTS   = (1, 3)                          # ≤ 1 active account = Single, 2–3 = Moderate, 4+ = Deep
VALUE_LEVELS = ['High value', 'Mid value', 'Low value']
DEPTH_LEVELS = ['Single', 'Moderate', 'Deep']
ACTION = {('High value', 'Single'): 'Protect & deepen', ('High value', 'Moderate'): 'Deepen', ('High value', 'Deep'): 'Retain — core',
          ('Mid value', 'Single'): 'Grow', ('Mid value', 'Moderate'): 'Develop', ('Mid value', 'Deep'): 'Maintain',
          ('Low value', 'Single'): 'Serve efficiently', ('Low value', 'Moderate'): 'Activate / reprice', ('Low value', 'Deep'): 'Review cost-to-serve'}
ACTION_ORDER = list(ACTION.values())
ACTION_COLOR = dict(zip(ACTION_ORDER, [CMAP_GREEN(0.95), CMAP_GREEN(0.7), CMAP_GREEN(0.45), CMAP_BLUE(0.95), CMAP_BLUE(0.7), CMAP_BLUE(0.45), '#8B95A5', '#B4BBC6', '#D7DBE1']))

rk = (df.groupby('IorB')['TotalBalance'].rank(ascending=False, method='first', pct=True) if VALUE_WITHIN_IORB
      else df['TotalBalance'].rank(ascending=False, method='first', pct=True))
df['ValueLevel'] = np.select([rk <= 0.10, rk <= 0.50], VALUE_LEVELS[:2], 'Low value')
acc_ = df['ActiveAccountsCount'].fillna(0)
df['DepthLevel'] = np.select([acc_ <= DEPTH_CUTS[0], acc_ <= DEPTH_CUTS[1]], DEPTH_LEVELS[:2], 'Deep')
df['ValueAction'] = [ACTION[(v, d)] for v, d in zip(df['ValueLevel'], df['DepthLevel'])]

fig, axes = plt.subplots(1, 2, figsize=(16, 4.6))
for ax, g in zip(axes, ORDER['IorB']):
    d_ = df[is_(df['IorB'], g)]
    n = pd.crosstab(d_['ValueLevel'], d_['DepthLevel']).reindex(index=VALUE_LEVELS, columns=DEPTH_LEVELS).fillna(0)
    bal = d_.pivot_table(index='ValueLevel', columns='DepthLevel', values='TotalBalance', aggfunc='sum').reindex(index=VALUE_LEVELS, columns=DEPTH_LEVELS).fillna(0).clip(lower=0)
    bal_pct = bal / max(bal.values.sum(), 1) * 100
    ann = [[f'{ACTION[(v, dd)]}\n{int(n.loc[v, dd]):,} CIFs ({n.loc[v, dd] / max(len(d_), 1):.0%})\n{bal_pct.loc[v, dd]:.0f}% of balance' for dd in DEPTH_LEVELS] for v in VALUE_LEVELS]
    sns.heatmap(bal_pct, annot=np.array(ann), fmt='', cmap=CMAP_GREEN if g == 'Business' else CMAP_BLUE, cbar=False, ax=ax, linewidths=2, linecolor=INK['surface'],
                annot_kws={'size': 8.3}, vmin=0, vmax=max(bal_pct.values.max(), 1))
    ax.set_xlabel('relationship depth (active accounts)'); ax.set_ylabel(''); ax.grid(False); ax.tick_params(axis='x', labelrotation=0); style(ax, title=f'{g} · color = share of the {g.lower()} balance')
finish(fig, 'Customer value matrix', f'Active CIFs · value = rank by total balance {"within IorB" if VALUE_WITHIN_IORB else "over all CIFs"} · depth = active accounts of any type', name='23_value_matrix')

va = df.groupby('ValueAction').agg(CIFs=('CIF', 'size'), Deposits=('Deposits', 'sum'), Loans=('Loans', 'sum'), MedianDep=('Deposits', lambda s: s[s > 0].median()),
                                   Accounts=('ActiveAccountsCount', 'mean'), Families=('ProductFamilies', 'mean'), Select=('Select', lambda s: is_(s, 'Select').mean() * 100),
                                   Business=('IorB', lambda s: is_(s, 'Business').mean() * 100), Foreign=('ForD', lambda s: is_(s, 'Foreign').mean() * 100),
                                   Tenure=('RelationshipYears', 'median'), Closed12=('MonthsSinceLastClose', lambda s: (s <= 12).mean() * 100)).reindex(ACTION_ORDER).dropna(subset=['CIFs'])
va['% CIFs'] = va['CIFs'] / len(df) * 100; va['% Deposits'] = va['Deposits'] / df['Deposits'].sum() * 100; va['% Loans'] = va['Loans'] / max(df['Loans'].sum(), 1) * 100
va = va.rename(columns={'MedianDep': 'Median deposits (holders)', 'Accounts': 'Avg active accts', 'Families': 'Avg DD/CD/LN families', 'Select': '% Select', 'Business': '% Business',
                        'Foreign': '% Foreign', 'Tenure': 'Median tenure (y)', 'Closed12': '% closed an account in last 12m'})
display(styled(va[['CIFs', '% CIFs', 'Deposits', '% Deposits', 'Loans', '% Loans', 'Median deposits (holders)', 'Avg active accts', 'Avg DD/CD/LN families', '% Select', '% Business',
                   '% Foreign', 'Median tenure (y)', '% closed an account in last 12m']],
               int_cols=['CIFs'], money_cols=['Deposits', 'Loans', 'Median deposits (holders)'], one_dec=['Avg active accts', 'Avg DD/CD/LN families', 'Median tenure (y)'],
               pct_cols=['% CIFs', '% Deposits', '% Loans', '% Select', '% Business', '% Foreign', '% closed an account in last 12m'], gradient_cols=['% CIFs', '% Deposits', '% Loans']))

ct = pd.crosstab(df['Group3'], df['ValueAction'], normalize='index').reindex(index=GROUP3_ORDER, columns=ACTION_ORDER).fillna(0) * 100
fig, ax = plt.subplots(figsize=(14, 0.9 + 0.5 * len(ct))); left = np.zeros(len(ct))
for a in ACTION_ORDER:
    vals = ct[a].values
    ax.barh(ct.index, vals, left=left, color=ACTION_COLOR[a], label=a, height=0.62, edgecolor=INK['surface'], linewidth=1)
    for i, (l, v) in enumerate(zip(left, vals)):
        if v >= 7: ax.text(l + v / 2, i, f'{v:.0f}%', ha='center', va='center', fontsize=7.6, color=text_on(ACTION_COLOR[a]))
    left += vals
ax.invert_yaxis(); ax.set_xlim(0, 100); ax.xaxis.set_major_formatter(PCT0); ax.grid(axis='y', visible=False)
ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.42), ncol=5, fontsize=7.8)
finish(fig, 'Value-matrix action by segment', 'IorB × ForD × Select (today) · rows sum to 100%', name='23_value_actions')

hist_single = rate_by(base, 'AccountsEverBand', ACC_EVER_BANDS)
if {'1', '6+'} <= set(hist_single.index):
    print(f'Why depth matters (from §20): customers who ever had a single account churned {hist_single.loc["1", "rate"]:.0f}% of the time, '
          f'customers with 6+ accounts {hist_single.loc["6+", "rate"]:.0f}% (average {overall:.0f}%).')
pd_ = df[df['ValueAction'].eq('Protect & deepen')]
print(f'"Protect & deepen": {len(pd_):,} high-value customers with a single active account hold {money(pd_["Deposits"].sum())} of deposits '
      f'({pd_["Deposits"].sum() / df["Deposits"].sum():.0%}) and {money(pd_["Loans"].sum())} of loans.')

**Observations §23** *(fill in after the run)*

- **Protect & deepen:** … high-value customers rely on a single account and hold … of deposits — the first cross-sell and retention list.
- **Retain — core:** … customers, … of the balance, mostly …
- **Low value:** … % of customers; … of them are deep relationships with small balances (review cost-to-serve).
- **By segment:** the segment with the most high-value single-product customers is …

# 24. Funding profile — how stable are the deposits?

Deposits are the bank's funding, so *who* holds them matters as much as how much there is. Four standard questions:

1. **Insured vs. uninsured** — deposit insurance covers up to `INSURED_LIMIT` per depositor (per ownership category). Uninsured balances are the ones that leave first in a stress event. *Approximation:* the limit is applied once per CIF to its DD + CD balance; joint, trust and business ownership categories can raise real coverage.
2. **Non-maturity vs. time deposits** — DDA balances can leave any day; CDs are locked until maturity but are rate-sensitive at rollover.
3. **Concentration** — how much of the funding depends on the largest depositors.
4. **Country concentration of foreign funding** — Herfindahl index (HHI) across countries: above 2,500 is highly concentrated.

In [ ]:
INSURED_LIMIT = 250_000
dep_ = df['Deposits'].clip(lower=0)
df['DepInsured'] = dep_.clip(upper=INSURED_LIMIT); df['DepUninsured'] = (dep_ - INSURED_LIMIT).clip(lower=0)

def top_n_share(s, n):
    s = s.clip(lower=0).sort_values(ascending=False); return s.iloc[:n].sum() / s.sum() * 100 if s.sum() > 0 else np.nan
g = df.groupby('Group3')
fund = pd.DataFrame({'Deposits': g['Deposits'].sum(), 'DDA': g['DD Accounts Balance'].sum(), 'CD': g['CD Accounts Balance'].sum(),
                     'Uninsured': g['DepUninsured'].sum(), 'CIFs above limit': g['DepUninsured'].apply(lambda s: int((s > 0).sum())),
                     'Top 20 depositors %': g['Deposits'].apply(lambda s: top_n_share(s, 20))}).reindex(GROUP3_ORDER).dropna(subset=['Deposits'])
fund.loc['All CIFs'] = [df['Deposits'].sum(), df['DD Accounts Balance'].sum(), df['CD Accounts Balance'].sum(), df['DepUninsured'].sum(), int((df['DepUninsured'] > 0).sum()), top_n_share(df['Deposits'], 20)]
fund['% of deposits'] = fund['Deposits'] / df['Deposits'].sum() * 100
fund['DDA share %'] = fund['DDA'] / fund['Deposits'].replace(0, np.nan) * 100
fund['CD share %'] = fund['CD'] / fund['Deposits'].replace(0, np.nan) * 100
fund['Uninsured %'] = fund['Uninsured'] / fund['Deposits'].replace(0, np.nan) * 100
display(styled(fund[['Deposits', '% of deposits', 'DDA share %', 'CD share %', 'Uninsured', 'Uninsured %', 'CIFs above limit', 'Top 20 depositors %']],
               money_cols=['Deposits', 'Uninsured'], pct_cols=['% of deposits', 'DDA share %', 'CD share %', 'Uninsured %', 'Top 20 depositors %'], int_cols=['CIFs above limit'],
               gradient_cols=['Uninsured %'], cmap=CMAP_RED))

conc = pd.DataFrame({'Top N depositors': [10, 20, 50, 100, 500, 1000]})
conc['% of deposits'] = [top_n_share(df['Deposits'], n) for n in conc['Top N depositors']]
for thr in [1e6, 10e6]:
    m = df['Deposits'] >= thr
    print(f'Depositors with ≥ {money(thr)}: {m.sum():,} CIFs ({m.mean():.1%}) hold {df.loc[m, "Deposits"].sum() / df["Deposits"].sum():.0%} of deposits')
fd = df[is_(df['ForD'], 'Foreign')].groupby(df['Country'].fillna('(missing)'))['Deposits'].sum().clip(lower=0)
if fd.sum() > 0:
    sh = fd / fd.sum(); hhi = float((sh ** 2).sum() * 10000)
    print(f'Foreign funding: {money(fd.sum())} from {len(fd)} countries · largest {sh.idxmax()} {sh.max():.0%} · HHI {hhi:,.0f} '
          f'({"highly concentrated" if hhi > 2500 else "moderately concentrated" if hhi > 1500 else "diversified"})')

fig, axes = plt.subplots(1, 3, figsize=(19, 0.9 + 0.5 * len(GROUP3_ORDER)), gridspec_kw={'width_ratios': [1.25, 1, 0.9]})
f8 = fund.drop(index='All CIFs')
axes[0].barh(f8.index, (f8['Deposits'] - f8['Uninsured']).values, color=BLUE, label=f'Insured (≤ {money(INSURED_LIMIT)})', height=0.62)
axes[0].barh(f8.index, f8['Uninsured'].values, left=(f8['Deposits'] - f8['Uninsured']).values, color=RED, label='Uninsured', height=0.62)
for i, (tot, pct) in enumerate(zip(f8['Deposits'], f8['Uninsured %'])):
    axes[0].text(tot, i, f'  {pct:.0f}% uninsured', va='center', fontsize=7.8, color=INK['secondary'])
axes[0].invert_yaxis(); axes[0].xaxis.set_major_formatter(MONEY); axes[0].grid(axis='y', visible=False); axes[0].legend(fontsize=7.8, loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2)
axes[0].set_xlim(0, f8['Deposits'].max() * 1.35); style(axes[0], title='Insured vs uninsured deposits (approx.)')
left = np.zeros(len(f8))
for col, c in [('DDA share %', BLUE), ('CD share %', BLUE_T)]:
    vals = f8[col].fillna(0).values
    axes[1].barh(f8.index, vals, left=left, color=c, label=col.replace(' share %', ''), height=0.62)
    for i, (l, v) in enumerate(zip(left, vals)):
        if v >= 8: axes[1].text(l + v / 2, i, f'{v:.0f}%', ha='center', va='center', fontsize=7.8, color=text_on(c))
    left += vals
axes[1].invert_yaxis(); axes[1].set_xlim(0, 100); axes[1].xaxis.set_major_formatter(PCT0); axes[1].tick_params(labelleft=False); axes[1].grid(axis='y', visible=False)
axes[1].legend(fontsize=7.8, loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2); style(axes[1], title='Non-maturity (DDA) vs time (CD)')
axes[2].plot(conc['Top N depositors'], conc['% of deposits'], color=GREEN, marker='o', lw=2)
for n_, v in zip(conc['Top N depositors'], conc['% of deposits']): axes[2].text(n_, v + 1.5, f'{v:.0f}%', ha='center', fontsize=7.8, color=INK['secondary'])
axes[2].set_xscale('log'); axes[2].set_ylim(0, 105); axes[2].yaxis.set_major_formatter(PCT0); style(axes[2], title='Share of deposits held by the top N depositors', xlabel='N (log)')
finish(fig, 'Funding profile', 'Active CIFs · segments = IorB × ForD × Select (today)', name='24_funding')

**Observations §24** *(fill in after the run)*

- **Uninsured deposits:** … % of deposits (approx.), concentrated in … ; … customers above the limit.
- **Mix:** … % DDA / … % CD; the most CD-dependent segment is …
- **Concentration:** top 20 depositors = … % of deposits; customers with ≥ $10M hold … %.
- **Foreign funding:** HHI … — dependent on …

# 25. Early warning — active customers that look like the ones who left

§20 says who left; this section turns it into a **watch list of active customers** with rule-based flags. Each flag is a signal a relationship manager can check, and where the history allows it the table shows the evidence from §20 (the attrition share of comparable customers in the past). The flags are not a model and have not been validated against future churn — that needs two snapshots of the file a few months apart (see §19 / §20.9). The score is the number of flags; tiers are Low (0–1), Medium (2), High (3+).

In [ ]:
EW_LOW_BALANCE   = 100      # $ — total balance at or below this, and no account types outside the extract = dormant
EW_COUNTRY_RATIO = 1.25     # a country is "high attrition" if its historical attrition share is ≥ this × the average (n ≥ 50)
EXPORT_WATCHLIST = True
WATCHLIST_PATH   = 'attrition_watchlist.csv'

cr = rate_by(base, 'Country', min_n=50)
HIGH_CHURN_COUNTRIES = [c for c in cr.index[cr['rate'] >= overall * EW_COUNTRY_RATIO] if c != '(missing)']
print(f'Countries with historical attrition ≥ {EW_COUNTRY_RATIO}× the average ({overall:.0f}%): {HIGH_CHURN_COUNTRIES or "none"}')

FLAGS = {
    'Closed an account in the last 12 months': df['MonthsSinceLastClose'] <= 12,
    'Closed half or more of its accounts ever': df['ClosureRatio'] >= 0.5,
    'Single active account': df['ActiveAccountsCount'].fillna(0) <= 1,
    'Dormant (near-zero balance)': (df['TotalBalance'].abs() <= EW_LOW_BALANCE) & ~df['Has_Other'],   # customers with non-extract products are not dormant
    'First year of the relationship': df['RelationshipYears'] < 1,
    'CD only (maturity risk)': df['ProductMix'].eq('CD') & ~df['Has_Other'],
    'Loan only (payoff risk)': df['ProductMix'].eq('LN') & ~df['Has_Other'],
    'Country with high historical attrition': df['Country'].isin(HIGH_CHURN_COUNTRIES),
}
for k, v in FLAGS.items(): df['EW · ' + k] = v.fillna(False).astype(bool)
EW_COLS = ['EW · ' + k for k in FLAGS]
df['RiskScore'] = df[EW_COLS].sum(axis=1)
RISK_TIERS = ['High', 'Medium', 'Low']
df['RiskTier'] = np.select([df['RiskScore'] >= 3, df['RiskScore'] == 2], ['High', 'Medium'], 'Low')

def _hist(label, mask):
    m = mask.fillna(False); return f'{base.loc[m, "Churned"].mean() * 100:.0f}% of {m.sum():,} comparable past customers left' if m.sum() >= 30 else 'n too small'
evidence = {
    'Closed an account in the last 12 months': 'needs two snapshots to measure',
    'Closed half or more of its accounts ever': 'needs two snapshots to measure',
    'Single active account': _hist('single', base['AccountsEverBand'].eq('1')) + ' (ever had 1 account)',
    'Dormant (near-zero balance)': 'needs balance history',
    'First year of the relationship': f'{(ch["LifetimeYears"] < 1).mean() * 100:.0f}% of all leavers left in year one; {100 - km_at(x_all, s_all, 1):.0f}% of new relationships end within a year',
    'CD only (maturity risk)': _hist('cd', base['ExitProfile'].eq('CD only')) + ' (ever held only CDs)',
    'Loan only (payoff risk)': _hist('ln', base['ExitProfile'].eq('LN only')) + ' (ever held only loans)',
    'Country with high historical attrition': _hist('ctry', base['Country'].isin(HIGH_CHURN_COUNTRIES)),
}
ew = pd.DataFrame({'Active CIFs flagged': [int(df[c].sum()) for c in EW_COLS], '% of active': [df[c].mean() * 100 for c in EW_COLS],
                   'Deposits': [df.loc[df[c], 'Deposits'].sum() for c in EW_COLS], 'Loans': [df.loc[df[c], 'Loans'].sum() for c in EW_COLS],
                   'Historical evidence (§20)': [evidence[k] for k in FLAGS]}, index=list(FLAGS))
display(styled(ew, int_cols=['Active CIFs flagged'], pct_cols=['% of active'], money_cols=['Deposits', 'Loans']))

tier = df.groupby(['Group3', 'RiskTier']).size().unstack().reindex(index=GROUP3_ORDER, columns=RISK_TIERS).fillna(0).astype(int)
tier['% High'] = tier['High'] / tier.sum(axis=1).replace(0, np.nan) * 100
tier['Deposits · High'] = df[df['RiskTier'].eq('High')].groupby('Group3')['Deposits'].sum().reindex(GROUP3_ORDER).fillna(0)
tier['Loans · High'] = df[df['RiskTier'].eq('High')].groupby('Group3')['Loans'].sum().reindex(GROUP3_ORDER).fillna(0)
tier.loc['All CIFs'] = [*(df['RiskTier'].value_counts().reindex(RISK_TIERS).fillna(0).astype(int)), df['RiskTier'].eq('High').mean() * 100,
                        df.loc[df['RiskTier'].eq('High'), 'Deposits'].sum(), df.loc[df['RiskTier'].eq('High'), 'Loans'].sum()]
display(styled(tier, int_cols=RISK_TIERS, pct_cols=['% High'], money_cols=['Deposits · High', 'Loans · High'], gradient_cols=['% High'], cmap=CMAP_RED))

TIER_COLOR = {'High': RED, 'Medium': '#F2A39E', 'Low': '#D7DBE1'}
fig, axes = plt.subplots(1, 3, figsize=(19, 0.9 + 0.5 * len(GROUP3_ORDER)), gridspec_kw={'width_ratios': [1.15, 1, 1]})
hbar(axes[0], list(FLAGS), ew['% of active'].values, [RED if v > 20 else GRAY for v in ew['% of active']], fmt=fmt_pct)
axes[0].xaxis.set_major_formatter(PCT0); style(axes[0], title='% of active customers with each flag')
for ax, (vals_from, ttl) in zip(axes[1:], [('n', '% of customers by risk tier'), ('dep', '% of deposits by risk tier')]):
    if vals_from == 'n': ct = pd.crosstab(df['Group3'], df['RiskTier'], normalize='index')
    else: ct = df.pivot_table(index='Group3', columns='RiskTier', values='Deposits', aggfunc='sum').clip(lower=0); ct = ct.div(ct.sum(axis=1).replace(0, np.nan), axis=0)
    ct = ct.reindex(index=GROUP3_ORDER, columns=RISK_TIERS).fillna(0) * 100; left = np.zeros(len(ct))
    for tname in RISK_TIERS:
        vals = ct[tname].values
        ax.barh(ct.index, vals, left=left, color=TIER_COLOR[tname], label=tname, height=0.62, edgecolor=INK['surface'], linewidth=1)
        for i, (l, v) in enumerate(zip(left, vals)):
            if v >= 8: ax.text(l + v / 2, i, f'{v:.0f}%', ha='center', va='center', fontsize=7.8, color=text_on(TIER_COLOR[tname]))
        left += vals
    ax.invert_yaxis(); ax.set_xlim(0, 100); ax.xaxis.set_major_formatter(PCT0); ax.grid(axis='y', visible=False); style(ax, title=ttl); ax.legend(fontsize=7.8, loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3)
axes[2].tick_params(labelleft=False)
finish(fig, 'Early-warning flags on the active base', 'Rule-based, not validated against future churn · tier: High = 3+ flags, Medium = 2, Low = 0–1', name='25_early_warning')

if EXPORT_WATCHLIST:
    wl = df[df['RiskTier'].ne('Low')].sort_values(['RiskScore', 'TotalBalance'], ascending=[False, False])
    wl[[ID_COL, 'IorB', 'ForD', 'Select', 'Segment', 'BranchGroupName', 'OfficerCode', 'Country', 'RelationshipYears', 'ActiveAccountsCount', 'ProductMix',
        'Deposits', 'Loans', 'ValueAction', 'RiskScore', 'RiskTier'] + EW_COLS].to_csv(WATCHLIST_PATH, index=False)
    print(f'Watch list exported → {WATCHLIST_PATH} ({len(wl):,} Medium / High customers, sorted by score then balance)')

**Observations §25** *(fill in after the run)*

- **Most common flags:** … ; the flag with the strongest historical evidence is …
- **High-risk tier:** … customers holding … of deposits and … of loans; concentrated in …
- **Overlap with value:** … high-value customers (§23) are in the High tier — the first calls to make.

# 26. Synthesis — who the customer is, and who it has been

One row per IorB × ForD with the headline of every view: what the segment is **today** (size, money, value, funding), what it **has been** (attrition, tenure of the leavers, survival, growth, BCG quadrant) and what is **at risk**. Today's measures use the active base; historical measures use active + churned with stable attributes only (no Select). The enriched CIF-level table is exported for Power BI.

In [ ]:
EXPORT_ENRICHED = True
ENRICHED_PATH   = 'customer_profile_enriched.csv'

syn = pd.DataFrame(index=order4)
gA = df.groupby('Group2')
syn['Active CIFs'] = gA.size(); syn['% of CIFs'] = syn['Active CIFs'] / len(df) * 100
syn['% of deposits'] = gA['Deposits'].sum() / df['Deposits'].sum() * 100
syn['% of loans'] = gA['Loans'].sum() / max(df['Loans'].sum(), 1) * 100
syn['Median deposits (holders)'] = gA['Deposits'].apply(lambda s: s[s > 0].median())
syn['% Select (today)'] = gA['Select'].apply(lambda s: is_(s, 'Select').mean() * 100)
syn['Churned CIFs'] = t4['Churned'].reindex([tuple(g.split(SEP)) for g in order4]).values
syn['Attrition share %'] = t4['Attrition share %'].reindex([tuple(g.split(SEP)) for g in order4]).values
syn['Median lifetime of leavers (y)'] = ch.groupby('Group2')['LifetimeYears'].median()
syn['5-year survival %'] = [km_tbl.loc[('IorB × ForD', g), 'after 5y %'] if ('IorB × ForD', g) in km_tbl.index else np.nan for g in order4]
syn[f'Customer growth % a year ({BCG_YEARS}y)'] = bcg2['CIF growth % a year'].reindex(order4)
syn['BCG quadrant'] = bcg2['Quadrant'].reindex(order4)
syn['% High value'] = gA['ValueLevel'].apply(lambda s: s.eq('High value').mean() * 100)
syn['% single account'] = gA['DepthLevel'].apply(lambda s: s.eq('Single').mean() * 100)
syn['% deposits uninsured'] = gA['DepUninsured'].sum() / gA['Deposits'].sum().replace(0, np.nan) * 100
syn['% High risk'] = gA['RiskTier'].apply(lambda s: s.eq('High').mean() * 100)
syn['Deposits at High risk'] = df[df['RiskTier'].eq('High')].groupby('Group2')['Deposits'].sum().reindex(order4).fillna(0)
display(styled(syn, int_cols=['Active CIFs', 'Churned CIFs'], money_cols=['Median deposits (holders)', 'Deposits at High risk'],
               pct_cols=['% of CIFs', '% of deposits', '% of loans', '% Select (today)', 'Attrition share %', '5-year survival %', f'Customer growth % a year ({BCG_YEARS}y)',
                         '% High value', '% single account', '% deposits uninsured', '% High risk'],
               one_dec=['Median lifetime of leavers (y)'], gradient_cols=['% of deposits', '% of loans']))

# Fingerprint: every column scaled 0–1 across the four segments
fp_cols = ['% of CIFs', '% of deposits', '% of loans', 'Median deposits (holders)', 'Attrition share %', '5-year survival %', f'Customer growth % a year ({BCG_YEARS}y)',
           '% High value', '% single account', '% deposits uninsured', '% High risk']
fp = syn[fp_cols].astype(float); fp_s = (fp - fp.min()) / (fp.max() - fp.min()).replace(0, np.nan)
fig, ax = plt.subplots(figsize=(16, 0.9 + 0.6 * len(fp)))
sns.heatmap(fp_s.fillna(0.5), annot=fp.map(lambda v: '' if pd.isna(v) else (money(v) if abs(v) >= 1000 else f'{v:.0f}')).values, fmt='', cmap=CMAP_BLUE, cbar=False, ax=ax,
            linewidths=1, linecolor=INK['surface'], annot_kws={'size': 8.5}, vmin=0, vmax=1)
ax.set_ylabel(''); ax.set_xlabel(''); ax.grid(False); ax.tick_params(axis='x', labelrotation=30)
for lab in ax.get_xticklabels(): lab.set_ha('right')
finish(fig, 'Segment fingerprint — today and history', 'Color = rank across the four segments (dark = highest) · cells show the actual value', name='26_fingerprint')

if EXPORT_ENRICHED:
    ENRICHED_COLS = [ID_COL, 'IorB', 'ForD', 'Select', 'Group2', 'Group3', 'Segment', 'BranchGroupName', 'OfficerCode', 'Country', 'Region', 'ForeignSplit', 'Age', 'BusinessAge',
                     'RelationshipStart', 'RelationshipYears', 'CohortYear', 'ActiveAccountsCount', 'ProductFamilies', 'ProductMix', 'Deposits', 'Loans', 'TotalBalance', 'ValueBand',
                     'ValueLevel', 'DepthLevel', 'ValueAction', 'DepInsured', 'DepUninsured', 'RiskScore', 'RiskTier'] + EW_COLS
    df[[c for c in ENRICHED_COLS if c in df.columns]].to_csv(ENRICHED_PATH, index=False)
    print(f'Enriched active-CIF table exported → {ENRICHED_PATH} ({len(df):,} rows)')

**Observations §26 — who the customer is and who it has been** *(fill in after the run — one paragraph per segment)*

- **Individual · Domestic** — today: … ; history: … ; at risk: …
- **Individual · Foreign** — today: … ; history: … ; at risk: …
- **Business · Domestic** — today: … ; history: … ; at risk: …
- **Business · Foreign** — today: … ; history: … ; at risk: …

**Data that would most improve §21–§26:** monthly balance history (balance-based BCG growth, deposit run-off, a validated early-warning model), deposit ownership category (exact insured amounts), and a second snapshot of this file 3–6 months later (to measure which flagged customers actually left).